# Area Coverage — CPU, bangunan800m, toko/kompetitor2km
Target pengguna: **latitude -6.409737552410453; longitude 106.82835008971841**.

Semua implementasi Python ditampilkan sebagai source cell, lalu dipanggil oleh cell eksekusi.
Tidak ada base64, arsip implementasi tersembunyi, `git clone`, atau kode remote yang dieksekusi.
Dataset terlampir merupakan snapshot DATA sumber nyata; file `engine_source` di dalamnya hanya arsip provenance dan tidak diimpor.

Alur: verifikasi sumber → selection/capacity → graph/access → connectors → routing → BigQuery → Folium.
CPU saja. Accepted adalah keputusan skenario7km/jam/round(menit)≤10; bukan validasi akses fisik atau populasi.
Tidak menjalankan Traffic, PPAC, model atau GPU.


## 1. Konfigurasi yang bisa diedit
Radius bangunan800m berbeda dari radius marker toko/kompetitor2000m.
Network extraction5500m mempertahankan kontrol eksperimen untuk routing; tampilan network dipotong2km tanpa memotong graph perhitungan.
`FROZEN` memakai data terlampir. Koordinat/radius sumber berbeda ditolak; pilih `REFRESH` dan revision baru untuk akuisisi baru.
`BQ_MODE=ACQUIRE_ONCE` membuat snapshot private pertama setelah secret tersedia, lalu memakai cache sukses dalam session.
Untuk hasil lintas session identik, lampirkan output/cache BigQuery versi sebelumnya dan gunakan `FROZEN`; tabel live dapat berubah.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,shutil
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['PROJ_NETWORK']='OFF'
SOURCE_ROOT=Path('/kaggle/working/area_coverage_cpu_source')
(SOURCE_ROOT/'notebooks').mkdir(parents=True,exist_ok=True)
SOURCE_MODE='FROZEN'       # FROZEN or REFRESH; never silently fetch on a cache mismatch
SOURCE_REVISION='user-target-800m-v1'
BQ_MODE='ACQUIRE_ONCE'     # ACQUIRE_ONCE or FROZEN
BQ_SECRET_NAME='GCP_SERVICE_ACCOUNT_JSON'
BQ_FROZEN_CACHE_INPUT=None  # Optional path to a previously saved bq_snapshot directory in /kaggle/input
BQ_SCHEMA_OVERRIDES={}    # e.g. {'indomaret': {'point':'KOORDINAT'}}; verified against actual table.schema
CONFIG={
 'latitude':-6.409737552410453,'longitude':106.82835008971841,
 'building_radius_m':800.0,'poi_radius_m':2000.0,'building_query_padding_m':50.0,
 'min_area_m2':45.0,'max_area_m2':550.0,'metric_crs':'EPSG:32748','network_radius_m':5500.0,
 'overture_release':'2026-10-21.0','overture_schema':'2.0.0','max_building_rows':50000,
 'bq_billing_project':'locdeproject','bq_maximum_bytes_billed':268435456,
 'bq_tables':{'competitor':'locdeproject.DWH_NS_LOC.COMPETITOR','indomaret':'locdeproject.DWH_NS_LOC.AREA_COVERAGE'}
}
print(json.dumps(CONFIG,indent=2));print('Execution: CPU only; no GPU requested')


## 2. Dependencies dan runtime CPU eksplisit
Cell ini hanya memasang dependency. Source algoritme berada lengkap di bawah. GDAL worker memakai system Python, terpisah dari runtime notebook.


In [ ]:
packages=['duckdb==2.0.0','osmnx==2.1.1','shapely==2.2.0','pyproj==3.8.0','numpy==2.5.4','pandas==3.1.0','geopandas==1.2.0','google-cloud-bigquery==3.46.2','folium==0.20.0']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*packages],check=True)
# Avoid passing a different Conda PROJ database to the system GDAL worker.
for name in ['PROJ_LIB','PROJ_DATA','GDAL_DATA']:
 os.environ.pop(name,None)
CORE_PYTHON='/usr/bin/python3'
probe=subprocess.run([CORE_PYTHON,'-c','from osgeo import gdal;import pyproj;assert int(gdal.VersionInfo("VERSION_NUM"))>=3070000'],capture_output=True,text=True)
if probe.returncode:
 subprocess.run(['apt-get','update','-qq'],check=True)
 subprocess.run(['apt-get','install','-y','-qq','python3-gdal','python3-pyproj'],check=True)
subprocess.run([CORE_PYTHON,'-c','import platform,pyproj;from osgeo import gdal;print("CORE",platform.python_version(),gdal.VersionInfo("RELEASE_NAME"),pyproj.__version__);assert int(gdal.VersionInfo("VERSION_NUM"))>=3070000'],check=True)
print('Notebook Python',sys.version)



## 3. Implementasi yang benar-benar diimpor
Setiap cell berikut menulis file Python yang isinya terlihat penuh. Buka cell nama modul untuk memeriksa algoritme. Folder ini menjadi satu-satunya source helper yang digunakan.


### location_intelligent_config.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_config.py
"""Editable LI access scenario; no retrieval or routing runs on import.

Edit CONFIG here, then export a NEW snapshot for each analysis run:
    python notebooks/location_intelligent_config.py > /tmp/li-policy.json

This is a policy snapshot, NOT the li_routing_config_v1 consumed by the
historical N1 diagnostic engine. Integration requires a separately tested
policy-aware routing profile. None denotes an unresolved decision, except
an explicitly UNBOUNDED limit, where it means no finite cap. Never zero.
"""
from copy import deepcopy
import json


CONFIG = {
    "schema_version": "li_access_policy_v1",
    "policy_version": "motorcycle_experiment_v8_li3",
    "status": "LI3_PILOT_SCENARIO_NOT_PRODUCTION_VALIDATED",
    "decision_basis": "Owner clarification: D02/D03/D05/D10, 2026-10-22",
    "travel": {
        "mode": "motorcycle",
        "direction": "building_to_store",
        "max_travel_min": {"value": 10.0, "unit": "min"},
        "threshold_status": "TEMPORARY_BUSINESS_SCENARIO_NOT_VALIDATED",
        "threshold_comparison": "<=",
        "threshold_rounding": "PYTHON_ROUND_HALF_EVEN_TOTAL_MINUTES_BEFORE_COMPARISON",
        "reference_constant_speed": {"value": 7.0, "unit": "km/h"},
        "speed_status": "N1_ASSUMPTION_NOT_VALIDATED_FOR_MOTORCYCLE",
        "segment_speed_model": "ALL_SEGMENTS_USE_REFERENCE_CONSTANT_SPEED",
    },
    "connectors": {
        "max_building_to_road_m": {"value": 125.0, "unit": "m"},
        "distance_status": "EXPERIMENTAL_CANDIDATE_SEARCH_LIMIT",
        "reference_search_origin": "BUILDING_REPRESENTATIVE_POINT",
        "distance_measure": "PROJECTED_POINT_TO_ROAD_GEOMETRY_DISTANCE",
        "candidate_distance_is_access_evidence": False,
        "building_access": "USE_EVIDENCE_OR_EXPLICIT_OWNER_ACCEPTED_ASSUMPTION",
        "store_entrance": "USE_EVIDENCE_OR_EXPLICIT_OWNER_ACCEPTED_ASSUMPTION",
        "require_access_evidence": True,
        "unknown_access_usable": False,
        "infer_straight_line_access": False,
        "max_target_snap_m": {"value": None, "unit": "m",
                              "limit_type": "UNBOUNDED"},
        "target_snap_search_scope": "AVAILABLE_GRAPH_ONLY",
        "target_snap_selection": "NEAREST_VALID_ROAD",
        "candidate_selection": "HIGHEST_VALID_ACCESS_CLASS_THEN_MINIMUM_TOTAL_TIME",
        "candidate_comparison_precision": "FULL_PRECISION_BEFORE_THRESHOLD_ROUND",
        "entrance_location_policy": "ASSUME_ROAD_FACING_BUILDING_LOCATION_IS_ENTRANCE",
        "entrance_location_claim_type": "OWNER_ACCEPTED_ASSUMPTION_NOT_OBSERVATION",
        "road_facing_location_method": "FACE_HIGHEST_ROAD_CLASS",
        "road_class_hierarchy": [
            ["primary"],
            ["secondary"],
            ["tertiary"],
            ["unclassified", "residential", "living_street"],
            ["service"],
            ["track", "path"],
        ],
        "highest_class_vs_valid_access_precedence": "VALID_ACCESS_BEFORE_CLASS_RANK",
        "highest_class_vs_minimum_time_precedence": "CLASS_RANK_BEFORE_TOTAL_TIME",
        "unreachable_higher_class": "TRY_NEXT_REACHABLE_CLASS",
        "link_class_mapping": "USE_PARENT_CLASS",
        "entrance_assumption_overrides_access_restrictions": False,
        "entrance_candidate_distance_rule": None,
        "target_connector_cost_model": "CONNECTOR_DISTANCE_AT_REFERENCE_CONSTANT_SPEED",
    },
    "sources": {
        "target": {
            "provider": "EXPLICIT_TARGET_INPUT",
            "coordinate_source": None,
            "source_version_or_as_of": None,
        },
        "buildings": {
            "provider": "OVERTURE_MAPS",
            "planned_release": "2026-10-21.0",
            "planned_schema_version": "2.0.0",
            "status": "PLANNED_NOT_ACQUIRED_OR_ADAPTER_VALIDATED",
            "reference_n1_release": "2026-08-19.0",
        },
        "roads_access_barriers_turns": {
            "provider": "OPENSTREETMAP_VIA_OVERPASS",
            "library": "OSMNX",
            "planned_library_version": "2.1.1",
            "network_type": "drive_service",
            "network_filter_is_complete_motorcycle_profile": False,
            "overpass_endpoint": None,
            "server_version": None,
            "snapshot_as_of": None,
            "restriction_relation_acquisition": None,
            "status": "PLANNED_NOT_ACQUIRED_OR_ADAPTER_VALIDATED",
        },
        "provenance": {
            "record_raw_and_normalized_checksums": True,
            "record_extent_request_filter_versions_and_missing_reasons": True,
        },
    },
    "capacity_reference": {
        "profile": "N1_SPLIT_REFERENCE_ONLY",
        "standard_residential_value": 4,
        "estimate_level_from_height": True,
        "floor_height_m": 3.0,
        "apartment": {"net_ratio": 0.70, "average_area_m2": 45.0, "occupancy": 0.70},
        "dormitory": {"net_ratio": 0.70, "average_area_m2": 15.0, "occupancy": 0.75},
        "hotel": {"net_ratio": 0.65, "average_area_m2": 30.0, "occupancy": 0.45,
                  "guests_per_occupied_room": 2.0},
        "allow_mixed_unit_total": False,
    },
    "li3": {
        "pilot_location_source": "OWNER_APPROVED_N1_EXAMPLE",
        "longitude": 106.8101931806858,
        "latitude": -6.430669461543375,
        "metric_crs": "EPSG:32748",
        "network_bbox_radius_m": 5500.0,
        "building_cache": "results/location_intelligent/overture_20261021_n1_verified",
        "overpass_endpoints": ["https://overpass-api.de/api/interpreter",
                               "https://overpass.kumi.systems/api/interpreter"],
        "request_timeout_s": 45,
        "max_response_bytes": 67108864,
        "network_type": "drive_service",
        "motor_supplements": True,
        "unsupported_restrictions": "QUARANTINE_AFFECTED_MOVEMENTS_AND_REPORT_UNKNOWN",
        "source_node_topology_only": True,
        "projection_position_precision_m": 0.000001,
        "entrance_inset_m": 0.01,
        "connector_assumption": "OWNER_FRONTAGE_ASSUMPTION_WITH_MAPPED_BARRIER_CHECK",
        "live_physical_verification": False,
    },
    "motorcycle_access": {
        "verified_motorcycle_alley_usable": True,
        "mapped_alley_missing_motorcycle_permission": "ASSUME_USABLE_UNLESS_PROHIBITED",
        "pedestrian_only_usable": False,
        "stairs_usable": False,
        "private_access_usable": False,
        "unknown_access_usable": False,
        "gates_and_prohibitions": "GATE_ASSUMED_PASSABLE_UNLESS_CLOSED_OR_PROHIBITED",
        "respect_direction": True,
        "cross_unverified_barriers": False,
        "invent_u_turns_or_reverse_edges": False,
        "provider_tag_mapping": None,
        "turn_restriction_model": None,
        "park_and_walk_policy": "MOTORCYCLE_ONLY_NO_WALKING_LEG",
        "no_route_policy": "EXCLUDE_FROM_COUNT_WITH_REASON",
    },
    "evidence": {
        "unknown_access_status": "UNVERIFIED_EXCLUDED_FROM_ROUTING",
        "exclusion_is_proof_of_physical_inaccessibility": False,
        "owner_accepted_assumptions": [
            "MAPPED_ALLEY_USABLE_UNLESS_PROHIBITED",
            "GATE_PASSABLE_IF_NO_CLOSURE_INFORMATION",
            "ROAD_FACING_BUILDING_LOCATION_IS_ENTRANCE",
        ],
        "accepted_evidence_specification": None,
        "freshness_policy": None,
    },
    "baseline": {
        "profile": "N1_REFERENCE_ONLY",
        "preserve_existing_inputs_and_outputs": True,
        "threshold_rule": "round(total_min, 3) < 10",
        "candidate_selection": "NEAREST_REACHABLE",
        "target_connector": "OMITTED_WITH_OFFSET_DIAGNOSTIC",
    },
}


def get_config():
    """Return an independent JSON-compatible snapshot of current settings."""
    return deepcopy(CONFIG)


if __name__ == "__main__":
    print(json.dumps(get_config(), ensure_ascii=False, sort_keys=True,
                     indent=2, allow_nan=False))


### location_intelligent_validator.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_validator.py
"""LI-0: read-only JSON contract validation; standard library, no engine imports.

Canonical contract: docs/AREA_COVERAGE_DATA_CONTRACT.md (LI-0 v1).
READY means the local input gate for LI-1 building selection, not calculation,
geographic coverage, live access, capacity semantics or production validation.
"""
from __future__ import annotations

import argparse
from datetime import datetime, timezone
import hashlib
from graphlib import CycleError, TopologicalSorter
import json
import math
from pathlib import Path, PurePosixPath
import re
from typing import Any


SCHEMA = "location_intelligent_v1"
RESULT_SCHEMA = "location_intelligent_validation_v1"
ID = re.compile(r"[A-Za-z0-9_-]+\Z")
SHA256 = re.compile(r"[0-9a-f]{64}\Z")
AVAILABILITY = {"NOT_ATTEMPTED", "UNAVAILABLE", "REQUEST_FAILED",
                "AVAILABLE_EMPTY", "AVAILABLE", "PARTIAL"}
STATUSES = {"NOT_RUN", "RUNNING", "BLOCKED", "SUCCEEDED",
            "SUCCEEDED_WITH_WARNINGS", "PARTIAL", "FAILED", "STALE"}
LAYERS = {"RAW", "NORMALIZED", "FROZEN", "DERIVED_GEOSPATIAL", "TRAFFIC_MACHINE",
          "REVIEWED", "PPAC_FEATURES", "PPAC_RESULT"}
# These are reference CFG names/units, never implicit defaults or business approval.
POSITIVE = {"building_radius_m": "m", "min_area_m2": "m2", "max_area_m2": "m2",
            "apartment_average_unit_area_m2": "m2", "dormitory_average_room_area_m2": "m2",
            "hotel_average_room_area_m2": "m2", "default_floor_height_m": "m",
            "max_building_to_road_m": "m", "max_travel_min": "min",
            "motorcycle_speed_kmh": "km/h", "network_radius_m": "m",
            "hotel_average_guests_per_room": "guest/room",
            "standard_residential_household": "legacy_unknown"}
RATIOS = {"standard_residential_occupancy", "apartment_net_floor_ratio",
          "apartment_occupancy", "dormitory_net_floor_ratio", "dormitory_occupancy",
          "hotel_net_floor_ratio", "hotel_occupancy"}
REQUIRED_PARAMETERS = {"building_radius_m", "min_area_m2", "max_area_m2"}


def canonical_hash(value: Any) -> str:
    """Python JSON v1: sorted keys, compact, UTF-8, no nonfinite numbers."""
    raw = json.dumps(value, sort_keys=True, separators=(",", ":"),
                     ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def context_hash(context: dict) -> str:
    return canonical_hash({key: value for key, value in context.items() if key != "context_hash"})


def _number(value: Any) -> bool:
    return type(value) in (int, float) and (type(value) is int or math.isfinite(value))


def _text(value: Any) -> bool:
    return isinstance(value, str) and bool(value.strip())


def _strict_pairs(pairs: list) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON object key")
        result[key] = value
    return result


def _reject_constant(value: str) -> None:
    raise ValueError("Nonfinite JSON constant")


class _Validator:
    def __init__(self, root: Path):
        self.root = root.resolve()
        self.diagnostics: list[dict] = []
        self.verified_sources: set[str] = set()

    def issue(self, code: str, path: str, message: str, severity: str = "ERROR",
              blocking: bool = True) -> None:
        # Never echo arbitrary input, credential values, or file contents.
        self.diagnostics.append(dict(code=code, path=path, severity=severity,
                                     blocking=blocking, message=message))

    def obj(self, value: Any, path: str, required: str, optional: str = "") -> dict:
        if not isinstance(value, dict):
            self.issue("TYPE_OBJECT", path, "Supply a JSON object.")
            return {}
        req, opt = set(required.split()), set(optional.split())
        for key in sorted(req - value.keys()):
            self.issue("FIELD_REQUIRED", f"{path}.{key}", "Supply the required field; do not substitute a default.")
        for key in sorted(value.keys() - req - opt):
            self.issue("FIELD_UNKNOWN", f"{path}.{key}", "Use a field defined by this schema version.")
        return value

    def text(self, value: Any, path: str) -> bool:
        if not _text(value):
            self.issue("TYPE_STRING", path, "Supply a nonempty string.")
            return False
        return True

    def identifier(self, value: Any, path: str) -> bool:
        if not isinstance(value, str) or not ID.fullmatch(value):
            self.issue("ID_INVALID", path, "Use a nonempty path-safe ID containing only letters, digits, _ or -.")
            return False
        return True

    def enum(self, value: Any, allowed: set, path: str) -> bool:
        if not isinstance(value, str) or value not in allowed:
            self.issue("ENUM_INVALID", path, "Use an explicitly supported value from the canonical contract.")
            return False
        return True

    def array(self, value: Any, path: str) -> list:
        if not isinstance(value, list):
            self.issue("TYPE_ARRAY", path, "Supply a JSON array.")
            return []
        return value

    def strings(self, value: Any, path: str) -> list[str]:
        items = self.array(value, path)
        valid = []
        for index, item in enumerate(items):
            if self.text(item, f"{path}[{index}]"):
                valid.append(item)
        if len(set(valid)) != len(valid):
            self.issue("REFERENCE_DUPLICATE", path, "Remove duplicate references.")
        return valid

    def stamp(self, value: Any, path: str) -> datetime | None:
        try:
            if not isinstance(value, str) or "T" not in value:
                raise ValueError
            stamp = datetime.fromisoformat(value.replace("Z", "+00:00"))
            if stamp.utcoffset() != timezone.utc.utcoffset(stamp):
                raise ValueError
            return stamp
        except (ValueError, TypeError, OverflowError):
            self.issue("TIMESTAMP_INVALID", path, "Supply an ISO8601 UTC timestamp with timezone.")
            return None

    def digest(self, value: Any, path: str) -> bool:
        if not isinstance(value, str) or not SHA256.fullmatch(value):
            self.issue("CHECKSUM_INVALID", path, "Supply a lowercase 64-digit SHA256 digest.")
            return False
        return True

    def fingerprint(self, value: Any, byte_count: Any, path: str) -> bool:
        fp = self.obj(value, f"{path}.fingerprint", "algorithm value")
        valid = self.digest(fp.get("value"), f"{path}.fingerprint.value")
        if fp.get("algorithm") != "sha256":
            self.issue("CHECKSUM_ALGORITHM", f"{path}.fingerprint.algorithm", "Only sha256 is supported in v1.")
            valid = False
        if type(byte_count) is not int or byte_count < 0:
            self.issue("BYTE_COUNT_INVALID", f"{path}.byte_count", "Supply an explicit nonnegative integer byte count.")
            valid = False
        return valid

    def unknown(self, data: dict, key: str, path: str, *, blocking: bool) -> None:
        reasons = data.get("unknown_reasons")
        if not isinstance(reasons, dict) or not _text(reasons.get(key)):
            self.issue("UNKNOWN_REASON_REQUIRED", f"{path}.unknown_reasons.{key}",
                       "Explain the explicitly unknown field.")
        self.issue("VALUE_UNKNOWN", f"{path}.{key}", "Value remains unknown; supply evidence before using it.",
                   "UNKNOWN", blocking)

    def reasons(self, value: Any, path: str) -> None:
        if not isinstance(value, dict) or not all(_text(k) and _text(v) for k, v in value.items()):
            self.issue("UNKNOWN_REASONS_INVALID", path, "Supply a mapping of unknown field names to nonempty reasons.")

    def json_values(self, value: Any, path: str) -> None:
        if isinstance(value, dict):
            for key in sorted(value, key=str):
                if not isinstance(key, str):
                    self.issue("JSON_KEY_INVALID", path, "JSON object keys must be strings.")
                    continue
                try:
                    key.encode("utf-8")
                except UnicodeError:
                    self.issue("JSON_STRING_INVALID", path, "JSON keys must be encodable as valid UTF-8 Unicode.")
                    continue
                self.json_values(value[key], f"{path}.{key}")
        elif isinstance(value, list):
            for index, child in enumerate(value):
                self.json_values(child, f"{path}[{index}]")
        elif value is not None and type(value) not in (str, bool, int, float):
            self.issue("JSON_VALUE_INVALID", path, "Supply a JSON-compatible value.")
        elif type(value) is float and not math.isfinite(value):
            self.issue("NUMBER_NONFINITE", path, "Replace NaN/Infinity with explicit null and a reason.")
        elif isinstance(value, str):
            try:
                value.encode("utf-8")
            except UnicodeError:
                self.issue("JSON_STRING_INVALID", path, "Supply valid Unicode before computing canonical UTF-8 hashes.")

    def config(self, value: Any) -> None:
        cfg = self.obj(value, "context.config", "version area_coverage")
        self.text(cfg.get("version"), "context.config.version")
        area = self.obj(cfg.get("area_coverage"), "context.config.area_coverage",
                        "parameters metric_crs unresolved_decisions")
        crs = area.get("metric_crs")
        if crs is None:
            self.issue("METRIC_CRS_REQUIRED", "context.config.area_coverage.metric_crs",
                       "Select an explicit supported metric CRS before calculation.", "BLOCKER")
        elif not isinstance(crs, str) or not re.fullmatch(r"EPSG:32[67](0[1-9]|[1-5][0-9]|60)", crs):
            self.issue("METRIC_CRS_UNSUPPORTED", "context.config.area_coverage.metric_crs",
                       "LI-0 v1 supports explicit WGS84 UTM EPSG:32601–32660/32701–32760 only.")
        decisions = self.strings(area.get("unresolved_decisions"), "context.config.area_coverage.unresolved_decisions")
        if decisions:
            self.issue("POLICY_UNRESOLVED", "context.config.area_coverage.unresolved_decisions",
                       "Resolve the listed decisions for the declared calculation scope.", "BLOCKER")
        params = area.get("parameters")
        if not isinstance(params, dict):
            self.issue("TYPE_OBJECT", "context.config.area_coverage.parameters", "Supply parameter quantities.")
            return
        for key in sorted(REQUIRED_PARAMETERS - params.keys()):
            self.issue("CONFIG_REQUIRED", f"context.config.area_coverage.parameters.{key}",
                       "Supply the explicit reference parameter and unit.", "BLOCKER")
        values = {}
        for key in sorted(params):
            path = f"context.config.area_coverage.parameters.{key}"
            if key not in POSITIVE and key not in RATIOS:
                self.issue("CONFIG_UNSUPPORTED", path, "This LI-0 version does not validate this parameter; do not silently ignore it.")
                continue
            q = self.obj(params[key], path, "value unit", "unknown_reason")
            if "unknown_reason" in q:
                self.text(q["unknown_reason"], f"{path}.unknown_reason")
            expected = POSITIVE.get(key, "ratio")
            if q.get("unit") != expected:
                self.issue("UNIT_INVALID", f"{path}.unit", f"Use the canonical unit {expected}.")
            val = q.get("value")
            if val is None:
                if not _text(q.get("unknown_reason")):
                    self.issue("UNKNOWN_REASON_REQUIRED", path, "Explain the missing parameter value.")
                self.issue("CONFIG_UNKNOWN", path, "Supply the parameter before calculation; no default is applied.", "UNKNOWN")
            elif not _number(val):
                self.issue("TYPE_NUMBER", f"{path}.value", "Supply a finite number; bool and numeric strings are invalid.")
            elif (key in RATIOS and not 0 <= val <= 1) or (key in POSITIVE and val <= 0):
                self.issue("CONFIG_RANGE", f"{path}.value", "Use a positive quantity or a ratio within [0,1], as applicable.")
            else:
                values[key] = val
        if "min_area_m2" in values and "max_area_m2" in values and values["min_area_m2"] >= values["max_area_m2"]:
            self.issue("CONFIG_ORDER", "context.config.area_coverage.parameters", "min_area_m2 must be less than max_area_m2.")
        if "standard_residential_household" in params:
            self.issue("CAPACITY_UNIT_UNRESOLVED", "context.config.area_coverage.parameters.standard_residential_household",
                       "Legacy mixed capacity is not an approved household/resident unit.", "WARNING", False)

    def context(self, value: Any) -> dict:
        c = self.obj(value, "context", "schema_version analysis_run_id parent_analysis_run_id created_at target config context_hash")
        if c.get("schema_version") != SCHEMA:
            self.issue("SCHEMA_UNSUPPORTED", "context.schema_version", f"Use {SCHEMA}.")
        self.identifier(c.get("analysis_run_id"), "context.analysis_run_id")
        if c.get("parent_analysis_run_id") is not None:
            self.identifier(c["parent_analysis_run_id"], "context.parent_analysis_run_id")
            if c["parent_analysis_run_id"] == c.get("analysis_run_id"):
                self.issue("PARENT_SELF", "context.parent_analysis_run_id", "A revision cannot be its own parent.")
        self.stamp(c.get("created_at"), "context.created_at")
        target = self.obj(c.get("target"), "context.target", "store_code store_name latitude longitude crs unknown_reasons")
        self.reasons(target.get("unknown_reasons"), "context.target.unknown_reasons")
        for key in ("store_code", "store_name"):
            if target.get(key) is None:
                self.unknown(target, key, "context.target", blocking=False)
            else:
                self.text(target[key], f"context.target.{key}")
        if target.get("crs") != "EPSG:4326":
            self.issue("CRS_INVALID", "context.target.crs", "Declare target coordinates as EPSG:4326; do not infer axis order.")
        lat, lon = target.get("latitude"), target.get("longitude")
        if (lat is None) != (lon is None):
            self.issue("COORDINATE_PAIR", "context.target", "Supply both coordinates or explicitly mark both unknown.")
        for key, limit in (("latitude", 90), ("longitude", 180)):
            v = target.get(key)
            if v is None:
                self.unknown(target, key, "context.target", blocking=True)
            elif not _number(v):
                self.issue("TYPE_NUMBER", f"context.target.{key}", "Supply a finite number; bool and strings are invalid.")
            elif not -limit <= v <= limit:
                self.issue("COORDINATE_RANGE", f"context.target.{key}", "Use the documented geographic coordinate range.")
        self.config(c.get("config"))
        # Check declared UTM area-of-use without projection or auto-selecting a CRS.
        cfg = c.get("config")
        area = cfg.get("area_coverage") if isinstance(cfg, dict) else None
        crs = area.get("metric_crs") if isinstance(area, dict) else None
        if _number(lat) and _number(lon) and isinstance(crs, str) and re.fullmatch(r"EPSG:32[67](0[1-9]|[1-5][0-9]|60)", crs):
            zone, south = int(crs[-2:]), crs.startswith("EPSG:327")
            if not (-80 <= lat <= 84 and (lat <= 0 if south else lat >= 0)
                    and -180 + 6 * (zone - 1) <= lon <= -180 + 6 * zone):
                self.issue("METRIC_CRS_DOMAIN", "context.config.area_coverage.metric_crs",
                           "Target lies outside the declared UTM zone/hemisphere domain; select/review CRS explicitly.")
        if self.digest(c.get("context_hash"), "context.context_hash"):
            try:
                actual = context_hash(c)
            except (TypeError, ValueError, OverflowError):
                actual = None
            if actual is not None and actual != c["context_hash"]:
                self.issue("CONTEXT_HASH_MISMATCH", "context.context_hash", "Recompute the context hash after an intentional revision.")
        return c

    def local_file(self, locator: Any, fingerprint: Any, byte_count: Any, path: str,
                   *, historical: bool = False) -> bool:
        if not self.fingerprint(fingerprint, byte_count, path):
            return False
        if not self.text(locator, f"{path}.path"):
            return False
        raw = PurePosixPath(locator)
        if "\\" in locator or ":" in locator or ".." in raw.parts or (raw.is_absolute() and not historical):
            self.issue("PATH_UNSAFE", f"{path}.path", "Use a confined relative path; historical sources may explicitly use an absolute local path.")
            return False
        candidate = Path(locator) if raw.is_absolute() else self.root / locator
        try:
            resolved = candidate.resolve()
            if not raw.is_absolute() and not resolved.is_relative_to(self.root):
                self.issue("PATH_ESCAPE", f"{path}.path", "The resolved path or symlink must remain inside the input root.")
                return False
            if not resolved.is_file():
                self.issue("LOCAL_FILE_MISSING", path, "Supply the referenced regular file; no fetch or stub is created.", "BLOCKER")
                return False
            with resolved.open("rb") as stream:
                digest = hashlib.sha256()
                size = 0
                for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                    size += len(chunk)
                    digest.update(chunk)
            if size != byte_count or digest.hexdigest() != fingerprint["value"]:
                self.issue("FILE_FINGERPRINT_MISMATCH", path, "Verify file identity/byte count and provide the matching immutable reference.")
                return False
            return True
        except (OSError, RuntimeError, ValueError):
            self.issue("LOCAL_READ_FAILED", path, "Check the local path and read permission; no network fallback is attempted.")
            return False

    def source(self, value: Any, path: str) -> dict:
        s = self.obj(value, path, "source_ref_id inventory_id kind provider locator acquisition_mode availability source_version source_as_of acquired_at schema_version fingerprint byte_count row_count coverage crs units provenance unknown_reasons",
                     "auth_ref local_path freshness freshness_policy_ref")
        self.reasons(s.get("unknown_reasons"), f"{path}.unknown_reasons")
        self.identifier(s.get("source_ref_id"), f"{path}.source_ref_id")
        self.enum(s.get("inventory_id"), {f"SRC{x:02d}" for x in range(1, 14)}, f"{path}.inventory_id")
        self.enum(s.get("kind"), {"BIGQUERY", "OBJECT_STORAGE", "API", "LOCAL_FILE"}, f"{path}.kind")
        self.enum(s.get("acquisition_mode"), {"LIVE_ACQUIRED", "OFFLINE_SNAPSHOT", "SYNTHETIC_FIXTURE"}, f"{path}.acquisition_mode")
        available = s.get("availability") in ("AVAILABLE", "AVAILABLE_EMPTY")
        if s.get("availability") is None:
            self.unknown(s, "availability", path, blocking=True)
        else:
            self.enum(s.get("availability"), AVAILABILITY, f"{path}.availability")
        for key in ("provider", "locator", "source_version", "source_as_of", "acquired_at", "schema_version", "crs", "units"):
            v = s.get(key)
            if v is None:
                self.unknown(s, key, path, blocking=available and key in {"schema_version", "crs", "units", "acquired_at"})
            elif key in {"source_as_of", "acquired_at"}:
                self.stamp(v, f"{path}.{key}")
            elif key == "units":
                if not isinstance(v, dict) or not v or not all(_text(k) and _text(x) for k, x in v.items()):
                    self.issue("UNITS_INVALID", f"{path}.units", "Supply a nonempty mapping from field to explicit unit.")
            else:
                self.text(v, f"{path}.{key}")
        prov = self.obj(s.get("provenance"), f"{path}.provenance", "origin description dependencies")
        self.enum(prov.get("origin"), {"LIVE_ACQUIRED", "OFFLINE_SNAPSHOT", "SYNTHETIC_FIXTURE"}, f"{path}.provenance.origin")
        if prov.get("origin") != s.get("acquisition_mode"):
            self.issue("PROVENANCE_MODE_MISMATCH", f"{path}.provenance.origin", "Match provenance origin to the declared acquisition mode.")
        self.text(prov.get("description"), f"{path}.provenance.description")
        self.strings(prov.get("dependencies"), f"{path}.provenance.dependencies")
        for key in ("auth_ref", "local_path"):
            if key in s:
                self.text(s[key], f"{path}.{key}")
        freshness = s.get("freshness")
        if freshness is None or freshness == "UNKNOWN":
            self.issue("FRESHNESS_UNKNOWN", path, "Freshness is unverified; offline does not automatically mean stale.", "UNKNOWN", False)
        elif self.enum(freshness, {"WITHIN_POLICY", "STALE_FOR_POLICY"}, f"{path}.freshness"):
            self.text(s.get("freshness_policy_ref"), f"{path}.freshness_policy_ref")
            if freshness == "STALE_FOR_POLICY":
                self.issue("SOURCE_STALE", path, "Select input permitted by the declared freshness policy.", "BLOCKER")
        coverage = s.get("coverage")
        if coverage is None:
            self.unknown(s, "coverage", path, blocking=available)
        else:
            cov = self.obj(coverage, f"{path}.coverage", "scope completeness")
            self.text(cov.get("scope"), f"{path}.coverage.scope")
            self.enum(cov.get("completeness"), {"COMPLETE", "PARTIAL", "UNKNOWN"}, f"{path}.coverage.completeness")
            if available and cov.get("completeness") != "COMPLETE":
                self.issue("SOURCE_COVERAGE_INCOMPLETE", path, "Resolve partial/unknown coverage before the offline input gate.", "BLOCKER")
        count = s.get("row_count")
        if s.get("availability") in ("NOT_ATTEMPTED", "UNAVAILABLE", "REQUEST_FAILED") or s.get("availability") is None:
            if count is not None:
                self.issue("ROW_COUNT_UNOBSERVED", f"{path}.row_count", "Keep unobserved row_count null; unavailable/failed is not zero.")
        elif type(count) is not int or count < 0:
            self.issue("ROW_COUNT_INVALID", f"{path}.row_count", "Supply an observed nonnegative integer row count.")
        elif (s.get("availability") == "AVAILABLE_EMPTY" and count != 0) or (s.get("availability") == "AVAILABLE" and count == 0):
            self.issue("EMPTY_STATUS_MISMATCH", f"{path}.row_count", "Use AVAILABLE_EMPTY for an observed successful zero-row result.")
        if available:
            local = s.get("local_path") or (s.get("locator") if s.get("kind") == "LOCAL_FILE" else None)
            if local is None:
                self.issue("SOURCE_NOT_LOCAL", path, "Provide a checksummed local snapshot; remote locators are never fetched.", "BLOCKER")
                # Still validate the declared checksum even without local bytes.
                self.fingerprint(s.get("fingerprint"), s.get("byte_count"), path)
            elif self.local_file(local, s.get("fingerprint"), s.get("byte_count"), path, historical=True):
                if isinstance(s.get("source_ref_id"), str):
                    self.verified_sources.add(s["source_ref_id"])
        elif s.get("availability") in ("NOT_ATTEMPTED", "UNAVAILABLE", "REQUEST_FAILED") or s.get("availability") is None:
            if s.get("fingerprint") is not None or s.get("byte_count") is not None:
                self.issue("FINGERPRINT_UNOBSERVED", path, "Unobserved sources must keep fingerprint/byte_count null; retain prior artifacts as separate snapshot references.")
        elif s.get("availability") == "PARTIAL" and (s.get("fingerprint") is not None or s.get("byte_count") is not None):
            self.fingerprint(s.get("fingerprint"), s.get("byte_count"), path)
        return s

    def records(self, value: Any, path: str, id_key: str) -> dict:
        result = {}
        for index, item in enumerate(self.array(value, path)):
            key = item.get(id_key) if isinstance(item, dict) else None
            if self.identifier(key, f"{path}[{index}].{id_key}"):
                if key in result:
                    self.issue("ID_DUPLICATE", f"{path}[{index}].{id_key}", "Use unique identifiers within the manifest.")
                result[key] = item
        return result

    def manifest(self, value: Any, c: dict) -> None:
        m = self.obj(value, "manifest", "schema_version analysis_run_id context_hash sources modules artifacts")
        if m.get("schema_version") != SCHEMA:
            self.issue("SCHEMA_UNSUPPORTED", "manifest.schema_version", f"Use {SCHEMA}.")
        for key in ("analysis_run_id", "context_hash"):
            if m.get(key) != c.get(key):
                self.issue("CONTEXT_BINDING_MISMATCH", f"manifest.{key}", "Bind the manifest to the supplied immutable context.")
        self.identifier(m.get("analysis_run_id"), "manifest.analysis_run_id")
        self.digest(m.get("context_hash"), "manifest.context_hash")
        sources = self.records(m.get("sources"), "manifest.sources", "source_ref_id")
        modules = self.records(m.get("modules"), "manifest.modules", "module_run_id")
        artifacts = self.records(m.get("artifacts"), "manifest.artifacts", "artifact_id")
        if sources.keys() & artifacts.keys():
            self.issue("REFERENCE_NAMESPACE_COLLISION", "manifest", "Use distinct source and artifact IDs to keep dependency references unambiguous.")
        area_required = set()
        for r in modules.values():
            if r.get("module") == "area_coverage" and isinstance(r.get("required_inputs"), list):
                area_required.update(x for x in r["required_inputs"] if isinstance(x, str))
        inventories = set()
        for index, value in enumerate(self.array(m.get("sources"), "manifest.sources")):
            start = len(self.diagnostics)
            s = self.source(value, f"manifest.sources[{index}]")
            required = isinstance(s.get("source_ref_id"), str) and s["source_ref_id"] in area_required
            if not required:
                for d in self.diagnostics[start:]:
                    if d["severity"] != "ERROR":
                        d["blocking"] = False
                        if d["severity"] == "BLOCKER":
                            d["severity"] = "WARNING"
            if required and s.get("inventory_id") == "SRC04":
                if isinstance(s.get("source_ref_id"), str) and s["source_ref_id"] in self.verified_sources:
                    inventories.add("SRC04")
                if s.get("availability") not in ("AVAILABLE", "AVAILABLE_EMPTY"):
                    self.issue("REQUIRED_SOURCE_UNAVAILABLE", f"manifest.sources[{index}]", "Building source is not available; preserve its status instead of zero.", "BLOCKER")
            prov = s.get("provenance")
            deps = prov.get("dependencies") if isinstance(prov, dict) else None
            if isinstance(deps, list):
                for ref in deps:
                    if isinstance(ref, str) and (ref not in sources or ref == s.get("source_ref_id")):
                        self.issue("SOURCE_DEPENDENCY_INVALID", f"manifest.sources[{index}].provenance.dependencies", "Reference another declared source.")
        if "SRC04" not in inventories:
            self.issue("BUILDINGS_SNAPSHOT_REQUIRED", "manifest.sources", "Provide a verifiable local SRC04 buildings snapshot for LI-1.", "BLOCKER")
        area_runs = []
        for index, value in enumerate(self.array(m.get("modules"), "manifest.modules")):
            p = f"manifest.modules[{index}]"
            r = self.obj(value, p, "module_run_id module analysis_run_id context_hash config_hash status required_inputs artifact_refs started_at finished_at engine error")
            self.enum(r.get("module"), {"area_coverage", "traffic_counter", "recount", "ppac"}, f"{p}.module")
            self.enum(r.get("status"), STATUSES, f"{p}.status")
            if r.get("module") == "area_coverage":
                area_runs.append(r)
            for key in ("analysis_run_id", "context_hash"):
                if r.get(key) != c.get(key):
                    self.issue("CONTEXT_BINDING_MISMATCH", f"{p}.{key}", "Match module identity to this context.")
            if self.digest(r.get("config_hash"), f"{p}.config_hash"):
                try:
                    expected = canonical_hash(c.get("config"))
                except (TypeError, ValueError, OverflowError):
                    expected = None
                if expected is not None and expected != r["config_hash"]:
                    self.issue("CONFIG_HASH_MISMATCH", f"{p}.config_hash", "Bind the module to the supplied configuration hash.")
            required = self.strings(r.get("required_inputs"), f"{p}.required_inputs")
            stage_required = r.get("module") == "area_coverage"
            for ref in required:
                if ref not in sources:
                    self.issue("SOURCE_REF_MISSING", f"{p}.required_inputs", "Declare each required source reference.",
                               "BLOCKER" if stage_required else "WARNING", stage_required)
                elif ref not in self.verified_sources:
                    self.issue("MODULE_SOURCE_NOT_VERIFIED", f"{p}.required_inputs", "Supply the required local file and verify its fingerprint.",
                               "BLOCKER" if stage_required else "WARNING", stage_required)
            if r.get("module") == "area_coverage" and not any(sources.get(ref, {}).get("inventory_id") == "SRC04" for ref in required):
                self.issue("AREA_SOURCE_REQUIRED", f"{p}.required_inputs", "Explicitly bind the area module to its SRC04 source.", "BLOCKER")
            refs = self.strings(r.get("artifact_refs"), f"{p}.artifact_refs")
            for ref in refs:
                if ref not in artifacts or artifacts[ref].get("module_run_id") != r.get("module_run_id"):
                    self.issue("ARTIFACT_REF_INVALID", f"{p}.artifact_refs", "Reference an artifact produced by this module run.")
            stamps = {}
            for key in ("started_at", "finished_at"):
                if r.get(key) is not None:
                    stamps[key] = self.stamp(r[key], f"{p}.{key}")
            if r.get("status") == "NOT_RUN" and (r.get("started_at") is not None or r.get("finished_at") is not None):
                self.issue("NOT_RUN_TIMESTAMPS", p, "NOT_RUN cannot carry execution start/finish timestamps.")
            if r.get("status") == "RUNNING" and (stamps.get("started_at") is None or r.get("finished_at") is not None):
                self.issue("RUNNING_TIMESTAMPS", p, "RUNNING needs started_at and no finished_at.")
            if r.get("status") in ("FAILED", "PARTIAL") and any(stamps.get(k) is None for k in ("started_at", "finished_at")):
                self.issue("EXECUTION_TIMESTAMPS_REQUIRED", p, "Executed FAILED/PARTIAL runs require both execution timestamps.")
            if all(stamps.get(k) is not None for k in ("started_at", "finished_at")) and stamps["finished_at"] < stamps["started_at"]:
                self.issue("TIMESTAMP_ORDER", p, "finished_at cannot precede started_at.")
            if r.get("status") in ("BLOCKED", "FAILED", "PARTIAL"):
                self.text(r.get("error"), f"{p}.error")
            elif r.get("error") is not None:
                self.text(r["error"], f"{p}.error")
            if r.get("engine") is not None:
                engine = self.obj(r["engine"], f"{p}.engine", "name version source_hashes")
                self.text(engine.get("name"), f"{p}.engine.name")
                self.text(engine.get("version"), f"{p}.engine.version")
                hashes = engine.get("source_hashes")
                if not isinstance(hashes, dict) or not hashes:
                    self.issue("ENGINE_PROVENANCE_REQUIRED", f"{p}.engine.source_hashes", "Supply immutable engine source hashes.")
                else:
                    for key in sorted(hashes):
                        self.digest(hashes[key], f"{p}.engine.source_hashes.{key}")
            elif r.get("status") in ("RUNNING", "FAILED", "PARTIAL"):
                self.issue("ENGINE_PROVENANCE_REQUIRED", f"{p}.engine", "Executed/running modules require engine provenance.")
            if r.get("status") in ("SUCCEEDED", "SUCCEEDED_WITH_WARNINGS"):
                if r.get("engine") is None or not refs or any(stamps.get(k) is None for k in ("started_at", "finished_at")):
                    self.issue("SUCCESS_EVIDENCE_REQUIRED", p, "Successful runs require engine provenance, timestamps and output references.")
        if len(area_runs) != 1:
            self.issue("AREA_MODULE_REQUIRED", "manifest.modules", "LI-0 input gate needs exactly one selected area_coverage module run.", "BLOCKER")
        elif area_runs[0].get("status") != "NOT_RUN":
            self.issue("AREA_STAGE_NOT_PENDING", "manifest.modules", "LI-0 prepares a NOT_RUN area stage; historical/completed jobs are not a new-stage readiness claim.", "BLOCKER")
        for index, value in enumerate(self.array(m.get("artifacts"), "manifest.artifacts")):
            p = f"manifest.artifacts[{index}]"
            a = self.obj(value, p, "artifact_id layer schema_version module_run_id path fingerprint byte_count dependencies status completeness")
            self.enum(a.get("layer"), LAYERS, f"{p}.layer")
            self.text(a.get("schema_version"), f"{p}.schema_version")
            self.enum(a.get("status"), STATUSES, f"{p}.status")
            self.enum(a.get("completeness"), {"COMPLETE", "PARTIAL", "UNKNOWN"}, f"{p}.completeness")
            owner = a.get("module_run_id")
            if not isinstance(owner, str) or owner not in modules:
                self.issue("PRODUCER_REF_INVALID", f"{p}.module_run_id", "Declare the producing module run.")
            elif not isinstance(modules[owner].get("artifact_refs"), list) or a.get("artifact_id") not in modules[owner]["artifact_refs"]:
                self.issue("ARTIFACT_REF_INVALID", p, "The producer must also reference this artifact.")
            elif modules[owner].get("status") in ("NOT_RUN", "BLOCKED"):
                self.issue("ARTIFACT_BEFORE_EXECUTION", p, "A module that has not executed cannot produce an existing output artifact.")
            elif modules[owner].get("status") in ("SUCCEEDED", "SUCCEEDED_WITH_WARNINGS") and (a.get("status") not in ("SUCCEEDED", "SUCCEEDED_WITH_WARNINGS") or a.get("completeness") != "COMPLETE"):
                self.issue("ARTIFACT_STATUS_MISMATCH", p, "A completed-success producer must reference complete successful outputs.")
            deps = self.strings(a.get("dependencies"), f"{p}.dependencies")
            if not deps:
                self.issue("ARTIFACT_LINEAGE_REQUIRED", f"{p}.dependencies", "Bind an existing output to at least one source or upstream artifact.")
            for ref in deps:
                if ref == a.get("artifact_id") or (ref not in sources and ref not in artifacts):
                    self.issue("ARTIFACT_DEPENDENCY_INVALID", f"{p}.dependencies", "Reference existing source/artifact IDs without self-dependency.")
            self.local_file(a.get("path"), a.get("fingerprint"), a.get("byte_count"), p)
        graph = {}
        for key, record in sources.items():
            prov = record.get("provenance")
            deps = prov.get("dependencies") if isinstance(prov, dict) else None
            graph[key] = [x for x in deps if isinstance(x, str)] if isinstance(deps, list) else []
        for key, record in artifacts.items():
            deps = record.get("dependencies")
            graph[key] = [x for x in deps if isinstance(x, str)] if isinstance(deps, list) else []
        try:
            tuple(TopologicalSorter(graph).static_order())
        except CycleError:
            self.issue("DEPENDENCY_CYCLE", "manifest", "Source/artifact provenance must form an acyclic dependency graph.")

    def result(self) -> dict:
        diagnostics = sorted(self.diagnostics, key=lambda d: (d["path"], d["code"], d["severity"], d["message"]))
        invalid = any(d["severity"] == "ERROR" for d in diagnostics)
        blocked = any(d["blocking"] for d in diagnostics)
        return {"schema_version": RESULT_SCHEMA, "validation_status": "INVALID" if invalid else "VALID",
                "offline_readiness": "BLOCKED" if blocked else "READY",
                "readiness_scope": "LI1_BUILDING_SELECTION_INPUT_CONTRACT_ONLY",
                "live_readiness": "NOT_VERIFIED", "verified_local_source_count": len(self.verified_sources),
                "limitations": ["No source retrieval or live authorization check.",
                                "No payload geometry/schema, geographic coverage or calculation validation.",
                                "No capacity, routing, Recount, Traffic or PPAC readiness claim."],
                "diagnostics": diagnostics}


def validate(context: Any, manifest: Any, *, root: Path | str) -> dict:
    """Validate supplied JSON values and declared local fingerprints; never mutate inputs."""
    v = _Validator(Path(root))
    v.json_values(context, "context")
    v.json_values(manifest, "manifest")
    if any(d["severity"] == "ERROR" for d in v.diagnostics):
        return v.result()
    c = v.context(context)
    v.manifest(manifest, c)
    return v.result()


def validate_files(context_path: Path | str, manifest_path: Path | str, *, root: Path | str) -> dict:
    """Strict local JSON loading: malformed, duplicate keys and nonfinite fail closed."""
    v = _Validator(Path(root))
    loaded = []
    for label, path in (("context", context_path), ("manifest", manifest_path)):
        try:
            if not Path(path).is_file():
                v.issue("JSON_FILE_UNAVAILABLE", label, "Supply a readable regular local JSON file; directories/devices/FIFOs are not inputs.")
                continue
            with Path(path).open(encoding="utf-8") as stream:
                loaded.append(json.load(stream, object_pairs_hook=_strict_pairs, parse_constant=_reject_constant))
        except (ValueError, UnicodeError, RecursionError):
            v.issue("JSON_INVALID", label, "Supply valid UTF-8 JSON without duplicate keys or nonfinite constants.")
        except OSError:
            v.issue("JSON_FILE_UNAVAILABLE", label, "Supply a readable local JSON input file.")
    if v.diagnostics:
        return v.result()
    return validate(*loaded, root=root)


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--context", required=True, type=Path)
    parser.add_argument("--manifest", required=True, type=Path)
    parser.add_argument("--root", required=True, type=Path, help="Local reference root; no files are written.")
    args = parser.parse_args(argv)
    result = validate_files(args.context, args.manifest, root=args.root)
    print(json.dumps(result, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False))
    return 1 if result["validation_status"] == "INVALID" else (2 if result["offline_readiness"] == "BLOCKED" else 0)


if __name__ == "__main__":
    raise SystemExit(main())


### location_intelligent_buildings.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_buildings.py
"""LI-1: local canonical GeoJSON validation/selection, no capacity or routing.

Uses installed GDAL/OGR (GEOS PointOnSurface) and OSR/PROJ with explicit XY.
Imports geometry dependencies only after the LI-0 input gate passes.
No dataset driver, URL opener, geometry repair, implicit CRS or source fetch.
"""
from __future__ import annotations

import argparse
from collections import Counter
from copy import deepcopy
import hashlib
import json
import math
from pathlib import Path
from typing import Any

from location_intelligent_validator import (
    _reject_constant, _strict_pairs, canonical_hash, validate, validate_files,
)


SCHEMA = "li_building_snapshot_v1"
RESULT_SCHEMA = "location_intelligent_building_selection_v1"
RULE_VERSION = "N1_BUILDING_SELECTION_V1"
SPECIAL = {"apartments": "apartment", "dormitory": "dormitory", "hotel": "hotel"}
MESSAGES = {
    "FEATURE_INVALID": "Each row must be a GeoJSON Feature object.",
    "PROPERTIES_INVALID": "Supply a properties object with canonical building attributes.",
    "BUILDING_ID_MISSING": "Supply a nonempty stable string building_id; no ID is generated.",
    "BUILDING_ID_CONFLICT": "Feature ID and canonical building_id must identify the same source record.",
    "DUPLICATE_BUILDING_ID": "All occurrences of this source ID are rejected; no arbitrary winner is selected.",
    "ATTRIBUTE_REQUIRED": "Supply class and subclass explicitly; null is allowed, omission is not a value.",
    "ATTRIBUTE_TYPE": "Class/subclass must be string or null; optional floors/height must be finite positive numbers or null.",
    "RECORD_NONFINITE": "Record metadata/geometry contains a nonfinite or unusable number; no output value is fabricated.",
    "GEOMETRY_MISSING": "Supply a non-null footprint geometry.",
    "GEOMETRY_TYPE": "Only 2D Polygon and MultiPolygon footprints are supported.",
    "GEOMETRY_EMPTY": "Empty footprints or rings do not provide a usable building geometry.",
    "GEOMETRY_COORDINATES": "Supply closed rings with at least four explicit finite [longitude,latitude] positions.",
    "COORDINATE_RANGE": "Coordinates must lie within EPSG:4326 longitude/latitude ranges.",
    "GEOMETRY_INVALID": "Invalid polygon topology is rejected; no automatic repair is performed.",
    "GEOMETRY_CRS_DOMAIN": "Footprint lies outside the declared supported UTM zone/hemisphere domain.",
    "GEOMETRY_TRANSFORM": "Metric transformation or representative-point/area calculation failed.",
    "OUTSIDE_RADIUS": "A valid representative point lies outside the configured building radius.",
    "CLASS_UNSUPPORTED": "Neither a supported special class nor null/empty/residential subtype applies.",
    "AREA_NOT_ABOVE_MIN": "Full projected footprint area must be strictly greater than min_area_m2.",
    "AREA_NOT_BELOW_MAX": "Standard residential footprint area must be strictly smaller than max_area_m2.",
}


def classify_building(building_class: str | None, subclass: str | None) -> str | None:
    """N1 cell6/cell10 precedence; null subtype fallback is intentionally retained."""
    cls = (building_class or "").strip().lower()
    subtype = (subclass or "").strip().lower()
    if cls in SPECIAL:
        return SPECIAL[cls]
    return "standard_residential" if subtype in ("", "residential") else None


def selection_reason(distance: float, area: float, building_type: str | None,
                     *, radius: float, minimum: float, maximum: float) -> str | None:
    """Full precision, order and strictness match N1; no jitter tolerance/rounding."""
    if distance > radius:
        return "OUTSIDE_RADIUS"
    if building_type is None:
        return "CLASS_UNSUPPORTED"
    if area <= minimum:
        return "AREA_NOT_ABOVE_MIN"
    if building_type == "standard_residential" and area >= maximum:
        return "AREA_NOT_BELOW_MAX"
    return None


def _finite(value: Any) -> bool:
    if type(value) not in (int, float):
        return False
    try:
        return math.isfinite(value)
    except OverflowError:
        return False


def _unicode(value: Any) -> None:
    if isinstance(value, str):
        value.encode("utf-8")
    elif isinstance(value, dict):
        for key, item in value.items():
            key.encode("utf-8")
            _unicode(item)
    elif isinstance(value, list):
        for item in value:
            _unicode(item)


def _finite_json(value: Any) -> bool:
    if isinstance(value, dict):
        return all(_finite_json(item) for item in value.values())
    if isinstance(value, list):
        return all(_finite_json(item) for item in value)
    return _finite(value) if type(value) in (int, float) else True


def _rings(geometry: Any) -> tuple[list, str | None]:
    if geometry is None:
        return [], "GEOMETRY_MISSING"
    if not isinstance(geometry, dict) or geometry.get("type") not in ("Polygon", "MultiPolygon"):
        return [], "GEOMETRY_TYPE"
    coords = geometry.get("coordinates")
    if not isinstance(coords, list):
        return [], "GEOMETRY_COORDINATES"
    if not coords:
        return [], "GEOMETRY_EMPTY"
    polygons = [coords] if geometry["type"] == "Polygon" else coords
    rings = []
    for polygon in polygons:
        if not isinstance(polygon, list):
            return [], "GEOMETRY_COORDINATES"
        if not polygon:
            return [], "GEOMETRY_EMPTY"
        for ring in polygon:
            if not isinstance(ring, list):
                return [], "GEOMETRY_COORDINATES"
            if not ring:
                return [], "GEOMETRY_EMPTY"
            if len(ring) < 4:
                return [], "GEOMETRY_COORDINATES"
            for xy in ring:
                if not isinstance(xy, list) or len(xy) != 2 or not all(_finite(v) for v in xy):
                    return [], "GEOMETRY_COORDINATES"
                if not (-180 <= xy[0] <= 180 and -90 <= xy[1] <= 90):
                    return [], "COORDINATE_RANGE"
            if ring[0] != ring[-1]:
                return [], "GEOMETRY_COORDINATES"
            rings.append(ring)
    return rings, None


def _backend():
    # No global dependency/environment modification; OGR parses an in-memory
    # geometry only. It never opens a dataset driver or a remote data source.
    from osgeo import gdal, ogr, osr
    return gdal, ogr, osr


def _spatial(crs: str, target: dict, ogr, osr):
    wgs, local = osr.SpatialReference(), osr.SpatialReference()
    if wgs.ImportFromEPSG(4326) != 0 or local.ImportFromEPSG(int(crs.split(":")[1])) != 0:
        raise ValueError("Unsupported spatial reference")
    for ref in (wgs, local):
        ref.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    transform = osr.CoordinateTransformation(wgs, local)
    point = ogr.Geometry(ogr.wkbPoint)
    point.AddPoint_2D(target["longitude"], target["latitude"])
    if point.Transform(transform) != 0:
        raise ValueError("Target projection failed")
    return transform, point


def _measure(geometry: dict, crs: str, transform, target, ogr, gdal):
    rings, reason = _rings(geometry)
    if reason:
        return None, reason
    # Conservative ordinary UTM domain, same explicit gate as LI-0. No zone
    # auto-selection, geodesic fallback or projection of out-of-domain vertices.
    zone, south = int(crs[-2:]), crs.startswith("EPSG:327")
    for ring in rings:
        for lon, lat in ring:
            if not (-80 <= lat <= 84 and (lat <= 0 if south else lat >= 0)
                    and -180 + 6 * (zone - 1) <= lon <= -180 + 6 * zone):
                return None, "GEOMETRY_CRS_DOMAIN"
    try:
        # Keep native GEOS diagnostics out of the deterministic ledger/stderr.
        with gdal.quiet_errors():
            geom = ogr.CreateGeometryFromJson(json.dumps(geometry, allow_nan=False))
            if geom is None or geom.IsEmpty():
                return None, "GEOMETRY_EMPTY"
            if not geom.IsValid():
                return None, "GEOMETRY_INVALID"
            representative = geom.PointOnSurface()
            if representative is None or representative.IsEmpty():
                return None, "GEOMETRY_TRANSFORM"
            # N1 obtains PointOnSurface BEFORE projection; projected centroid
            # or PointOnSurface on the projected polygon is a different rule.
            lon, lat = representative.GetX(), representative.GetY()
            metric_rep, metric_geom = representative.Clone(), geom.Clone()
            if metric_rep.Transform(transform) != 0 or metric_geom.Transform(transform) != 0:
                return None, "GEOMETRY_TRANSFORM"
            if not metric_geom.IsValid():
                return None, "GEOMETRY_TRANSFORM"
            area, distance = metric_geom.GetArea(), metric_rep.Distance(target)
        if not _finite(area) or not _finite(distance) or area <= 0 or distance < 0:
            return None, "GEOMETRY_TRANSFORM"
        return {"area_m2": area, "distance_to_store_m": distance,
                "representative_point": {"type": "Point", "coordinates": [lon, lat]},
                "lat": lat, "lon": lon}, None
    except (RuntimeError, ValueError, TypeError, OverflowError):
        return None, "GEOMETRY_TRANSFORM"


def select_buildings(context: Any, manifest: Any, *, root: Path | str,
                     source_ref_id: str) -> dict:
    """Return a deterministic local selection and one disposition per source row.

    Every invocation verifies LI-0 and the exact bytes consumed. A successful
    empty snapshot has count0; source failures have summary=null, never count0.
    """
    gate = validate(context, manifest, root=root)
    result = {"schema_version": RESULT_SCHEMA, "status": "BLOCKED", "live_readiness": "NOT_VERIFIED",
              "analysis_run_id": context.get("analysis_run_id") if isinstance(context, dict) else None,
              "rule_version": RULE_VERSION, "li0": gate, "snapshot": None,
              "summary": None, "selected": [], "reject_ledger": [], "diagnostics": [],
              "limitations": ["Reference building selection only; not routing/accessibility or capacity.",
                              "Declared coverage is not verified geographic completeness.",
                              "Synthetic fixtures do not validate live Overture integration."]}

    def fail(code, message, status="FAILED"):
        result["status"] = status
        result["diagnostics"].append({"code": code, "message": message})
        return result

    if gate["offline_readiness"] != "READY":
        return fail("LI0_GATE_BLOCKED", "Correct invalid or unavailable required LI-0 inputs first.", "BLOCKED")
    area_run = next(r for r in manifest["modules"] if r["module"] == "area_coverage")
    source = next((s for s in manifest["sources"] if s["source_ref_id"] == source_ref_id), None)
    if source is None or source_ref_id not in area_run["required_inputs"] or source["inventory_id"] != "SRC04":
        return fail("SOURCE_BINDING_INVALID", "Select an SRC04 source explicitly required by this area module.")
    result["snapshot"] = {"source_ref_id": source_ref_id, "fingerprint": deepcopy(source["fingerprint"]),
                          "schema_version": source["schema_version"], "provider": source["provider"],
                          "source_version": source["source_version"], "source_as_of": source["source_as_of"],
                          "acquisition_mode": source["acquisition_mode"], "provenance": deepcopy(source["provenance"]),
                          "coverage": deepcopy(source["coverage"]), "crs": source["crs"],
                          "context_hash": context["context_hash"], "config_hash": canonical_hash(context["config"])}
    if source["schema_version"] != SCHEMA:
        return fail("SNAPSHOT_SCHEMA_UNSUPPORTED", f"Supply the canonical local {SCHEMA} payload; no provider schema is inferred.")
    if source["crs"] != "EPSG:4326" or source["units"].get("coordinates") != "degree":
        return fail("SNAPSHOT_CRS_INVALID", "Canonical GeoJSON requires explicit EPSG:4326 [lon,lat] and degree units in SourceRef.")
    locator = source.get("local_path") or source["locator"]
    path = Path(locator) if Path(locator).is_absolute() else Path(root) / locator
    if path.suffix.lower() not in (".json", ".geojson"):
        return fail("SNAPSHOT_FORMAT_UNSUPPORTED", "Only canonical local JSON/GeoJSON FeatureCollection is supported.")
    try:
        if not path.is_file():
            return fail("SNAPSHOT_UNAVAILABLE", "Snapshot must remain an available regular local file.")
        raw = path.read_bytes()
        if len(raw) != source["byte_count"] or hashlib.sha256(raw).hexdigest() != source["fingerprint"]["value"]:
            return fail("SNAPSHOT_CHANGED", "The bytes consumed no longer match the immutable manifest reference.")
        data = json.loads(raw.decode("utf-8"), object_pairs_hook=_strict_pairs, parse_constant=_reject_constant)
        _unicode(data)
    except OSError:
        return fail("SNAPSHOT_UNREADABLE", "Local snapshot cannot be read; no live fallback is attempted.")
    except (ValueError, UnicodeError, RecursionError):
        return fail("SNAPSHOT_JSON_INVALID", "Supply strict UTF-8 JSON with unique keys and valid Unicode.")
    if not isinstance(data, dict) or data.get("type") != "FeatureCollection" or not isinstance(data.get("features"), list):
        return fail("SNAPSHOT_SCHEMA_INVALID", "Snapshot must contain type=FeatureCollection and an array of features.")
    if "crs" in data and data["crs"] != {"type": "name", "properties": {"name": "EPSG:4326"}}:
        return fail("SNAPSHOT_CRS_CONFLICT", "Payload CRS declaration conflicts with the supported canonical CRS.")
    features = data["features"]
    if len(features) != source["row_count"]:
        return fail("SNAPSHOT_ROW_COUNT_MISMATCH", "Feature count must match the manifest's observed row_count.")
    try:
        gdal, ogr, osr = _backend()
        if ogr.GetGEOSVersionMajor() < 1:
            raise ImportError("GEOS support required")
        crs = context["config"]["area_coverage"]["metric_crs"]
        with osr.ExceptionMgr(useExceptions=True):
            transform, target = _spatial(crs, context["target"], ogr, osr)
    except (ImportError, OSError):
        return fail("GEOMETRY_BACKEND_UNAVAILABLE", "Use an offline environment with GDAL/OGR, GEOS and PROJ installed.", "BLOCKED")
    except (RuntimeError, ValueError):
        return fail("GEOMETRY_BACKEND_FAILED", "Check local CRS support and transform initialization.")
    result["backend"] = {"gdal": gdal.VersionInfo("RELEASE_NAME"),
                         "geos": f"{ogr.GetGEOSVersionMajor()}.{ogr.GetGEOSVersionMinor()}.{ogr.GetGEOSVersionMicro()}",
                         "proj": f"{osr.GetPROJVersionMajor()}.{osr.GetPROJVersionMinor()}.{osr.GetPROJVersionMicro()}",
                         "metric_crs": crs, "axis_order": "XY", "representative_point": "GEOS PointOnSurface in EPSG:4326 before projection"}
    ids = []
    for f in features:
        props = f.get("properties") if isinstance(f, dict) else None
        key = props.get("building_id") if isinstance(props, dict) else None
        ids.append(key if isinstance(key, str) and key.strip() else None)
    duplicates = {key for key, count in Counter(x for x in ids if x is not None).items() if count > 1}
    params = context["config"]["area_coverage"]["parameters"]
    controls = {"radius": params["building_radius_m"]["value"], "minimum": params["min_area_m2"]["value"],
                "maximum": params["max_area_m2"]["value"]}
    fingerprint = source["fingerprint"]["value"]

    def reject(index, key, stage, code, field, metrics=None):
        result["reject_ledger"].append({"analysis_run_id": context["analysis_run_id"],
            "source_ref_id": source_ref_id, "snapshot_sha256": fingerprint,
            "source_row_index": index, "building_id": key, "stage": stage,
            "disposition": "EXCLUDED" if stage == "SELECTION" else "REJECTED",
            "code": code, "message": MESSAGES[code], "field": field,
            "context_hash": context["context_hash"], "config_hash": result["snapshot"]["config_hash"],
            "rule_version": RULE_VERSION, "metrics": metrics})

    for index, feature in enumerate(features):
        key = ids[index]
        if not isinstance(feature, dict) or feature.get("type") != "Feature":
            reject(index, key, "PAYLOAD", "FEATURE_INVALID", "type")
            continue
        props = feature.get("properties")
        if not isinstance(props, dict):
            reject(index, key, "PAYLOAD", "PROPERTIES_INVALID", "properties")
            continue
        if key is None:
            reject(index, None, "IDENTITY", "BUILDING_ID_MISSING", "properties.building_id")
            continue
        if key in duplicates:
            reject(index, key, "IDENTITY", "DUPLICATE_BUILDING_ID", "properties.building_id")
            continue
        if "id" in feature and feature["id"] != key:
            reject(index, key, "IDENTITY", "BUILDING_ID_CONFLICT", "id")
            continue
        if "class" not in props or "subclass" not in props:
            reject(index, key, "PAYLOAD", "ATTRIBUTE_REQUIRED", "properties.class/subclass")
            continue
        if any(props[k] is not None and not isinstance(props[k], str) for k in ("class", "subclass")) or any(
                props.get(k) is not None and (not _finite(props[k]) or props[k] <= 0) for k in ("num_floors", "height")):
            reject(index, key, "PAYLOAD", "ATTRIBUTE_TYPE", "properties")
            continue
        if not _finite_json(feature):
            reject(index, key, "PAYLOAD", "RECORD_NONFINITE", "feature")
            continue
        metrics, reason = _measure(feature.get("geometry"), crs, transform, target, ogr, gdal)
        if reason:
            reject(index, key, "GEOMETRY", reason, "geometry")
            continue
        kind = classify_building(props["class"], props["subclass"])
        reason = selection_reason(metrics["distance_to_store_m"], metrics["area_m2"], kind, **controls)
        if reason:
            reject(index, key, "SELECTION", reason, "geometry/class/subclass", metrics)
            continue
        result["selected"].append({"analysis_run_id": context["analysis_run_id"], "source_ref_id": source_ref_id,
            "source_row_index": index, "building_id": key, "snapshot_sha256": fingerprint,
            "building_key": {"provider": source["provider"], "source_version": source["source_version"],
                             "snapshot_sha256": fingerprint, "building_id": key},
            "class": props["class"], "subclass": (props["subclass"] or "").strip().lower() or None,
            "building_type": kind, "geometry": deepcopy(feature["geometry"]), "metric_crs": crs,
            "raw_properties": deepcopy(props), **metrics})
    rejected = sum(r["disposition"] == "REJECTED" for r in result["reject_ledger"])
    excluded = len(result["reject_ledger"]) - rejected
    result["summary"] = {"source_records": len(features), "selected": len(result["selected"]),
                         "excluded": excluded, "rejected": rejected,
                         "accounted_records": len(result["selected"]) + len(result["reject_ledger"]),
                         "complete_for_selection": rejected == 0,
                         "reason_counts": dict(sorted(Counter(r["code"] for r in result["reject_ledger"]).items()))}
    assert result["summary"]["accounted_records"] == len(features)
    result["status"] = "PARTIAL" if rejected else "SUCCEEDED"
    if duplicates:
        result["diagnostics"].append({"code": "SNAPSHOT_IDENTITY_CONFLICT", "message": "Repeated source IDs affect snapshot integrity; all occurrences are held outside selected records."})
    return result


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    for key in ("context", "manifest", "root"):
        parser.add_argument(f"--{key}", required=True, type=Path)
    parser.add_argument("--source-ref", required=True)
    args = parser.parse_args(argv)
    gate = validate_files(args.context, args.manifest, root=args.root)
    if gate["offline_readiness"] != "READY":
        result = {"schema_version": RESULT_SCHEMA, "status": "BLOCKED", "li0": gate,
                  "summary": None, "live_readiness": "NOT_VERIFIED"}
    else:
        # Already strictly validated; no fallback input or remote source.
        try:
            with args.context.open(encoding="utf-8") as stream:
                context = json.load(stream, object_pairs_hook=_strict_pairs, parse_constant=_reject_constant)
            with args.manifest.open(encoding="utf-8") as stream:
                manifest = json.load(stream, object_pairs_hook=_strict_pairs, parse_constant=_reject_constant)
            result = select_buildings(context, manifest, root=args.root, source_ref_id=args.source_ref)
        except (OSError, ValueError, RecursionError):
            result = {"schema_version": RESULT_SCHEMA, "status": "FAILED", "summary": None,
                      "live_readiness": "NOT_VERIFIED", "diagnostics": [{"code": "INPUT_CHANGED",
                      "message": "Context/manifest changed after validation; supply stable local inputs."}]}
    print(json.dumps(result, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False))
    return {"SUCCEEDED": 0, "PARTIAL": 3, "BLOCKED": 2, "FAILED": 1}[result["status"]]


if __name__ == "__main__":
    raise SystemExit(main())


### location_intelligent_capacity.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_capacity.py
"""N1 reference capacity, split units; no final population/household claims."""
from copy import deepcopy
import math

from location_intelligent_config import get_config
from location_intelligent_validator import canonical_hash


def estimate_capacity(selection, policy=None):
    policy = get_config() if policy is None else deepcopy(policy)
    cfg = policy["capacity_reference"]
    if cfg["profile"] != "N1_SPLIT_REFERENCE_ONLY" or cfg["allow_mixed_unit_total"] is not False:
        raise ValueError("Only split N1 reference capacity is supported")
    if type(cfg["estimate_level_from_height"]) is not bool:
        raise ValueError("Height fallback must be an explicit boolean")
    if selection.get("schema_version") != "location_intelligent_building_selection_v1" or selection.get("status") not in {"SUCCEEDED", "PARTIAL"}:
        raise ValueError("Supply an available LI-1 selection")
    for name, value in [("floor_height_m", cfg["floor_height_m"]), ("standard_residential_value", cfg["standard_residential_value"])]:
        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
            raise ValueError("Invalid capacity parameter: " + name)
    for name in ("apartment", "dormitory", "hotel"):
        for key, value in cfg[name].items():
            if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
                raise ValueError("Invalid capacity parameter: " + name + "." + key)
            if key in {"occupancy", "net_ratio"} and value > 1:
                raise ValueError("Ratio exceeds one")
    units = {"standard_residential": "N1_STANDARD_VALUE_UNIT_UNRESOLVED",
             "apartment": "OCCUPIED_DWELLING_UNITS_ESTIMATED",
             "dormitory": "OCCUPIED_ROOMS_ESTIMATED", "hotel": "GUESTS_ESTIMATED"}
    rows, seen = [], set()
    for building in selection["selected"]:
        key = building["building_id"]
        if key in seen:
            raise ValueError("Duplicate building identity")
        seen.add(key)
        kind, props = building["building_type"], building["raw_properties"]
        if kind not in units:
            raise ValueError("Unsupported building type")
        area = building["area_m2"]
        if type(area) not in (int, float) or not math.isfinite(area) or area <= 0:
            raise ValueError("Invalid footprint area")
        for name in ("num_floors", "height"):
            x = props.get(name)
            if x is not None and (type(x) not in (int, float) or not math.isfinite(x) or x <= 0):
                raise ValueError("Invalid optional level input")
        level, source = None, "UNKNOWN"
        if props.get("num_floors") is not None:
            level, source = max(1, round(props["num_floors"])), "NUM_FLOORS"
        elif cfg["estimate_level_from_height"] and props.get("height") is not None:
            level, source = max(1, round(props["height"] / cfg["floor_height_m"])), "ESTIMATED_FROM_HEIGHT"
        gfa = area * level if level is not None else None
        if gfa is not None and not math.isfinite(gfa):
            raise ValueError("Gross floor area overflow")
        raw = occupied = None
        if kind == "standard_residential":
            raw = cfg["standard_residential_value"]
        elif gfa is not None:
            c = cfg[kind]
            occupied = gfa * c["net_ratio"] / c["average_area_m2"] * c["occupancy"]
            raw = occupied * c.get("guests_per_occupied_room", 1)
        if raw is not None and not math.isfinite(raw):
            raise ValueError("Capacity overflow")
        rows.append({"building_id": key, "building_type": kind, "level": level,
                     "level_source": source, "gross_floor_area_m2": gfa,
                     "occupied_units_or_rooms_unrounded": occupied,
                     "capacity_unrounded": raw, "capacity_reference_rounded": round(raw) if raw is not None else None,
                     "unit": units[kind], "claim_type": "REFERENCE_ESTIMATE_NOT_OBSERVATION",
                     "reason": "MISSING_LEVEL" if raw is None else "REFERENCE_FORMULA"})
    groups = {}
    for unit in units.values():
        group = [r for r in rows if r["unit"] == unit]
        groups[unit] = {"buildings": len(group), "known_subtotal": sum(r["capacity_reference_rounded"] for r in group if r["capacity_reference_rounded"] is not None),
                        "unknown_buildings": sum(r["capacity_reference_rounded"] is None for r in group)}
    return {"schema_version": "li_capacity_reference_v1", "status": "PARTIAL" if selection["status"] == "PARTIAL" or any(r["capacity_reference_rounded"] is None for r in rows) else "SUCCEEDED",
            "profile": cfg["profile"], "selection_hash": canonical_hash(selection), "policy_hash": canonical_hash(policy),
            "rows": rows, "summary_by_unit": groups, "mixed_unit_total": None,
            "final_business_capacity": None, "limitations": ["Standard value4 has no approved household/population unit", "Not observed occupancy; no mixed-unit total", "Capacity describes selected buildings, not accessible buildings"]}


### location_intelligent_routing.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_routing.py
"""LI-2: offline directed multigraph diagnostics; final accessibility is blocked.

D02/D03/D05/D10 are OPEN in the audited decision register. Numeric N1 reference
results are explicitly separate from approved business/physical accessibility.
No live retrieval, OSMnx, implicit reverse edges, repair or framework imports.
"""
from __future__ import annotations

import argparse
from collections import Counter
from copy import deepcopy
import hashlib
import heapq
import json
import math
from pathlib import Path
from typing import Any

from location_intelligent_validator import _Validator, _strict_pairs, _reject_constant, canonical_hash, validate


GRAPH_SCHEMA = "li_directed_graph_v1"
CONFIG_SCHEMA = "li_routing_config_v1"
RESULT_SCHEMA = "location_intelligent_routing_v1"
OPEN_DECISIONS = ("D02", "D03", "D05", "D10")


def _number(value):
    try:
        return type(value) in (int, float) and math.isfinite(value)
    except OverflowError:
        return False


def inside_bbox(point, bbox):
    """Closed boundary, no hidden tolerance."""
    return bbox[0] <= point[0] <= bbox[2] and bbox[1] <= point[1] <= bbox[3]


def within_snap_limit(distance, maximum):
    return distance <= maximum


def reference_time_pass(total_min, maximum=10.0):
    """N1 compares its already-rounded three-decimal time strictly below limit."""
    return round(total_min, 3) < maximum


def classify_road(highway, service):
    h, s = (highway or "").strip().lower(), (service or "").strip().lower()
    if h in {"service", "track", "path", "footway", "pedestrian", "steps", "cycleway", "bridleway"} or s in {"driveway", "alley", "parking_aisle"}:
        return "gang"
    if h in {"residential", "unclassified", "living_street", "road"}:
        return "street"
    if h in {"tertiary", "tertiary_link", "secondary", "secondary_link"}:
        return "main street"
    if h in {"primary", "primary_link", "trunk", "trunk_link", "motorway", "motorway_link"}:
        return "highway"
    return None


def shortest_route(graph, start, target, weight="travel_time_s"):
    """Dijkstra: cost, then fewest hops, then lexicographic edge-ID sequence.

    Nonnegative weights and validated node/edge references are prerequisites.
    Hops before lexicographic IDs also terminates equal-cost zero-edge cycles.
    Every stored directed parallel edge is considered; nothing is reversed.
    """
    adjacency = {row["node_id"]: [] for row in graph["nodes"]}
    edges = {row["edge_id"]: row for row in graph["edges"]}
    if start not in adjacency or target not in adjacency:
        return None
    for edge in graph["edges"]:
        adjacency[edge["u"]].append(edge)
    best = {start: (0.0, 0, ())}
    queue = [(0.0, 0, (), start)]
    while queue:
        cost, hops, path, node = heapq.heappop(queue)
        if best.get(node) != (cost, hops, path):
            continue
        if node == target:
            records = [edges[key] for key in path]
            return {"nodes": [start, *[edge["v"] for edge in records]], "edges": list(path),
                    "weight": weight, "weight_value": cost,
                    "distance_m": sum(edge["length_m"] for edge in records),
                    "time_s": sum(edge["travel_time_s"] for edge in records)}
        for edge in sorted(adjacency[node], key=lambda row: row["edge_id"]):
            label = (cost + edge[weight], hops + 1, (*path, edge["edge_id"]))
            if edge["v"] not in best or label < best[edge["v"]]:
                best[edge["v"]] = label
                heapq.heappush(queue, (*label, edge["v"]))
    return None


def _project_to_line(point, coords):
    lengths = [math.dist(a, b) for a, b in zip(coords, coords[1:])]
    total, walked, best = sum(lengths), 0.0, None
    for i, (a, b, length) in enumerate(zip(coords, coords[1:], lengths)):
        if length == 0:
            continue
        fraction = max(0.0, min(1.0, ((point[0]-a[0])*((b[0]-a[0])/length)+(point[1]-a[1])*((b[1]-a[1])/length)) / length))
        access = [a[0]+fraction*(b[0]-a[0]), a[1]+fraction*(b[1]-a[1])]
        position = walked + fraction*length
        candidate = (math.dist(point, access), position, i, access)
        if best is None or candidate[:3] < best[:3]:
            best = candidate
        walked += length
    return best[0], best[1]/total, best[3], [best[3], *coords[best[2]+1:]]


def _configuration(v, config, context, selection, graph):
    c = v.obj(config, "routing_config", "schema_version version analysis_run_id context_hash selection_hash graph_hash profile path_weight parameters policy_refs connector_policy target_connector_policy")
    if c.get("schema_version") != CONFIG_SCHEMA:
        v.issue("ROUTING_SCHEMA_UNSUPPORTED", "routing_config.schema_version", f"Use {CONFIG_SCHEMA}.")
    for key in ("version",):
        v.text(c.get(key), "routing_config."+key)
    for key in ("analysis_run_id", "context_hash"):
        if c.get(key) != context.get(key):
            v.issue("ROUTING_CONTEXT_MISMATCH", "routing_config."+key, "Bind routing configuration to the validated current context.")
    for key, value in (("selection_hash", selection), ("graph_hash", graph)):
        if v.digest(c.get(key), "routing_config."+key) and c[key] != canonical_hash(value):
            v.issue("ROUTING_HASH_MISMATCH", "routing_config."+key, "Verify the canonical JSON input hash; do not use another run's result.")
    v.enum(c.get("profile"), {"N1_REFERENCE_ONLY"}, "routing_config.profile")
    v.enum(c.get("path_weight"), {"length_m", "travel_time_s"}, "routing_config.path_weight")
    v.enum(c.get("connector_policy"), {"EXPLICIT_FIXTURE_ACCESS_AND_N1_ROAD_CLASS"}, "routing_config.connector_policy")
    v.enum(c.get("target_connector_policy"), {"OMIT_N1_REFERENCE_REPORT_OFFSET"}, "routing_config.target_connector_policy")
    refs = v.obj(c.get("policy_refs"), "routing_config.policy_refs", " ".join(OPEN_DECISIONS))
    for key in OPEN_DECISIONS:
        v.text(refs.get(key), "routing_config.policy_refs."+key)
    params = v.obj(c.get("parameters"), "routing_config.parameters", "max_building_to_road_m max_travel_min motorcycle_speed_kmh max_target_snap_m")
    expected = {"max_building_to_road_m": ("m",125.0), "max_travel_min": ("min",10.0),
                "motorcycle_speed_kmh": ("km/h",7.0), "max_target_snap_m": ("m",None)}
    values = {}
    for key, (unit, reference) in expected.items():
        path = "routing_config.parameters."+key
        q = v.obj(params.get(key), path, "value unit", "unknown_reason")
        if q.get("unit") != unit:
            v.issue("ROUTING_UNIT_INVALID", path, f"Declare {unit} explicitly.")
        value = q.get("value")
        if key == "max_target_snap_m" and value is None:
            v.text(q.get("unknown_reason"), path+".unknown_reason")
            values[key] = None
        elif not _number(value) or value < 0 or (key != "max_target_snap_m" and value == 0):
            v.issue("ROUTING_PARAMETER_INVALID", path, "Supply a finite positive reference quantity, or a nonnegative snap diagnostic limit.")
        elif reference is not None and value != reference:
            v.issue("REFERENCE_CONTROL_CHANGED", path, "This profile preserves N1 controls125m/10min/7kmh; no business policy change is approved.")
        else:
            values[key] = value
        if "unknown_reason" in q:
            v.text(q["unknown_reason"], path+".unknown_reason")
    return values


def _graph(v, graph, metric_crs):
    g = v.obj(graph, "graph", "schema_version graph_id directed crs units coverage provenance nodes edges")
    if g.get("schema_version") != GRAPH_SCHEMA:
        v.issue("GRAPH_SCHEMA_UNSUPPORTED", "graph.schema_version", f"Use {GRAPH_SCHEMA}.")
    v.identifier(g.get("graph_id"), "graph.graph_id")
    if g.get("directed") is not True:
        v.issue("GRAPH_NOT_DIRECTED", "graph.directed", "Supply explicit directed edges; undirected data is not expanded.")
    if g.get("crs") != metric_crs:
        v.issue("GRAPH_CRS_MISMATCH", "graph.crs", "Graph must use the current explicit metric CRS.")
    if g.get("units") != {"coordinates":"m", "distance":"m", "travel_time":"s"}:
        v.issue("GRAPH_UNITS_INVALID", "graph.units", "Graph coordinates/distances must be meters and travel time seconds.")
    cov = v.obj(g.get("coverage"), "graph.coverage", "bbox completeness scope")
    bbox = cov.get("bbox")
    valid_bbox = isinstance(bbox,list) and len(bbox)==4 and all(_number(x) for x in bbox) and bbox[0] < bbox[2] and bbox[1] < bbox[3]
    if not valid_bbox:
        v.issue("GRAPH_COVERAGE_INVALID", "graph.coverage.bbox", "Declare a finite [xmin,ymin,xmax,ymax] metric coverage rectangle.")
    elif not math.isfinite(math.dist(bbox[:2],bbox[2:])):
        v.issue("GRAPH_COVERAGE_UNUSABLE", "graph.coverage.bbox", "Coverage span must support finite metric calculations.")
    v.enum(cov.get("completeness"), {"COMPLETE","PARTIAL","UNKNOWN"}, "graph.coverage.completeness")
    v.text(cov.get("scope"), "graph.coverage.scope")
    prov = v.obj(g.get("provenance"), "graph.provenance", "origin source_version source_as_of description")
    v.enum(prov.get("origin"), {"SYNTHETIC_FIXTURE","OFFLINE_SNAPSHOT"}, "graph.provenance.origin")
    v.text(prov.get("source_version"), "graph.provenance.source_version")
    v.text(prov.get("description"), "graph.provenance.description")
    if prov.get("source_as_of") is not None:
        v.stamp(prov["source_as_of"], "graph.provenance.source_as_of")
    nodes, edge_ids, triples = {}, set(), set()
    for index, row in enumerate(v.array(g.get("nodes"), "graph.nodes")):
        path=f"graph.nodes[{index}]";n=v.obj(row,path,"node_id x y")
        key=n.get("node_id")
        if not v.identifier(key,path+".node_id"):
            continue
        if key in nodes:
            v.issue("GRAPH_NODE_DUPLICATE",path,"Keep unique node identifiers.")
        if not all(_number(n.get(k)) for k in ("x","y")):
            v.issue("GRAPH_NODE_COORDINATES",path,"Supply finite projected XY coordinates.")
        elif valid_bbox and not inside_bbox([n["x"],n["y"]],bbox):
            v.issue("GRAPH_NODE_OUTSIDE_COVERAGE",path,"Nodes must lie inside the declared graph extent.")
        nodes[key]=n
    for index, row in enumerate(v.array(g.get("edges"), "graph.edges")):
        path=f"graph.edges[{index}]";e=v.obj(row,path,"edge_id u v key length_m travel_time_s highway service access_allowed geometry")
        key=e.get("edge_id")
        if not v.identifier(key,path+".edge_id"):
            continue
        if key in edge_ids:
            v.issue("GRAPH_EDGE_DUPLICATE",path,"Keep unique edge identifiers.")
        edge_ids.add(key)
        endpoints=True
        for name in ("u","v"):
            if not isinstance(e.get(name),str) or e[name] not in nodes:
                v.issue("GRAPH_NODE_REF_INVALID",path+"."+name,"Reference an existing node.")
                endpoints=False
        if v.text(e.get("key"),path+".key") and endpoints:
            triple=(e["u"],e["v"],e["key"])
            if triple in triples:
                v.issue("GRAPH_PARALLEL_KEY_DUPLICATE",path,"Use unique (u,v,key) identities for parallel edges.")
            triples.add(triple)
        for name in ("length_m","travel_time_s"):
            if not _number(e.get(name)) or e[name] < 0:
                v.issue("GRAPH_COST_INVALID",path+"."+name,"Supply explicit finite nonnegative costs; missing is not zero.")
        for name in ("highway","service"):
            if e.get(name) is not None:
                v.text(e[name],path+"."+name)
        if e.get("access_allowed") is not None and type(e["access_allowed"]) is not bool:
            v.issue("CONNECTOR_PERMISSION_INVALID",path+".access_allowed","Declare a fixture permission bool or explicit unknown null.")
        geom=e.get("geometry")
        if geom is None:
            continue
        coords=geom.get("coordinates") if isinstance(geom,dict) else None
        valid=isinstance(geom,dict) and geom.get("type")=="LineString" and isinstance(coords,list) and len(coords)>=2 and all(isinstance(x,list) and len(x)==2 and all(_number(v) for v in x) for x in coords)
        if not valid:
            v.issue("GRAPH_GEOMETRY_INVALID",path+".geometry","Supply an explicit 2D metric LineString or null; do not infer road geometry.")
            continue
        if valid_bbox and not all(inside_bbox(x,bbox) for x in coords):
            v.issue("GRAPH_EDGE_OUTSIDE_COVERAGE",path,"Stored road geometry leaves the declared extent.")
        geom_length=sum(math.dist(a,b) for a,b in zip(coords,coords[1:]))
        if not math.isfinite(geom_length):
            v.issue("GRAPH_GEOMETRY_UNUSABLE",path,"Road length must support finite metric calculations.")
        elif geom_length <= 0:
            v.issue("GRAPH_GEOMETRY_EMPTY",path,"Access road geometry must have positive length.")
        if endpoints and all(_number(nodes[e[k]].get(axis)) for k in ("u","v") for axis in ("x","y")):
            if coords[0] != [nodes[e["u"]]["x"],nodes[e["u"]]["y"]] or coords[-1] != [nodes[e["v"]]["x"],nodes[e["v"]]["y"]]:
                v.issue("GRAPH_EDGE_ORIENTATION",path,"Geometry endpoints must match stored u→v; it is never auto-reversed.")
        if _number(e.get("length_m")) and e["length_m"]==0:
            v.issue("GRAPH_COST_GEOMETRY_CONFLICT",path,"A positive road geometry cannot declare zero physical length.")
    for key in ("length_m","travel_time_s"):
        edge_rows=g.get("edges") if isinstance(g.get("edges"),list) else []
        vals=[e[key] for e in edge_rows if isinstance(e,dict) and _number(e.get(key))]
        if not math.isfinite(sum(vals)):
            v.issue("GRAPH_COST_UNUSABLE","graph.edges","Aggregated nonnegative route costs must remain finite.")
    return nodes


def _selection(v, selection, context, manifest, metric_crs):
    if not isinstance(selection,dict) or selection.get("schema_version") != "location_intelligent_building_selection_v1":
        v.issue("BUILDING_RESULT_INVALID","buildings","Supply the selected LI-1 result envelope.")
        return []
    if selection.get("status") not in ("SUCCEEDED","PARTIAL"):
        v.issue("BUILDING_RESULT_UNAVAILABLE","buildings.status","Only available LI-1 selections can be routed.","BLOCKER")
    snap=selection.get("snapshot")
    if not isinstance(snap,dict):
        v.issue("BUILDING_PROVENANCE_INVALID","buildings.snapshot","Supply source/config/context provenance.")
        return []
    if not isinstance(snap.get("fingerprint"),dict):
        v.issue("BUILDING_PROVENANCE_INVALID","buildings.snapshot.fingerprint","Supply the verified snapshot fingerprint object.")
        return []
    if selection.get("analysis_run_id") != context["analysis_run_id"] or snap.get("context_hash") != context["context_hash"] or snap.get("config_hash") != canonical_hash(context["config"]):
        v.issue("BUILDING_CONTEXT_MISMATCH","buildings","Select the LI-1 output for this immutable target/config.")
    source=next((x for x in manifest["sources"] if x["source_ref_id"]==snap.get("source_ref_id")),None)
    required={ref for r in manifest["modules"] if r["module"]=="area_coverage" for ref in r["required_inputs"]}
    if source is None or source["source_ref_id"] not in required or source["inventory_id"]!="SRC04" or source.get("fingerprint") != snap.get("fingerprint") or snap.get("crs") != "EPSG:4326" or source.get("crs")!=snap.get("crs") or source.get("schema_version")!=snap.get("schema_version"):
        v.issue("BUILDING_SOURCE_MISMATCH","buildings.snapshot","Bind selected records to their verified current source snapshot.")
    rows=v.array(selection.get("selected"),"buildings.selected");seen=set()
    summary=selection.get("summary")
    ledger=selection.get("reject_ledger")
    valid_counts=isinstance(summary,dict) and all(type(summary.get(k)) is int and summary[k]>=0 for k in ("selected","excluded","rejected","source_records","accounted_records"))
    if not valid_counts or summary.get("selected") != len(rows) or not isinstance(ledger,list) or summary["excluded"]+summary["rejected"]!=len(ledger) or summary["accounted_records"]!=len(rows)+len(ledger) or summary["source_records"]!=summary["accounted_records"] or (source is not None and source.get("row_count")!=summary["source_records"]):
        v.issue("BUILDING_COUNT_MISMATCH","buildings.summary","Selected count must match actual records.")
    indices=[]
    for item in [*rows,*(ledger if isinstance(ledger,list) else [])]:
        if not isinstance(item,dict) or type(item.get("source_row_index")) is not int:
            v.issue("BUILDING_ROW_IDENTITY_INVALID","buildings","Every disposition needs an explicit integer source row.")
        else:indices.append(item["source_row_index"])
    if valid_counts and sorted(indices)!=list(range(summary["source_records"])):
        v.issue("BUILDING_ROW_RECONCILIATION","buildings","Source rows must appear exactly once across selected and ledger.")
    for index,row in enumerate(rows):
        p=f"buildings.selected[{index}]"
        if not isinstance(row,dict):
            v.issue("BUILDING_RECORD_INVALID",p,"Supply a selected BuildingRecord.");continue
        key=row.get("building_id")
        if v.text(key,p+".building_id"):
            if key in seen:v.issue("BUILDING_ID_DUPLICATE",p,"Route each selected source identity once.")
            seen.add(key)
        if row.get("metric_crs") != metric_crs or row.get("analysis_run_id") != context["analysis_run_id"] or row.get("source_ref_id") != snap.get("source_ref_id") or row.get("snapshot_sha256") != (snap.get("fingerprint") or {}).get("value"):
            v.issue("BUILDING_RECORD_BINDING",p,"Record source/CRS/context must match the selected snapshot.")
        point=row.get("representative_point");coords=point.get("coordinates") if isinstance(point,dict) else None
        if not isinstance(point,dict) or point.get("type") != "Point" or not isinstance(coords,list) or len(coords)!=2 or not all(_number(x) for x in coords) or not (-180<=coords[0]<=180 and -90<=coords[1]<=90):
            v.issue("BUILDING_POINT_INVALID",p,"Supply explicit WGS84 [lon,lat] representative point from LI-1.")
        else:
            zone,south=int(metric_crs[-2:]),metric_crs.startswith('EPSG:327')
            if not (-80<=coords[1]<=84 and (coords[1]<=0 if south else coords[1]>=0) and -180+6*(zone-1)<=coords[0]<=-180+6*zone):
                v.issue("BUILDING_POINT_CRS_DOMAIN",p,"Selected point must retain LI-1's declared UTM domain.")
    return rows


def _edge_ref(edge):
    return {key:edge[key] for key in ("edge_id","u","v","key")}


def _route_geometry(tail, path, edges):
    coords=deepcopy(tail)
    for key in path["edges"]:
        geom=edges[key]["geometry"]
        if geom is None:return None,"MISSING_NETWORK_EDGE_GEOMETRY"
        coords.extend(deepcopy(geom["coordinates"][1:]))
    # A zero partial/network path does not fabricate a nonzero LineString.
    compact=[]
    for xy in coords:
        if not compact or xy!=compact[-1]:compact.append(xy)
    return ({"type":"LineString","coordinates":compact},"AVAILABLE_ROAD_GEOMETRY") if len(compact)>=2 else (None,"ZERO_ROAD_LENGTH")


def route_buildings(context: Any, manifest: Any, buildings: Any, graph: Any,
                    config: Any, *, root: Path | str) -> dict:
    result={"schema_version":RESULT_SCHEMA,"status":"BLOCKED","routing_gate":"BLOCKED",
            "accessibility_gate":"BLOCKED","unresolved_decisions":list(OPEN_DECISIONS),
            "live_readiness":"NOT_VERIFIED","routes":[],"summary":None,"diagnostics":[],
            "limitations":["N1 reference/fixture diagnostics only; no approved business accessibility.",
                           "Declared graph coverage and access permission are not real-world/legal access proof."]}
    def stop(code,message,status="FAILED"):
        result["status"]=status;result["diagnostics"].append({"code":code,"message":message});return result
    gate=validate(context,manifest,root=root);result["li0"]=gate
    if gate["offline_readiness"]!="READY":return stop("LI0_INPUT_BLOCKED","Supply valid, available current context/source inputs.","BLOCKED")
    if graph is None:return stop("GRAPH_UNAVAILABLE","Supply a local graph; it is not retrieved automatically.","BLOCKED")
    if not isinstance(config,dict) or not isinstance(graph,dict):return stop("ROUTING_INPUT_INVALID","Supply graph and routing-config JSON objects.")
    v=_Validator(Path(root))
    for name,value in (("buildings",buildings),("graph",graph),("routing_config",config)):
        v.json_values(value,name)
    if v.diagnostics:
        result["diagnostics"]=v.diagnostics;return stop("ROUTING_JSON_INVALID","Correct malformed/nonfinite routing input.")
    params=_configuration(v,config,context,buildings,graph)
    metric_crs=context["config"]["area_coverage"]["metric_crs"]
    nodes=_graph(v,graph,metric_crs)
    selected=_selection(v,buildings,context,manifest,metric_crs)
    if v.diagnostics:
        result["diagnostics"]=sorted(v.diagnostics,key=lambda x:(x["path"],x["code"]))
        return stop("ROUTING_INPUT_INVALID","Correct graph/config/selected-result contract violations.","FAILED" if any(x["severity"]=="ERROR" for x in v.diagnostics) else "BLOCKED")
    result["provenance"]={"analysis_run_id":context["analysis_run_id"],"target":deepcopy(context["target"]),
                          "context_hash":context["context_hash"],"routing_config_hash":canonical_hash(config),
                          "selection_hash":canonical_hash(buildings),"graph_hash":canonical_hash(graph),
                          "graph_id":graph["graph_id"],"graph_provenance":deepcopy(graph["provenance"]),
                          "policy_refs":deepcopy(config["policy_refs"]),"metric_crs":metric_crs,
                          "profile":config["profile"],"path_weight":config["path_weight"]}
    try:
        from location_intelligent_buildings import _backend, _spatial
        _,ogr,osr=_backend()
        with osr.ExceptionMgr(useExceptions=True):
            transform,target=_spatial(metric_crs,context["target"],ogr,osr)
            inverse=transform.GetInverse()
        target_xy=[target.GetX(),target.GetY()]
    except (ImportError,OSError,RuntimeError,ValueError):return stop("ROUTING_TRANSFORM_UNAVAILABLE","Provide the offline CRS backend validated for LI-1.","BLOCKED")
    bbox=graph["coverage"]["bbox"];edges={e["edge_id"]:e for e in graph["edges"]}
    speed=params["motorcycle_speed_kmh"]*1000/3600
    result["provenance"]["constant_speed_mps"]=speed
    parity=config["path_weight"]=="travel_time_s" and all(e["travel_time_s"]==e["length_m"]/speed for e in graph["edges"])
    result["provenance"]["n1_network_cost_model_matches"]=parity
    if not parity:
        result["diagnostics"].append({"code":"WEIGHTED_GRAPH_DIAGNOSTIC_VARIANT","message":"Explicit path weight/edge costs differ from N1's constant-speed network; reference distance/time formula is reported separately."})
    if nodes:
        snap_key=min(nodes,key=lambda key:(math.dist(target_xy,[nodes[key]["x"],nodes[key]["y"]]),key))
        snap_distance=math.dist(target_xy,[nodes[snap_key]["x"],nodes[snap_key]["y"]])
    else:snap_key,snap_distance=None,None
    for building in selected:
        row={"analysis_run_id":context["analysis_run_id"],"building_id":building["building_id"],
             "source_row_index":building.get("source_row_index"),"route_found":None,"accessible":None,
             "reference_accessible":None,"accessibility_reason":"POLICY_UNRESOLVED",
             "unresolved_decisions":list(OPEN_DECISIONS),"selected_candidate":None,
             "candidates":[],"candidate_diagnostics":[],"route_geometry":None,
             "route_geometry_crs":"EPSG:4326",
             "route_geometry_status":"NOT_EVALUATED","coverage_status":graph["coverage"]["completeness"],
             "target_connection":{"node_id":snap_key,"snap_distance_m":snap_distance,
                 "snap_status":"NOT_EVALUATED","connector_included_in_reference":False,
                 "omitted_connector_distance_m":snap_distance,
                 "omitted_connector_time_s":snap_distance/speed if snap_distance is not None else None}}
        result["routes"].append(row)
        if not inside_bbox(target_xy,bbox):row.update(reason="TARGET_OUTSIDE_COVERAGE",coverage_status="TARGET_OUTSIDE");continue
        try:
            lon,lat=building["representative_point"]["coordinates"]
            with osr.ExceptionMgr(useExceptions=True):xy=list(transform.TransformPoint(lon,lat)[:2])
        except (RuntimeError,ValueError):row["reason"]="BUILDING_TRANSFORM_FAILED";continue
        if not all(_number(x) for x in xy):row["reason"]="BUILDING_TRANSFORM_FAILED";continue
        row["building_point_m"]=xy
        if not inside_bbox(xy,bbox):row.update(reason="BUILDING_OUTSIDE_COVERAGE",coverage_status="BUILDING_OUTSIDE");continue
        if not nodes:row.update(reason="EMPTY_GRAPH",route_found=False);continue
        limit=params["max_target_snap_m"]
        if limit is None:
            row["target_connection"]["snap_status"]="UNRESOLVED";row["reason"]="TARGET_SNAP_POLICY_UNRESOLVED";continue
        if not within_snap_limit(snap_distance,limit):
            row["target_connection"]["snap_status"]="LIMIT_EXCEEDED";row["reason"]="TARGET_SNAP_LIMIT_EXCEEDED";continue
        row["target_connection"]["snap_status"]="WITHIN_DECLARED_FIXTURE_LIMIT"
        for edge in sorted(graph["edges"],key=lambda x:x["edge_id"]):
            rejection=None
            road=classify_road(edge["highway"],edge["service"])
            if road not in ("street","main street","highway"):rejection="ROAD_CLASS_INELIGIBLE"
            elif edge["geometry"] is None:rejection="ACCESS_GEOMETRY_MISSING"
            if rejection:
                row["candidate_diagnostics"].append({"edge":_edge_ref(edge),"reason":rejection});continue
            offset,fraction,access,tail=_project_to_line(xy,edge["geometry"]["coordinates"])
            if offset>params["max_building_to_road_m"]:
                row["candidate_diagnostics"].append({"edge":_edge_ref(edge),"reason":"ACCESS_DISTANCE_EXCEEDED","distance_m":offset});continue
            if edge["access_allowed"] is not True:
                row["candidate_diagnostics"].append({"edge":_edge_ref(edge),"reason":"CONNECTOR_REJECTED" if edge["access_allowed"] is False else "CONNECTOR_PERMISSION_UNKNOWN","distance_m":offset});continue
            path=shortest_route(graph,edge["v"],snap_key,config["path_weight"])
            candidate={"edge":_edge_ref(edge),"road_type":road,"connector_status":"EXPLICIT_FIXTURE_PERMISSION_NOT_PHYSICAL_PROOF",
                       "building_connector_distance_m":offset,"building_connector_time_s":offset/speed,
                       "access_point_m":access,"partial_edge_fraction":1-fraction,
                       "partial_edge_distance_m":(1-fraction)*edge["length_m"],
                       "route_found":path is not None,"network":path,"reason":"NO_DIRECTED_PATH" if path is None else "DIRECTED_PATH_FOUND"}
            row["candidates"].append(candidate)
            if path is None:continue
            total=offset+candidate["partial_edge_distance_m"]+path["distance_m"]
            total_min=total/speed/60
            candidate.update(partial_edge_reference_time_s=candidate["partial_edge_distance_m"]/speed,
                network_reference_time_s=path["distance_m"]/speed,total_road_distance_m=total,
                total_reference_time_min=total_min,threshold_time_min=round(total_min,3),
                reference_accessible=reference_time_pass(total_min,params["max_travel_min"]),
                diagnostic_distance_including_target_m=total+snap_distance,
                diagnostic_time_including_target_min=(total+snap_distance)/speed/60,
                graph_cost_time_s=offset/speed+(1-fraction)*edge["travel_time_s"]+path["time_s"])
            metric_geometry,candidate["geometry_status"]=_route_geometry(tail,path,edges)
            candidate["road_coordinates_m"]=metric_geometry["coordinates"] if metric_geometry is not None else None
            candidate["road_geometry"]=None
            if metric_geometry is not None:
                try:
                    with osr.ExceptionMgr(useExceptions=True):
                        geo=[list(inverse.TransformPoint(*xy)[:2]) for xy in metric_geometry["coordinates"]]
                    if not all(all(_number(x) for x in xy) for xy in geo):raise ValueError("Nonfinite inverse transform")
                    candidate["road_geometry"]={"type":"LineString","coordinates":geo}
                except (RuntimeError,ValueError):
                    candidate["geometry_status"]="ROUTE_GEOMETRY_TRANSFORM_FAILED"
        reachable=[c for c in row["candidates"] if c["route_found"]]
        if not reachable:
            reason="NO_DIRECTED_PATH" if row["candidates"] else "NO_ELIGIBLE_ACCESS"
            if not row["candidates"] and any(d["reason"] in ("CONNECTOR_REJECTED","CONNECTOR_PERMISSION_UNKNOWN") for d in row["candidate_diagnostics"]):reason="CONNECTOR_VALIDATION_FAILED"
            row.update(reason=reason,route_found=False);continue
        best=min(reachable,key=lambda c:(round(c["building_connector_distance_m"],2),c["threshold_time_min"],c["edge"]["u"],c["edge"]["v"],c["edge"]["key"],c["edge"]["edge_id"]))
        row.update(route_found=True,selected_candidate=deepcopy(best),reference_accessible=best["reference_accessible"],
                   route_geometry=best["road_geometry"],route_geometry_status=best["geometry_status"],
                   reason="REFERENCE_WITHIN_TIME" if best["reference_accessible"] else "REFERENCE_TIME_LIMIT")
    counts=Counter(row["reason"] for row in result["routes"])
    result["summary"]={"input_selected_buildings":len(selected),"accounted_routes":len(result["routes"]),
                       "route_found":sum(r["route_found"] is True for r in result["routes"]),
                       "no_route":sum(r["route_found"] is False for r in result["routes"]),
                       "not_evaluated":sum(r["route_found"] is None for r in result["routes"]),
                       "reference_within_time":sum(r["reference_accessible"] is True for r in result["routes"]),
                       "reference_time_limit":sum(r["reference_accessible"] is False for r in result["routes"]),
                       "accepted_accessible":None,"policy_unresolved":len(selected),"reason_counts":dict(sorted(counts.items()))}
    assert result["summary"]["accounted_routes"]==len(selected)
    result["routing_gate"]="PASS"
    result["status"]="PARTIAL"
    result["diagnostics"].append({"code":"POLICY_UNRESOLVED","message":"D02/D03/D05/D10 remain OPEN; reference counts are not approved accessibility."})
    return result


def main(argv: list[str] | None = None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    for key in ("context","manifest","buildings","graph","config","root"):
        parser.add_argument("--"+key,required=True,type=Path)
    args=parser.parse_args(argv)
    loaded=[];file_hashes={}
    try:
        for key in ("context","manifest","buildings","graph","config"):
            path=getattr(args,key)
            if not path.is_file():raise OSError("Regular local inputs required")
            raw=path.read_bytes();file_hashes[key]=hashlib.sha256(raw).hexdigest()
            loaded.append(json.loads(raw.decode("utf-8"),object_pairs_hook=_strict_pairs,parse_constant=_reject_constant))
        result=route_buildings(*loaded,root=args.root)
        result["input_file_sha256"]=file_hashes
    except OSError:
        result={"schema_version":RESULT_SCHEMA,"status":"BLOCKED","summary":None,"diagnostics":[{"code":"INPUT_UNAVAILABLE","message":"Supply readable regular local inputs; no network fallback."}],"live_readiness":"NOT_VERIFIED"}
    except (ValueError,UnicodeError,RecursionError):
        result={"schema_version":RESULT_SCHEMA,"status":"FAILED","summary":None,"diagnostics":[{"code":"INPUT_JSON_INVALID","message":"Supply strict UTF-8 JSON inputs."}],"live_readiness":"NOT_VERIFIED"}
    print(json.dumps(result,sort_keys=True,indent=2,ensure_ascii=False,allow_nan=False))
    return {"PARTIAL":3,"BLOCKED":2,"FAILED":1}.get(result["status"],0)


if __name__=="__main__":
    raise SystemExit(main())


### location_intelligent_motor_routing.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_motor_routing.py
"""Policy-aware offline motorcycle routing on explicit, hash-bound evidence.

No entrance/bridge/connector is invented. Connectors and resolved restrictions
must be supplied by a source adapter or review. N1 engine remains unchanged.
"""
from copy import deepcopy
import argparse
import heapq
import json
import math
from pathlib import Path

from location_intelligent_config import get_config
from location_intelligent_routing import _graph, _selection, _number, inside_bbox
from location_intelligent_validator import _Validator, canonical_hash, validate


def motor_access(tags, highway):
    """Conservative normalized tag resolver, not a conditional-expression parser."""
    if not isinstance(tags, dict) or any(not isinstance(k, str) or not isinstance(v, str) for k, v in tags.items()):
        return None, "MALFORMED_TAGS"
    if any(":conditional" in k for k in tags):
        return None, "CONDITIONAL_REQUIRES_RESOLVED_EVIDENCE"
    if highway in {"steps", "footway", "pedestrian", "cycleway"}:
        return False, "NON_MOTOR_PROFILE"
    values = [tags.get(k, "").lower() for k in ("access", "vehicle", "motor_vehicle", "motorcycle")]
    if "private" in values or tags.get("gate:status") == "closed":
        return False, "EXPLICIT_PROHIBITION"
    for value in reversed(values):
        if value == 'no':
            return False, 'EXPLICIT_PROHIBITION'
        if value in {'yes', 'designated', 'permissive'}:
            return True, 'POSITIVE_TAG_PERMISSION'
        if value:
            return None, 'ACCESS_VALUE_REQUIRES_REVIEW'
    if highway in {"service", "living_street", "residential", "unclassified"}:
        return True, "OWNER_ALLEY_ACCESS_ASSUMPTION"
    return None, "ACCESS_UNVERIFIED"


def _path(graph, allowed, blocked, start, target):
    """Edge-state Dijkstra enforces prohibited turns without reversing roads."""
    adjacency = {n["node_id"]: [] for n in graph["nodes"]}
    for edge in graph["edges"]:
        if allowed.get(edge["edge_id"]) is True:
            adjacency[edge["u"]].append(edge)
    queue, best = [(0.0, 0, (), start, "")], {(start, ""): (0.0, 0, ())}
    while queue:
        cost, hops, ids, node, previous = heapq.heappop(queue)
        if best.get((node, previous)) != (cost, hops, ids):
            continue
        if node == target:
            return {"distance_m": cost, "edges": list(ids)}
        for edge in sorted(adjacency[node], key=lambda e: e["edge_id"]):
            key = edge["edge_id"]
            if (previous, key) in blocked:
                continue
            label = (cost + edge["length_m"], hops + 1, (*ids, key))
            state = (edge["v"], key)
            if state not in best or label < best[state]:
                best[state] = label
                heapq.heappush(queue, (*label, *state))
    return None


def _path_index(graph, allowed, blocked, target):
    """Reverse edge-state index; same cost/hops/edge-ID tie-break as _path.

    Built once per target per invocation. It never crosses analysis/config
    boundaries and does not drop turn restrictions to optimize real graphs.
    """
    incoming, outgoing, edges = {}, {}, {}
    for edge in graph['edges']:
        key = edge['edge_id']
        if allowed.get(key) is True:
            edges[key] = edge
            incoming.setdefault(edge['v'], []).append(key)
            outgoing.setdefault(edge['u'], []).append(key)
    best, queue = {}, []
    for key in incoming.get(target, []):
        label = (edges[key]['length_m'], 1, (key,))
        best[key] = label
        heapq.heappush(queue, (*label, key))
    while queue:
        cost, hops, path, key = heapq.heappop(queue)
        if best.get(key) != (cost, hops, path):
            continue
        for previous in incoming.get(edges[key]['u'], []):
            if (previous, key) in blocked:
                continue
            label = (edges[previous]['length_m'] + cost, hops+1, (previous, *path))
            if previous not in best or label < best[previous]:
                best[previous] = label
                heapq.heappush(queue, (*label, previous))
    def lookup(start):
        if start == target:
            return {'distance_m': 0.0, 'edges': []}
        options = [best[k] for k in outgoing.get(start, []) if k in best]
        if not options:
            return None
        label = min(options)
        # Report forward-ordered summation, matching legacy _path arithmetic.
        return {'distance_m': sum(edges[k]['length_m'] for k in label[2]), 'edges': list(label[2])}
    return lookup


def _potential_reachability(graph, allowed, blocked, target):
    """Unknown edges may explain a no-route result; never used for acceptance."""
    edges, incoming, outgoing = {}, {}, {}
    for edge in graph['edges']:
        if allowed.get(edge['edge_id']) is not False:
            key = edge['edge_id']; edges[key] = edge
            incoming.setdefault(edge['v'], []).append(key)
            outgoing.setdefault(edge['u'], []).append(key)
    reached = set(incoming.get(target, [])); queue = list(reached)
    while queue:
        key = queue.pop()
        for previous in incoming.get(edges[key]['u'], []):
            if previous not in reached and (previous, key) not in blocked:
                reached.add(previous); queue.append(previous)
    return lambda start: start == target or any(k in reached for k in outgoing.get(start, []))


def route_motor_buildings(context, manifest, buildings, graph, evidence, *, root, policy=None):
    """Integrate editable policy with verified local sources; no live fallback.

    Evidence schema li_motor_evidence_v1 binds hashes to context/selection/graph.
    edge_rules resolves all edge permissions with evidence_refs. blocked_turns
    lists explicit (from_edge,to_edge). connectors are metric LineStrings with
    kind BUILDING/TARGET, building_id, node_id, road_edge_id, status
    VERIFIED/OWNER_ASSUMPTION/BLOCKED/UNKNOWN, barrier_clear and evidence_refs.
    Incomplete restriction coverage keeps routing blocked rather than bypassed.
    """
    policy = get_config() if policy is None else deepcopy(policy)
    result = {"schema_version": "li_motor_routing_v1", "status": "BLOCKED", "routes": [],
              "summary": None, "diagnostics": [], "live_readiness": "NOT_VERIFIED",
              "physical_access_validated": False}
    def stop(code, status="BLOCKED"):
        result["status"] = status
        result["diagnostics"].append({"code": code})
        return result
    gate = validate(context, manifest, root=root)
    if gate["offline_readiness"] != "READY":
        return stop("LI0_INPUT_BLOCKED")
    if not isinstance(evidence, dict) or not isinstance(graph, dict):
        return stop("MOTOR_EVIDENCE_REQUIRED")
    v = _Validator(root)
    for name, value in [("graph", graph), ("buildings", buildings), ("policy", policy), ("evidence", evidence)]:
        v.json_values(value, name)
    if v.diagnostics:
        return stop("NONFINITE_OR_MALFORMED_INPUT", "FAILED")
    try:
        travel, conn = policy["travel"], policy["connectors"]
        speed = travel["reference_constant_speed"]["value"] * 1000 / 3600
        limit = conn["max_building_to_road_m"]["value"]
        threshold = travel["max_travel_min"]["value"]
        if (not _number(travel["reference_constant_speed"]["value"]) or
            not all(_number(x) and x > 0 for x in (speed, limit, threshold)) or
            any(policy["motorcycle_access"].get(k) is not False for k in ("private_access_usable", "pedestrian_only_usable", "stairs_usable", "invent_u_turns_or_reverse_edges")) or
            travel["reference_constant_speed"]["unit"] != "km/h" or
            conn["max_building_to_road_m"]["unit"] != "m" or travel["max_travel_min"]["unit"] != "min" or
            travel["mode"] != "motorcycle" or travel["threshold_comparison"] != "<=" or
            travel["threshold_rounding"] != "PYTHON_ROUND_HALF_EVEN_TOTAL_MINUTES_BEFORE_COMPARISON" or
            travel["segment_speed_model"] != "ALL_SEGMENTS_USE_REFERENCE_CONSTANT_SPEED" or
            conn["candidate_selection"] != "HIGHEST_VALID_ACCESS_CLASS_THEN_MINIMUM_TOTAL_TIME" or
            conn["max_target_snap_m"] != {"value": None, "unit": "m", "limit_type": "UNBOUNDED"} or
            conn["unreachable_higher_class"] != "TRY_NEXT_REACHABLE_CLASS"):
            return stop("UNSUPPORTED_MOTOR_POLICY", "FAILED")
        rank = {name: i for i, group in enumerate(conn["road_class_hierarchy"]) for name in group}
        if not rank or len(rank) != sum(map(len, conn["road_class_hierarchy"])):
            return stop("INVALID_ROAD_HIERARCHY", "FAILED")
    except (KeyError, TypeError, ValueError):
        return stop("INVALID_MOTOR_POLICY", "FAILED")
    crs = context["config"]["area_coverage"]["metric_crs"]
    nodes = _graph(v, graph, crs)
    selected = _selection(v, buildings, context, manifest, crs)
    if v.diagnostics:
        result["diagnostics"] = v.diagnostics
        return stop("INPUT_CONTRACT_INVALID", "FAILED")
    hashes = {"graph_hash": canonical_hash(graph), "selection_hash": canonical_hash(buildings), "context_hash": context["context_hash"], "policy_hash": canonical_hash(policy)}
    result["provenance"] = {**hashes, "effective_policy": policy, "graph_id": graph["graph_id"]}
    if evidence.get("schema_version") != "li_motor_evidence_v1" or any(evidence.get(k) != val for k, val in hashes.items()):
        return stop("EVIDENCE_BINDING_MISMATCH", "FAILED")
    edges = {e["edge_id"]: e for e in graph["edges"]}
    rules = evidence.get("edge_rules")
    if not isinstance(rules, dict) or set(rules) != set(edges):
        return stop("EDGE_PERMISSION_COVERAGE_INCOMPLETE")
    allowed = {}
    for key, rule in rules.items():
        if (not isinstance(rule, dict) or type(rule.get("allowed")) not in (bool, type(None)) or
            not isinstance(rule.get("evidence_refs"), list) or not rule["evidence_refs"] or
            not all(isinstance(x, str) and x for x in rule["evidence_refs"])):
            return stop("INVALID_EDGE_EVIDENCE", "FAILED")
        tags = rule.get("raw_tags", {})
        derived, reason = motor_access(tags, edges[key]["highway"])
        # Positive review cannot override explicit motor/private restrictions.
        if derived is False and rule["allowed"] is True:
            return stop("PERMISSION_OVERRIDES_PROHIBITION", "FAILED")
        if not isinstance(tags, dict) or any(not isinstance(k, str) or not isinstance(val, str) for k, val in tags.items()):
            return stop("MALFORMED_RAW_TAGS", "FAILED")
        if any(":conditional" in k for k in tags) and rule.get("conditional_resolved") is not True and rule['allowed'] is True:
            return stop("CONDITIONAL_RESTRICTION_UNRESOLVED")
        allowed[key] = rule["allowed"]
    if evidence.get("restrictions_complete") is not True:
        return stop("TURN_OR_BARRIER_COVERAGE_UNRESOLVED")
    turns = evidence.get("blocked_turns")
    if not isinstance(turns, list):
        return stop("INVALID_TURNS", "FAILED")
    blocked = set()
    for turn in turns:
        if (not isinstance(turn, dict) or turn.get("from_edge") not in edges or turn.get("to_edge") not in edges or
            not turn.get("evidence_ref") or edges[turn["from_edge"]]["v"] != edges[turn["to_edge"]]["u"]):
            return stop("INVALID_TURN_REFERENCE", "FAILED")
        blocked.add((turn["from_edge"], turn["to_edge"]))
    try:
        from location_intelligent_buildings import _backend, _spatial
        _, ogr, osr = _backend()
        transform, target = _spatial(crs, context["target"], ogr, osr)
        target_xy = [target.GetX(), target.GetY()]
    except (ImportError, RuntimeError, ValueError):
        return stop("CRS_BACKEND_UNAVAILABLE")
    if not inside_bbox(target_xy, graph["coverage"]["bbox"]):
        return stop("TARGET_OUTSIDE_COVERAGE")
    by_id = {b["building_id"]: b for b in selected}
    sources, targets, ids, building_geometry_cache = [], [], set(), {}
    if not isinstance(evidence.get("connectors"), list):
        return stop("INVALID_CONNECTOR_COLLECTION", "FAILED")
    for c in evidence["connectors"]:
        if not isinstance(c, dict):
            return stop("INVALID_CONNECTOR", "FAILED")
        key, node, edge = c.get("connector_id"), c.get("node_id"), edges.get(c.get("road_edge_id"))
        geom = c.get("geometry") if isinstance(c.get("geometry"), dict) else {}
        coords = geom.get("coordinates")
        if (not isinstance(key, str) or not key or key in ids or node not in nodes or edge is None or
            node not in {edge["u"], edge["v"]} or c.get("kind") not in {"BUILDING", "TARGET"} or
            c.get("status") not in {"VERIFIED", "OWNER_ASSUMPTION", "BLOCKED", "UNKNOWN"} or
            type(c.get("barrier_clear")) not in (bool, type(None)) or
            geom.get("type") != "LineString" or not isinstance(coords, list) or len(coords) < 2 or
            not all(isinstance(p, list) and len(p) == 2 and all(_number(x) for x in p) for p in coords) or
            not isinstance(c.get("evidence_refs"), list) or not c["evidence_refs"]):
            return stop("INVALID_CONNECTOR", "FAILED")
        ids.add(key)
        if not all(inside_bbox(p, graph["coverage"]["bbox"]) for p in coords):
            return stop("CONNECTOR_OUTSIDE_COVERAGE")
        node_xy = [nodes[node]["x"], nodes[node]["y"]]
        length = sum(math.dist(a, b) for a, b in zip(coords, coords[1:]))
        if not math.isfinite(length):
            return stop("CONNECTOR_DISTANCE_INVALID", "FAILED")
        r = deepcopy(c)
        r["distance_m"] = length
        r['structural_usable'] = c['status'] in {'VERIFIED', 'OWNER_ASSUMPTION'} and c['barrier_clear'] is True
        r['permission'] = allowed[edge['edge_id']]
        r["usable"] = c["status"] in {"VERIFIED", "OWNER_ASSUMPTION"} and c["barrier_clear"] is True and allowed[edge["edge_id"]] is True
        if c["kind"] == "BUILDING":
            if c.get("building_id") not in by_id or coords[-1] != node_xy:
                return stop("CONNECTOR_SOURCE_MISMATCH", "FAILED")
            b = by_id[c["building_id"]]
            if b['building_id'] not in building_geometry_cache:
                g = ogr.CreateGeometryFromJson(json.dumps(b["geometry"]))
                g.Transform(transform)
                rep = transform.TransformPoint(*b["representative_point"]["coordinates"])[:2]
                building_geometry_cache[b['building_id']] = (g, rep)
            g, rep = building_geometry_cache[b['building_id']]
            p = ogr.Geometry(ogr.wkbPoint); p.AddPoint_2D(*coords[0])
            if not g.Intersects(p):
                return stop("ENTRANCE_OUTSIDE_BUILDING", "FAILED")
            offset = math.dist(rep, node_xy)
            # Candidate search measures point-to-road, not connector detour length.
            if edge["geometry"] is None:
                r["usable"] = False
            else:
                from location_intelligent_routing import _project_to_line
                offset = _project_to_line(rep, edge["geometry"]["coordinates"])[0]
            r["candidate_distance_m"] = offset
            h = edge["highway"] or ""
            h = h[:-5] if h.endswith('_link') else h
            r["class_rank"] = rank.get(h)
            r["usable"] &= offset <= limit and r["class_rank"] is not None
            r['structural_usable'] &= edge['geometry'] is not None and offset <= limit and r['class_rank'] is not None
            sources.append(r)
        else:
            if coords[0] != node_xy or coords[-1] != target_xy:
                return stop("TARGET_CONNECTOR_MISMATCH", "FAILED")
            targets.append(r)
    indexes = {c['node_id']: _path_index(graph, allowed, blocked, c['node_id']) for c in targets if c['usable']}
    potential = {c['node_id']: _potential_reachability(graph, allowed, blocked, c['node_id']) for c in targets
                 if c['structural_usable'] and c['permission'] is not False} if any(v is None for v in allowed.values()) else {}
    sources_by_building = {}
    for source in sources:
        sources_by_building.setdefault(source['building_id'], []).append(source)
    for b in selected:
        row = {"building_id": b["building_id"], "route_found": False, "scenario_accessible": False,
               "physical_access_validated": False, "candidates": [], "selected_candidate": None}
        result["routes"].append(row)
        for source in sources_by_building.get(b['building_id'], []):
            if not source["usable"]:
                continue
            for target_conn in targets:
                if not target_conn["usable"]:
                    continue
                path = indexes[target_conn['node_id']](source['node_id'])
                if path is None:
                    continue
                total = source["distance_m"] + path["distance_m"] + target_conn["distance_m"]
                if not math.isfinite(total):
                    return stop("TOTAL_DISTANCE_OVERFLOW", "FAILED")
                minutes = total / speed / 60
                row["candidates"].append({"building_connector_id": source["connector_id"], "target_connector_id": target_conn["connector_id"],
                    "class_rank": source["class_rank"], "network_edges": path["edges"],
                    "building_connector_m": source["distance_m"], "network_distance_m": path["distance_m"],
                    "target_connector_m": target_conn["distance_m"], "total_distance_m": total,
                    "total_time_min": minutes, "threshold_time_min": round(minutes), "within_time": round(minutes) <= threshold})
        if row["candidates"]:
            best = min(row["candidates"], key=lambda c: (c["class_rank"], c["total_time_min"], c["building_connector_id"], c["target_connector_id"], tuple(c["network_edges"])))
            row.update(route_found=True, selected_candidate=best, scenario_accessible=best["within_time"],
                       reason="SCENARIO_WITHIN_TIME" if best["within_time"] else "SCENARIO_TIME_LIMIT")
        else:
            source_rows = [s for s in sources if s["building_id"] == b["building_id"]]
            unresolved = (not source_rows or not targets or
                          any(c["status"] == "UNKNOWN" or c["barrier_clear"] is None for c in [*source_rows, *targets]))
            if not unresolved and potential:
                unresolved = any(source['structural_usable'] and source['permission'] is not False and lookup(source['node_id'])
                                 for source in source_rows for lookup in potential.values())
            row["reason"] = "EVIDENCE_INCOMPLETE" if unresolved else "NO_VALID_REACHABLE_CONNECTORS"
            if unresolved:
                row.update(route_found=None, scenario_accessible=None)
    unknown = any(v is None for v in allowed.values())
    result["status"] = "PARTIAL" if unknown or any(r["route_found"] is None for r in result["routes"]) or graph["coverage"]["completeness"] != "COMPLETE" or buildings["status"] == "PARTIAL" else "SUCCEEDED"
    result["summary"] = {"input_selected": len(selected), "accounted": len(result["routes"]),
                         "scenario_accepted": sum(r["scenario_accessible"] is True for r in result["routes"]),
                         "no_route": sum(r["route_found"] is False for r in result["routes"]),
                         "not_evaluated": sum(r["route_found"] is None for r in result["routes"]),
                         "unknown_edge_permissions": sum(v is None for v in allowed.values())}
    result["limitations"] = ["Offline scenario on explicit evidence; not proof of real/legal access", "No automatic entrance/barrier/conditional parser or source retrieval", "Node-bound connectors only; adapters must split edges for interior access points"]
    return result


def main(argv=None):
    from location_intelligent_validator import _strict_pairs, _reject_constant
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ('context', 'manifest', 'buildings', 'graph', 'evidence', 'root'):
        parser.add_argument('--'+name, type=Path, required=True)
    parser.add_argument('--policy', type=Path)
    args = parser.parse_args(argv)
    def read(p):
        if not p.is_file():
            raise ValueError('Use regular local input files')
        return json.loads(p.read_text(), object_pairs_hook=_strict_pairs, parse_constant=_reject_constant)
    try:
        r = route_motor_buildings(*(read(getattr(args, k)) for k in ('context', 'manifest', 'buildings', 'graph', 'evidence')),
                                  root=args.root, policy=read(args.policy) if args.policy else None)
    except (ValueError, OSError, UnicodeError) as e:
        r = {'schema_version': 'li_motor_routing_v1', 'status': 'FAILED', 'summary': None,
             'diagnostics': [{'code': 'INPUT_READ_FAILED', 'message': str(e)}]}
    print(json.dumps(r, sort_keys=True, indent=2, allow_nan=False))
    return {'SUCCEEDED': 0, 'PARTIAL': 3, 'BLOCKED': 2, 'FAILED': 1}[r['status']]


if __name__ == '__main__':
    raise SystemExit(main())


### location_intelligent_overture_probe.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_overture_probe.py
"""Explicit bounded source probe, not a general LI-3 acquisition service.

Run using the isolated DuckDB environment; no network on import. Output raw
rows are normalized separately with system Python/GDAL. Never overwrite runs.
"""
import argparse
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
from pathlib import Path
import time
import urllib.request


RELEASE = '2026-10-21.0'
CATALOG = f'https://stac.overturemaps.org/{RELEASE}/buildings/building/collection.json'


def _read(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'LI-bounded-source-probe/1'})
    with urllib.request.urlopen(req, timeout=30) as response:
        data = response.read(4_000_001)
    if len(data) > 4_000_000:
        raise ValueError('Metadata response too large')
    return json.loads(data), hashlib.sha256(data).hexdigest()


def probe(bbox, row_limit, output, extension_directory):
    import duckdb
    if len(bbox) != 4 or not (-180 <= bbox[0] < bbox[2] <= 180 and -90 <= bbox[1] < bbox[3] <= 90) or bbox[2]-bbox[0] > .02 or bbox[3]-bbox[1] > .02:
        raise ValueError('Use an explicit small geographic bbox (<=.02 degrees per side)')
    if not 1 <= row_limit <= 10001:
        raise ValueError('Bound rows to1..10001')
    if output.exists():
        raise ValueError('Use a new output path; retained evidence must not be overwritten')
    collection, collection_hash = _read(CATALOG)
    boxes = collection['extent']['spatial']['bbox']
    links = [x for x in collection['links'] if x['rel'] == 'item']
    if len(boxes) != len(links)+1:
        raise ValueError('STAC partition extent layout changed')
    def intersects(b):
        return b[0] <= bbox[2] and b[2] >= bbox[0] and b[1] <= bbox[3] and b[3] >= bbox[1]
    indices = [i for i, b in enumerate(boxes[1:]) if intersects(b)]
    if not 1 <= len(indices) <= 4:
        raise ValueError('Probe scope requires1..4 candidate partitions')
    assets, items = [], []
    for i in indices:
        item, digest = _read(links[i]['href'])
        if not intersects(item['bbox']):
            raise ValueError('Item and indexed extent disagree')
        asset = item['assets']['aws']['href']
        if not asset.startswith(f'https://overturemaps-us-west-2.s3.us-west-2.amazonaws.com/release/{RELEASE}/theme=buildings/type=building/'):
            raise ValueError('Asset must belong to requested official release/theme')
        assets.append(asset)
        items.append({'url': links[i]['href'], 'metadata_sha256': digest, 'bbox': item['bbox'], 'asset': asset})
    c = duckdb.connect()
    c.execute("SET extension_directory = ?", [str(extension_directory)])
    c.execute('LOAD httpfs')
    for statement in ['SET enable_geoparquet_conversion=false', 'SET http_timeout=30', 'SET http_retries=1', 'SET threads=2', "SET memory_limit='256MB'"]:
        c.execute(statement)
    descriptor = c.execute('DESCRIBE SELECT * FROM read_parquet(?)', [assets]).fetchall()
    required = {'id': 'VARCHAR', 'geometry': 'BLOB', 'class': 'VARCHAR', 'subtype': 'VARCHAR', 'height': 'DOUBLE', 'num_floors': 'INTEGER', 'version': 'INTEGER'}
    actual = {r[0]: r[1] for r in descriptor}
    if any(actual.get(k) != val for k, val in required.items()) or 'bbox' not in actual or 'sources' not in actual:
        raise ValueError('Physical Parquet schema incompatible with expected source fields')
    sql = ('SELECT id, class, subtype, height, num_floors, version, sources, bbox, hex(geometry) AS geometry_wkb_hex '
           'FROM read_parquet(?) WHERE bbox.xmin <= ? AND bbox.xmax >= ? AND bbox.ymin <= ? AND bbox.ymax >= ? '
           'ORDER BY id LIMIT ?')
    start = time.monotonic()
    cursor = c.execute(sql, [assets, bbox[2], bbox[0], bbox[3], bbox[1], row_limit])
    cols = [x[0] for x in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    result = {'release': RELEASE, 'schema_version': '2.0.0', 'collection_url': CATALOG,
              'collection_sha256': collection_hash, 'items': items, 'bbox': bbox, 'sql': sql,
              'row_cap': row_limit, 'truncated': len(rows) == row_limit, 'physical_schema': descriptor,
              'schema_validation_scope': 'Consumed Parquet columns/types and downstream canonical geometry; not full official schema validation',
              'coverage_complete': False, 'coverage_reason': 'Bounded technical probe; partition candidate completeness not independently established',
              'duckdb_version': duckdb.__version__, 'elapsed_s': time.monotonic()-start, 'rows': rows}
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('x') as f:
        json.dump(result, f, sort_keys=True, ensure_ascii=False, allow_nan=False)
    return result


def normalize(raw):
    """Map provider subtype->canonical subclass without guessing taxonomy."""
    from osgeo import ogr
    if raw.get('release') != RELEASE or raw.get('schema_version') != '2.0.0':
        raise ValueError('Wrong release/schema')
    features = []
    for row in raw['rows']:
        if not {'id', 'class', 'subtype', 'height', 'num_floors', 'geometry_wkb_hex', 'version', 'sources'} <= row.keys():
            raise ValueError('Missing provider columns')
        geom = ogr.CreateGeometryFromWkb(bytes.fromhex(row['geometry_wkb_hex']))
        if geom is None:
            raise ValueError('Unreadable source WKB')
        features.append({'type': 'Feature', 'id': row['id'], 'geometry': json.loads(geom.ExportToJson()),
                         'properties': {'building_id': row['id'], 'class': row['class'], 'subclass': row['subtype'],
                                        'height': row['height'], 'num_floors': row['num_floors'],
                                        'source_feature_version': row['version'], 'sources': row['sources']}})
    return {'type': 'FeatureCollection', 'features': features}


def verify_partitions(bbox, output):
    """Independently read every advertised item bbox, not extent-order alone."""
    if output.exists():
        raise ValueError('Use a new partition verification output')
    collection, digest = _read(CATALOG)
    links = [x['href'] for x in collection['links'] if x['rel'] == 'item']
    def fetch(url):
        item, sha = _read(url)
        return {'url': url, 'bbox': item['bbox'], 'asset': item['assets']['aws']['href'], 'metadata_sha256': sha}
    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(pool.map(fetch, links))
    matches = [r for r in rows if r['bbox'][0] <= bbox[2] and r['bbox'][2] >= bbox[0] and r['bbox'][1] <= bbox[3] and r['bbox'][3] >= bbox[1]]
    result = {'release': RELEASE, 'collection_sha256': digest, 'items_checked': len(rows),
              'errors': [], 'matching': matches, 'items': rows}
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('x') as f:
        json.dump(result, f, sort_keys=True, allow_nan=False)
    return result


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--bbox', nargs=4, required=True, type=float)
    parser.add_argument('--row-limit', type=int, default=10001)
    parser.add_argument('--output', required=True, type=Path)
    parser.add_argument('--extension-directory', required=True, type=Path)
    parser.add_argument('--verify-partitions-output', type=Path)
    args = parser.parse_args()
    r = probe(args.bbox, args.row_limit, args.output, args.extension_directory)
    if args.verify_partitions_output:
        verify_partitions(args.bbox, args.verify_partitions_output)
    print(json.dumps({'rows': len(r['rows']), 'truncated': r['truncated'], 'elapsed_s': r['elapsed_s'], 'output': str(args.output)}))


### location_intelligent_li3_sources.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_li3_sources.py
"""Explicit LI-3 acquisition/cache I/O. No network at import or during replay."""
from datetime import datetime, timezone
import gzip
import hashlib
import json
from pathlib import Path


def utc():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')


def canonical_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode()


def write_json(path, value, *, exclusive=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = canonical_bytes(value)
    if path.suffix == '.gz':
        data = gzip.compress(data, mtime=0)
    with path.open('xb' if exclusive else 'wb') as stream:
        stream.write(data)
    return hashlib.sha256(data).hexdigest()


def read_json(path):
    data = Path(path).read_bytes()
    if str(path).endswith('.gz'):
        data = gzip.decompress(data)
    return json.loads(data)


def build_queries(bbox, policy):
    """Pinned OSMnx filter plus explicit motor-only supplements, raw topology."""
    import osmnx as ox
    from osmnx._overpass import _get_network_filter
    if ox.__version__ != policy['sources']['roads_access_barriers_turns']['planned_library_version']:
        raise ValueError('OSMnx version differs from frozen source plan')
    if policy['li3']['network_type'] != 'drive_service':
        raise ValueError('Only approved drive_service acquisition is supported')
    west, south, east, north = bbox
    region = f'({south},{west},{north},{east})'
    filters = [_get_network_filter('drive_service')]
    if policy['li3']['motor_supplements']:
        filters += ['["highway"~"^(track|path)$"]["motorcycle"~"^(yes|designated|permissive)$"]',
                    '["highway"~"^(track|path)$"]["motor_vehicle"~"^(yes|designated|permissive)$"]',
                    '["highway"]["motorcar"="no"]["motorcycle"~"^(yes|designated|permissive)$"]']
    settings = f'[out:json][timeout:{policy["li3"]["request_timeout_s"]}][maxsize:{policy["li3"]["max_response_bytes"]}];'
    roads = settings + '(' + ''.join('way'+f+region+';' for f in filters) + ');(._;>;);out meta;'
    features = settings + ('(node["barrier"]'+region+';way["barrier"]'+region+';way["waterway"]'+region+
        ';way["natural"="water"]'+region+';relation["natural"="water"]'+region+
        ';relation["type"="restriction"]'+region+';);(._;>>;);out meta;')
    return {'roads': roads, 'features': features, 'filters': filters}


def acquire_overpass(query, endpoints, timeout, max_bytes, session=None):
    """HTTP failure/remark/incomplete response never becomes valid empty data."""
    import requests
    if session is None:
        session = requests.Session()
        session.headers.update({'User-Agent': 'LocationIntelligent-LI3-Research/1.0 (+https://github.com/unChris666/traffic-vehicle-counter-IDM-LOC)'})
    attempts = []
    for endpoint in endpoints:
        attempt = {'endpoint': endpoint, 'started_at': utc(), 'query_sha256': hashlib.sha256(query.encode()).hexdigest()}
        attempts.append(attempt)
        try:
            response = session.post(endpoint, data={'data': query}, timeout=timeout+5, stream=True)
            attempt['http_status'] = response.status_code
            if response.status_code != 200:
                attempt['status'] = 'HTTP_ERROR'
                response.close()
                continue
            chunks, size = [], 0
            for block in response.iter_content(65536):
                size += len(block)
                if size > max_bytes:
                    response.close()
                    raise ValueError('RESPONSE_BYTE_LIMIT_EXCEEDED')
                chunks.append(block)
            response.close()
            raw = b''.join(chunks)
            payload = json.loads(raw)
            if payload.get('remark') or not isinstance(payload.get('elements'), list) or not isinstance(payload.get('osm3s'), dict):
                attempt.update(status='PARTIAL_OR_INVALID', remark=payload.get('remark'))
                continue
            attempt.update(status='SUCCEEDED', finished_at=utc(), response_bytes=size,
                           response_sha256=hashlib.sha256(raw).hexdigest(),
                           source_as_of=payload['osm3s'].get('timestamp_osm_base'), generator=payload.get('generator'))
            return {'status': 'SUCCEEDED', 'payload': payload, 'attempts': attempts}
        except (OSError, ValueError, RuntimeError) as exc:
            attempt.update(status='REQUEST_ERROR', error_type=type(exc).__name__, error=str(exc)[:300])
        except requests.RequestException as exc:
            attempt.update(status='REQUEST_ERROR', error_type=type(exc).__name__, error=str(exc)[:300])
    return {'status': 'RETRIEVAL_FAILED', 'payload': None, 'attempts': attempts}


### location_intelligent_li3_graph.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_li3_graph.py
"""OSM raw topology -> existing directed graph/evidence, never geometric joins."""
from collections import Counter, defaultdict
from copy import deepcopy
import math

from location_intelligent_motor_routing import motor_access


def node_permission(tags):
    barrier = tags.get('barrier')
    if not barrier:
        return True, 'NO_MAPPED_NODE_BARRIER'
    if tags.get('access') == 'private' or tags.get('motorcycle') == 'no' or tags.get('locked') == 'yes' or tags.get('gate:status') == 'closed':
        return False, 'CLOSED_OR_PROHIBITED_BARRIER'
    if any(':conditional' in k for k in tags) or (tags.get('opening_hours') and tags['opening_hours'] != '24/7'):
        return None, 'BARRIER_CONDITION_UNEVALUATED'
    if barrier in {'gate', 'lift_gate', 'swing_gate', 'chain'}:
        value, _ = motor_access(tags, 'service')
        return value, 'OWNER_PORTAL_OPEN_ASSUMPTION' if value else 'PORTAL_PERMISSION'
    if tags.get('motorcycle') in {'yes', 'designated', 'permissive'}:
        return True, 'EXPLICIT_MOTOR_BARRIER_PASSAGE'
    return None, 'PHYSICAL_BARRIER_UNVERIFIED'


def way_direction(tags):
    if any(k.startswith('oneway') and ':conditional' in k for k in tags):
        return None
    value = next((tags[k] for k in ['oneway:motorcycle', 'oneway:motor_vehicle', 'oneway:vehicle', 'oneway'] if k in tags), None)
    if value is None:
        return 1 if tags.get('junction') == 'roundabout' or tags.get('highway') == 'motorway' else 0
    return {'yes': 1, 'true': 1, '1': 1, '-1': -1, 'reverse': -1, 'no': 0, 'false': 0, '0': 0}.get(value)


def merge_elements(roads, features):
    elements, conflicts = {}, []
    for entry in [*roads, *features]:
        key = (entry['type'], entry['id'])
        if key in elements and elements[key] != entry:
            conflicts.append({'type': key[0], 'id': key[1], 'reason': 'SOURCE_ELEMENT_CONFLICT'})
        else:
            elements[key] = entry
    if conflicts:
        raise ValueError('OSM snapshot element conflicts: ' + str(conflicts[:5]))
    return elements


def build_graph(road_source, feature_source, frozen):
    from pyproj import Transformer
    if road_source.get('status') != 'SUCCEEDED':
        return {'status': 'SOURCE_MISSING' if road_source.get('status') == 'SOURCE_MISSING' else 'RETRIEVAL_FAILED', 'graph': None, 'diagnostics': {'reason': 'ROAD_SOURCE_UNAVAILABLE'}}
    if feature_source.get('status') != 'SUCCEEDED':
        return {'status': 'SOURCE_MISSING' if feature_source.get('status') == 'SOURCE_MISSING' else 'RETRIEVAL_FAILED', 'graph': None, 'diagnostics': {'reason': 'RESTRICTION_OR_BARRIER_SOURCE_UNAVAILABLE'}}
    raw = road_source['payload']
    features = feature_source['payload']
    elements = merge_elements(raw['elements'], features['elements'])
    road_way_ids = {e['id'] for e in raw['elements'] if e['type'] == 'way'}
    transform = Transformer.from_crs(4326, frozen['policy']['li3']['metric_crs'], always_xy=True)
    source_nodes = {i: e for (kind, i), e in elements.items() if kind == 'node'}
    nodes, edges, sources, rules, rejects = {}, [], {}, {}, []
    speed = frozen['policy']['travel']['reference_constant_speed']['value'] * 1000 / 3600
    for way_id in sorted(road_way_ids):
        way = elements[('way', way_id)]
        tags = way.get('tags', {})
        if not tags.get('highway') or not isinstance(way.get('nodes'), list) or len(way['nodes']) < 2:
            rejects.append({'way_id': way_id, 'reason': 'INVALID_ROAD_WAY'})
            continue
        if any(i not in source_nodes for i in way['nodes']):
            rejects.append({'way_id': way_id, 'reason': 'MISSING_SOURCE_NODE'})
            continue
        direction = way_direction(tags)
        permission, permission_reason = motor_access(tags, tags['highway'])
        if direction is None:
            permission, permission_reason = None, 'DIRECTION_UNVERIFIED'
        for index, (a, b) in enumerate(zip(way['nodes'], way['nodes'][1:])):
            if a == b:
                rejects.append({'way_id': way_id, 'segment_index': index, 'reason': 'ZERO_SOURCE_SEGMENT'})
                continue
            for i in (a, b):
                n = source_nodes[i]
                x, y = transform.transform(n['lon'], n['lat'])
                if not all(math.isfinite(v) for v in (x, y)):
                    raise ValueError('Invalid projected OSM coordinates')
                nodes[f'n{i}'] = {'node_id': f'n{i}', 'x': x, 'y': y}
            for u, v, label in ([(a, b, 'f')] if direction == 1 else [(b, a, 'r')] if direction == -1 else [(a, b, 'f'), (b, a, 'r')]):
                start, end = nodes[f'n{u}'], nodes[f'n{v}']
                distance = math.hypot(end['x']-start['x'], end['y']-start['y'])
                if distance == 0:
                    rejects.append({'way_id': way_id, 'segment_index': index, 'reason': 'COINCIDENT_DISTINCT_NODES'})
                    continue
                allowed, reason = permission, permission_reason
                barriers = []
                for i in (u, v):
                    status, why = node_permission(source_nodes[i].get('tags', {}))
                    if source_nodes[i].get('tags', {}).get('barrier'):
                        barriers.append({'node_id': i, 'allowed': status, 'reason': why})
                    if status is False:
                        allowed, reason = False, why
                    elif status is None and allowed is not False:
                        allowed, reason = None, why
                key = f'w{way_id}_s{index}_{label}'
                effective_tags = deepcopy(tags)
                for access_key in ('access', 'vehicle', 'motor_vehicle', 'motorcycle'):
                    directional = access_key + (':forward' if label == 'f' else ':backward')
                    if directional in tags:
                        effective_tags[access_key] = tags[directional]
                directional_allowed, directional_reason = motor_access(effective_tags, tags['highway'])
                if directional_allowed is False:
                    allowed, reason = False, directional_reason
                elif directional_allowed is None and allowed is True:
                    allowed, reason = None, directional_reason
                elif directional_allowed is True and permission is not True and direction is not None and not any(b['allowed'] is not True for b in barriers):
                    allowed, reason = True, directional_reason
                edges.append({'edge_id': key, 'u': start['node_id'], 'v': end['node_id'], 'key': key,
                              'length_m': distance, 'travel_time_s': distance/speed, 'highway': tags['highway'],
                              'service': tags.get('service'), 'access_allowed': allowed,
                              'geometry': {'type': 'LineString', 'coordinates': [[start['x'], start['y']], [end['x'], end['y']]]}})
                sources[key] = {'osm_way_id': way_id, 'segment_index': index, 'direction': label,
                                'raw_tags': deepcopy(tags), 'source_node_barriers': barriers,
                                'permission_reason': reason, 'source_version': way.get('version')}
                rules[key] = {'allowed': allowed, 'evidence_refs': [f'osm_way_{way_id}'], 'raw_tags': effective_tags, 'reason': reason}
    bbox = deepcopy(frozen['reference_metric_bbox'])
    if nodes:
        bbox = [min(bbox[0], min(n['x'] for n in nodes.values())), min(bbox[1], min(n['y'] for n in nodes.values())),
                max(bbox[2], max(n['x'] for n in nodes.values())), max(bbox[3], max(n['y'] for n in nodes.values()))]
    asof = raw.get('osm3s', {}).get('timestamp_osm_base')
    graph = {'schema_version': 'li_directed_graph_v1', 'graph_id': frozen['analysis_run_id'] + '-osm',
             'directed': True, 'crs': frozen['policy']['li3']['metric_crs'],
             'units': {'coordinates': 'm', 'distance': 'm', 'travel_time': 's'},
             'coverage': {'bbox': bbox, 'completeness': 'PARTIAL', 'scope': 'Bounded raw OSM extraction; no global connectivity/completeness claim'},
             'provenance': {'origin': 'OFFLINE_SNAPSHOT', 'source_version': 'OSM-' + (asof or 'unknown'),
                            'source_as_of': asof, 'description': 'Actual OSM nodes/ways; no geometry-crossing joins; fixed7kmh assumption'},
             'nodes': sorted(nodes.values(), key=lambda n: n['node_id']), 'edges': sorted(edges, key=lambda e: e['edge_id'])}
    return {'status': 'PARTIAL' if rejects else 'SUCCEEDED', 'graph': graph, 'edge_sources': sources,
            'edge_rules': rules, 'diagnostics': {'rejected_source_segments': rejects,
              'permissions': dict(Counter(str(r['allowed']) for r in rules.values())),
              'way_count': len(road_way_ids), 'source_as_of': asof}}


def apply_turns(graph, edge_sources, edge_rules, feature_source):
    incoming, outgoing = defaultdict(list), defaultdict(list)
    for edge in graph['edges']:
        incoming[edge['v']].append(edge)
        outgoing[edge['u']].append(edge)
    blocked, diagnostics = [], []
    for relation in sorted((e for e in feature_source['payload']['elements'] if e['type'] == 'relation' and e.get('tags', {}).get('type') == 'restriction'), key=lambda e: e['id']):
        tags, members = relation.get('tags', {}), relation.get('members', [])
        rid = relation['id']
        if set(tags.get('except', '').split(';')) & {'motorcycle', 'motor_vehicle', 'vehicle'}:
            diagnostics.append({'relation_id': rid, 'reason': 'MOTORCYCLE_EXEMPT'})
            continue
        restriction = tags.get('restriction:motorcycle', tags.get('restriction:motor_vehicle', tags.get('restriction')))
        if not restriction and not any('conditional' in k and k.startswith('restriction') for k in tags):
            diagnostics.append({'relation_id': rid, 'reason': 'NOT_MOTOR_RESTRICTION'})
            continue
        from_ids = [m['ref'] for m in members if m.get('role') == 'from' and m.get('type') == 'way']
        to_ids = [m['ref'] for m in members if m.get('role') == 'to' and m.get('type') == 'way']
        via = [m for m in members if m.get('role') == 'via']
        simple = len(from_ids) == len(to_ids) == len(via) == 1 and via[0]['type'] == 'node'
        conditional = any('conditional' in k for k in tags)
        supported = simple and not conditional and restriction and (restriction.startswith('no_') or restriction.startswith('only_'))
        if supported:
            node = 'n' + str(via[0]['ref'])
            ins = [e for e in incoming[node] if edge_sources[e['edge_id']]['osm_way_id'] == from_ids[0]]
            outs = [e for e in outgoing[node] if edge_sources[e['edge_id']]['osm_way_id'] == to_ids[0]]
            if from_ids[0] == to_ids[0] and restriction != 'no_u_turn':
                supported = False
            else:
                for a in ins:
                    targets = [b for b in outs if restriction != 'no_u_turn' or b['v'] == a['u']]
                    forbidden = targets if restriction.startswith('no_') else [b for b in outgoing[node] if b not in targets]
                    for b in forbidden:
                        blocked.append({'from_edge': a['edge_id'], 'to_edge': b['edge_id'], 'evidence_ref': f'osm_relation_{rid}'})
                diagnostics.append({'relation_id': rid, 'reason': 'NORMALIZED_VIA_NODE', 'restriction': restriction,
                                    'incoming_matched': len(ins), 'target_outgoing_matched': len(outs)})
        if not supported:
            affected = set(from_ids + to_ids + [m['ref'] for m in via if m['type'] == 'way'])
            count = 0
            for key, source in edge_sources.items():
                if source['osm_way_id'] in affected:
                    if edge_rules[key]['allowed'] is not False:
                        edge_rules[key]['allowed'] = None
                        edge_rules[key]['reason'] = 'UNRESOLVED_TURN_QUARANTINE'
                    count += 1
            diagnostics.append({'relation_id': rid, 'reason': 'UNRESOLVED_TURN_QUARANTINE', 'affected_edges': count,
                                'restriction': restriction, 'conditional': conditional})
    unique = {(b['from_edge'], b['to_edge'], b['evidence_ref']): b for b in blocked}
    return sorted(unique.values(), key=lambda b: (b['from_edge'], b['to_edge'], b['evidence_ref'])), diagnostics


def graph_diagnostics(graph, rules, gap_m):
    import networkx as nx
    from shapely import STRtree
    from shapely.geometry import Point
    g = nx.DiGraph()
    g.add_nodes_from(n['node_id'] for n in graph['nodes'])
    g.add_edges_from((e['u'], e['v']) for e in graph['edges'] if rules[e['edge_id']]['allowed'] is True)
    components = sorted((sorted(c) for c in nx.weakly_connected_components(g)), key=lambda c: (-len(c), c[0]))
    membership = {key: index for index, component in enumerate(components) for key in component}
    nodes = {n['node_id']: n for n in graph['nodes']}
    ends = sorted(n for n in g if g.degree(n) in {1, 2})
    points = [Point(nodes[n]['x'], nodes[n]['y']) for n in ends]
    tree = STRtree(points)
    gaps = []
    for i, point in enumerate(points):
        for j in tree.query(point.buffer(gap_m)):
            j = int(j)
            if j > i and membership[ends[i]] != membership[ends[j]] and point.distance(points[j]) <= gap_m:
                gaps.append({'node_a': ends[i], 'node_b': ends[j], 'distance_m': point.distance(points[j])})
    return {'nodes': len(nodes), 'directed_edges': len(graph['edges']), 'allowed_edges': sum(r['allowed'] is True for r in rules.values()),
            'denied_edges': sum(r['allowed'] is False for r in rules.values()), 'unknown_edges': sum(r['allowed'] is None for r in rules.values()),
            'weak_components': len(components), 'largest_weak_component_nodes': len(components[0]) if components else 0,
            'isolated_nodes': len(list(nx.isolates(g))), 'strong_components': nx.number_strongly_connected_components(g),
            'suspicious_gap_threshold_m': gap_m, 'suspicious_gap_count': len(gaps), 'suspicious_gap_examples': sorted(gaps, key=lambda x: (x['distance_m'], x['node_a'], x['node_b']))[:50],
            'gap_warning': 'Diagnostic only; no edge inserted to repair gaps'}


### location_intelligent_li3_connectors.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_li3_connectors.py
"""Audited frontage assumptions, mapped-barrier checks and source-edge splitting."""
from collections import defaultdict
from copy import deepcopy
import math


def barrier_index(feature_source, frozen):
    from pyproj import Transformer
    from shapely.geometry import LineString, Point, Polygon
    from shapely.ops import polygonize, transform, unary_union
    from shapely import STRtree
    from location_intelligent_li3_graph import node_permission
    elements = feature_source['payload']['elements']
    nodes = {e['id']: e for e in elements if e['type'] == 'node'}
    ways = {e['id']: e for e in elements if e['type'] == 'way'}
    tx = Transformer.from_crs(4326, frozen['policy']['li3']['metric_crs'], always_xy=True).transform
    shapes, records, open_gates, unresolved = [], [], [], []
    def way_line(way):
        if any(i not in nodes for i in way.get('nodes', [])) or len(way.get('nodes', [])) < 2:
            return None
        return transform(tx, LineString([(nodes[i]['lon'], nodes[i]['lat']) for i in way['nodes']]))
    for node in nodes.values():
        tags = node.get('tags', {})
        if not tags.get('barrier'):
            continue
        allowed, reason = node_permission(tags)
        p = transform(tx, Point(node['lon'], node['lat']))
        if allowed is True:
            open_gates.append(p)
        else:
            shapes.append(p); records.append({'source': f'osm_node_{node["id"]}', 'kind': 'node_barrier', 'reason': reason, 'verified_denial': allowed is False})
    for way in ways.values():
        tags = way.get('tags', {})
        kind = 'water' if tags.get('natural') == 'water' or tags.get('waterway') else 'barrier' if tags.get('barrier') else None
        if not kind:
            continue
        if kind == 'water' and tags.get('tunnel') in {'yes', 'culvert'}:
            continue
        line = way_line(way)
        if line is None:
            unresolved.append({'source': f'osm_way_{way["id"]}', 'reason': 'MISSING_BARRIER_NODES'})
            continue
        geom = Polygon(line) if kind == 'water' and line.is_ring else line
        if not geom.is_valid:
            unresolved.append({'source': f'osm_way_{way["id"]}', 'reason': 'INVALID_BARRIER_GEOMETRY'})
            continue
        shapes.append(geom); records.append({'source': f'osm_way_{way["id"]}', 'kind': kind, 'verified_denial': False})
    for relation in elements:
        if relation['type'] != 'relation' or relation.get('tags', {}).get('natural') != 'water':
            continue
        outer, inner = [], []
        missing = False
        for member in relation.get('members', []):
            if member.get('type') != 'way':
                continue
            line = way_line(ways[member['ref']]) if member['ref'] in ways else None
            if line is None:
                missing = True
            else:
                (inner if member.get('role') == 'inner' else outer).append(line)
        polygons = list(polygonize(outer))
        if missing or not polygons:
            unresolved.append({'source': f'osm_relation_{relation["id"]}', 'reason': 'WATER_RELATION_UNRESOLVED'})
            continue
        geom = unary_union(polygons)
        if inner:
            geom = geom.difference(unary_union(list(polygonize(inner))))
        if not geom.is_valid:
            unresolved.append({'source': f'osm_relation_{relation["id"]}', 'reason': 'INVALID_WATER_RELATION'})
            continue
        shapes.append(geom); records.append({'source': f'osm_relation_{relation["id"]}', 'kind': 'water', 'verified_denial': False})
    return {'tree': STRtree(shapes), 'shapes': shapes, 'records': records, 'open_gates': open_gates, 'unresolved': unresolved}


def check_barriers(line, index, epsilon):
    """Water cannot be crossed by an off-road connector; bridges stay in network."""
    hits = []
    for i in index['tree'].query(line):
        i = int(i)
        shape, info = index['shapes'][i], index['records'][i]
        if not line.intersects(shape):
            continue
        intersection = line.intersection(shape)
        if info['kind'] != 'water' and not info.get('verified_denial') and index['open_gates']:
            from shapely.ops import unary_union
            openings = unary_union([p.buffer(epsilon) for p in index['open_gates']])
            if openings.covers(intersection):
                continue
        hits.append(info)
    if hits:
        return False, hits
    if index['unresolved']:
        return None, index['unresolved']
    return True, []


def generate_connectors(base, metrics, feature_source, frozen):
    import numpy as np
    from shapely import STRtree
    from shapely.geometry import LineString, Point, shape
    from shapely.ops import nearest_points
    from location_intelligent_li3_graph import apply_turns
    graph = base['graph']
    sources, rules = base['edge_sources'], base['edge_rules']
    policy = frozen['policy']
    epsilon = policy['li3']['projection_position_precision_m']
    inset = policy['li3']['entrance_inset_m']
    maximum = policy['connectors']['max_building_to_road_m']['value']
    ranking = {name: i for i, names in enumerate(policy['connectors']['road_class_hierarchy']) for name in names}
    groups = {}
    for edge in graph['edges']:
        key = edge['edge_id']; source = sources[key]
        group = f'w{source["osm_way_id"]}_s{source["segment_index"]}'
        if group not in groups:
            coords = edge['geometry']['coordinates']
            if source['direction'] == 'r':
                coords = list(reversed(coords))
            a, b = (edge['u'], edge['v']) if source['direction'] == 'f' else (edge['v'], edge['u'])
            groups[group] = {'line': LineString(coords), 'a': a, 'b': b, 'members': [], 'tags': source['raw_tags'], 'road_class': edge['highway']}
        groups[group]['members'].append(edge)
    group_ids = sorted(groups)
    geometries = [groups[g]['line'] for g in group_ids]
    tree = STRtree(geometries)
    barrier = barrier_index(feature_source, frozen)
    footprints = {b['building_id']: shape(b['metric_geometry']) for b in metrics['all_buildings']}
    footprint_ids = sorted(footprints)
    footprint_geoms = [footprints[k] for k in footprint_ids]
    footprint_tree = STRtree(footprint_geoms)
    cuts, candidates, rejected = defaultdict(dict), [], []
    target = Point(*metrics['target_xy'])
    diagnostics = {'barrier_features': len(barrier['shapes']), 'unresolved_barrier_geometries': barrier['unresolved'],
                   'entrance_claim': 'OWNER_ACCEPTED_ASSUMPTION_NOT_OBSERVED_ENTRANCE'}

    def candidate(kind, building_id, rep, footprint, index):
        group_id = group_ids[index]; group = groups[group_id]
        reason = None
        name = group['road_class']
        name = name[:-5] if name.endswith('_link') else name
        if name not in ranking:
            reason = 'UNRANKED_FRONTAGE_CLASS'
        elif not any(rules[e['edge_id']]['allowed'] is True for e in group['members']):
            reason = 'ACCESS_UNVERIFIED' if any(rules[e['edge_id']]['allowed'] is None for e in group['members']) else 'ACCESS_PROHIBITED'
        elif group['tags'].get('bridge') not in {None, 'no'} or group['tags'].get('tunnel') not in {None, 'no'} or group['tags'].get('layer', '0') not in {'0', '0.0'}:
            reason = 'GRADE_SEPARATED_DIRECT_ENTRY_UNVERIFIED'
        distance = rep.distance(group['line'])
        if kind == 'BUILDING' and distance > maximum:
            return None
        if reason:
            rejected.append({'kind': kind, 'building_id': building_id, 'source_segment': group_id, 'reason': reason, 'distance_m': distance})
            return None
        if kind == 'BUILDING':
            interior = footprint.buffer(-inset)
            interior = interior if not interior.is_empty else footprint.representative_point()
            entrance, access = nearest_points(interior, group['line'])
        else:
            entrance, access = target, nearest_points(target, group['line'])[1]
        position = group['line'].project(access)
        length = group['line'].length
        if position <= epsilon:
            node, access, position = group['a'], Point(group['line'].coords[0]), 0.0
        elif length-position <= epsilon:
            node, access, position = group['b'], Point(group['line'].coords[-1]), length
        else:
            q = round(position/epsilon)
            position = min(max(q*epsilon, epsilon), length-epsilon)
            access = group['line'].interpolate(position)
            node = group_id + f'_x{q}'
        coords = [[entrance.x, entrance.y], [access.x, access.y]] if kind == 'BUILDING' else [[access.x, access.y], [target.x, target.y]]
        line = LineString(coords)
        clear, hits = check_barriers(line, barrier, epsilon)
        if clear is True:
            for j in footprint_tree.query(line):
                j = int(j); fid = footprint_ids[j]
                if fid == building_id or (kind == 'TARGET' and footprint_geoms[j].covers(target)):
                    continue
                if line.intersection(footprint_geoms[j].buffer(-inset)).length > epsilon:
                    clear = False; hits.append({'source': fid, 'kind': 'other_building'})
        if clear is not True:
            rejected.append({'kind': kind, 'building_id': building_id, 'source_segment': group_id,
                             'reason': 'ACCESS_UNVERIFIED' if clear is None else 'MAPPED_BARRIER_OR_BUILDING',
                             'distance_m': distance, 'geometry': {'type': 'LineString', 'coordinates': coords}, 'barrier_hits': hits})
            return None
        cuts[group_id][node] = position
        value = {'connector_id': ('target' if kind == 'TARGET' else building_id) + '_' + group_id + '_' + node,
                 'kind': kind, 'building_id': building_id, 'node_id': node, 'source_segment': group_id,
                 'status': 'OWNER_ASSUMPTION', 'barrier_clear': True,
                 'evidence_refs': [group_id, 'OWNER_FRONTAGE_ASSUMPTION', 'OSM_MAPPED_BARRIER_CHECK'],
                 'geometry': {'type': 'LineString', 'coordinates': coords}, 'class_rank': ranking[name],
                 'candidate_distance_m': distance, 'connector_distance_m': line.length,
                 'warnings': ['NOT_A_VERIFIED_PHYSICAL_ENTRANCE', 'OSM_ABSENCE_IS_NOT_PROOF_OF_NO_BARRIER']}
        candidates.append(value)
        return value

    # No numerical target search cap. Nearest valid source geometry wins;
    # grade/barrier checks can reject it and proceed to the next road.
    if group_ids:
        distances = np.array([target.distance(g) for g in geometries])
        for i in sorted(range(len(group_ids)), key=lambda i: (distances[i], group_ids[i])):
            if candidate('TARGET', None, target, None, i) is not None:
                break
    for b in sorted(metrics['selected_buildings'], key=lambda b: b['building_id']):
        point, footprint = Point(*b['representative_point_m']), footprints[b['building_id']]
        seen = set()
        for i in sorted(map(int, tree.query(point.buffer(maximum))), key=lambda i: group_ids[i]):
            c = candidate('BUILDING', b['building_id'], point, footprint, i)
            if c:
                unique = (c['node_id'], c['class_rank'])
                if unique in seen:
                    candidates.remove(c)
                    rejected.append({'kind': 'BUILDING', 'building_id': b['building_id'], 'source_segment': c['source_segment'], 'reason': 'DUPLICATE_NODE_AND_CLASS'})
                else:
                    seen.add(unique)
    nodes = {n['node_id']: deepcopy(n) for n in graph['nodes']}
    new_edges, new_sources, new_rules, incidents = [], {}, {}, defaultdict(list)
    speed = policy['travel']['reference_constant_speed']['value']*1000/3600
    for group_id in group_ids:
        group = groups[group_id]; line = group['line']
        positions = {group['a']: 0.0, group['b']: line.length, **cuts.get(group_id, {})}
        ordered = sorted(positions, key=lambda n: (positions[n], n))
        for node in ordered:
            point = line.interpolate(positions[node])
            if node not in nodes:
                nodes[node] = {'node_id': node, 'x': point.x, 'y': point.y}
        for parent in group['members']:
            pkey = parent['edge_id']; forward = sources[pkey]['direction'] == 'f'
            path = ordered if forward else list(reversed(ordered))
            for i, (a, b) in enumerate(zip(path, path[1:])):
                start, end = nodes[a], nodes[b]
                coords = [[start['x'], start['y']], [end['x'], end['y']]]
                distance = math.dist(*coords)
                if distance <= 0:
                    continue
                key = pkey + f'_p{i}'
                edge = {**parent, 'edge_id': key, 'u': a, 'v': b, 'key': key, 'length_m': distance,
                        'travel_time_s': distance/speed, 'geometry': {'type': 'LineString', 'coordinates': coords}}
                new_edges.append(edge)
                new_sources[key] = {**deepcopy(sources[pkey]), 'parent_edge_id': pkey}
                new_rules[key] = deepcopy(rules[pkey])
                incidents[(group_id, a)].append(key); incidents[(group_id, b)].append(key)
    new_graph = {**graph, 'nodes': sorted(nodes.values(), key=lambda n: n['node_id']), 'edges': sorted(new_edges, key=lambda e: e['edge_id'])}
    for c in candidates:
        c['road_edge_id'] = min(k for k in incidents[(c['source_segment'], c['node_id'])] if new_rules[k]['allowed'] is True)
    blocked, turn_diagnostics = apply_turns(new_graph, new_sources, new_rules, feature_source)
    # Quarantine inherited by edge rules after split; connector usability is
    # checked again by the existing motor engine, not asserted by this generator.
    diagnostics.update(connectors=len(candidates), rejected_candidates=len(rejected),
                       target_connector_found=any(c['kind'] == 'TARGET' for c in candidates),
                       turn_relations=turn_diagnostics, grade_separation_policy='NO_DIRECT_ENTRY_TO_ELEVATED_OR_TUNNEL_EDGE')
    return {'graph': new_graph, 'edge_sources': new_sources, 'edge_rules': new_rules,
            'blocked_turns': blocked, 'connectors': sorted(candidates, key=lambda c: c['connector_id']),
            'rejected_candidates': rejected, 'diagnostics': diagnostics}


### location_intelligent_li3_core.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_li3_core.py
"""System-Python worker reusing LI-0/LI-1/capacity/motor; no network calls."""
import argparse
from copy import deepcopy
import hashlib
import json
from pathlib import Path

from location_intelligent_buildings import select_buildings, _backend, _spatial
from location_intelligent_capacity import estimate_capacity
from location_intelligent_motor_routing import route_motor_buildings
from location_intelligent_validator import canonical_hash, context_hash
from location_intelligent_li3_sources import read_json, write_json


def prepare(root):
    frozen = read_json(root/'config/pilot.json')
    cache = root/'raw/building_cache'
    for original, expected in frozen['building_cache_files'].items():
        p = cache/Path(original).name
        if not p.is_file() or hashlib.sha256(p.read_bytes()).hexdigest() != expected:
            raise ValueError('Retained building cache changed: '+str(p))
    context = read_json(cache/'context.json')
    manifest = read_json(cache/'manifest.json')
    # The retained source packet includes raw WKB and the bounded-query proof.
    # Re-normalize and compare; do not rely on undocumented manual edits.
    from location_intelligent_overture_probe import normalize
    raw = read_json(cache/'raw_probe.json.gz')
    planned = frozen['policy']['sources']['buildings']
    if raw['release'] != planned['planned_release'] or raw['schema_version'] != planned['planned_schema_version']:
        raise ValueError('Cached building release/schema differs from frozen source plan')
    target = context['target']; pilot = frozen['policy']['li3']
    if target['longitude'] != pilot['longitude'] or target['latitude'] != pilot['latitude'] or context['config']['area_coverage']['metric_crs'] != pilot['metric_crs']:
        raise ValueError('Building cache target/CRS differs from approved pilot')
    normalized = normalize(raw)
    old = read_json(cache/'buildings.geojson')
    if normalized != old:
        raise ValueError('Cached normalized building data differs from retained raw source')
    payload = normalized
    write_json(root/'normalized/buildings.geojson', payload)
    context.update(analysis_run_id=frozen['analysis_run_id'], created_at=frozen['frozen_at'])
    context['context_hash'] = context_hash(context)
    manifest.update(analysis_run_id=context['analysis_run_id'], context_hash=context['context_hash'])
    source = manifest['sources'][0]
    data = (root/'normalized/buildings.geojson').read_bytes()
    source.update(locator='normalized/buildings.geojson', acquisition_mode='OFFLINE_SNAPSHOT',
                  byte_count=len(data), fingerprint={'algorithm': 'sha256', 'value': hashlib.sha256(data).hexdigest()},
                  provenance={'origin': 'OFFLINE_SNAPSHOT', 'description': 'Verified reuse of actual Overture2026-10-21.0 raw probe; original acquisition metadata retained under raw/building_cache', 'dependencies': []})
    for module in manifest['modules']:
        module.update(analysis_run_id=context['analysis_run_id'], context_hash=context['context_hash'], config_hash=canonical_hash(context['config']))
    write_json(root/'normalized/context.json', context)
    write_json(root/'normalized/input_manifest.json', manifest)
    result = select_buildings(context, manifest, root=root, source_ref_id=source['source_ref_id'])
    write_json(root/'building_results/selection.json', result)
    if result['status'] not in {'SUCCEEDED', 'PARTIAL'}:
        raise ValueError('LI-1 selection blocked: '+str(result['diagnostics']))
    capacity = estimate_capacity(result, frozen['policy'])
    write_json(root/'building_results/capacity.json', capacity)
    _, ogr, osr = _backend()
    with osr.ExceptionMgr(useExceptions=True):
        tx, target = _spatial(frozen['policy']['li3']['metric_crs'], context['target'], ogr, osr)
        all_buildings = []
        for feature in payload['features']:
            g = ogr.CreateGeometryFromJson(json.dumps(feature['geometry']))
            if g is None or not g.IsValid():
                continue
            if g.Transform(tx) != 0:
                raise ValueError('Building metric transform failed')
            all_buildings.append({'building_id': feature['properties']['building_id'], 'metric_geometry': json.loads(g.ExportToJson())})
        selected = []
        for b in result['selected']:
            coords = tx.TransformPoint(*b['representative_point']['coordinates'])[:2]
            selected.append({'building_id': b['building_id'], 'representative_point_m': list(coords)})
        metrics = {'target_xy': [target.GetX(), target.GetY()], 'all_buildings': sorted(all_buildings, key=lambda b: b['building_id']),
                   'selected_buildings': sorted(selected, key=lambda b: b['building_id'])}
    write_json(root/'normalized/metric_buildings.json', metrics)
    return {'selection': result['summary'], 'capacity': capacity['summary_by_unit']}


def route(root):
    frozen = read_json(root/'config/pilot.json')
    selection = read_json(root/'building_results/selection.json')
    context = read_json(root/'normalized/context.json')
    manifest = read_json(root/'normalized/input_manifest.json')
    graph = read_json(root/'graph/graph.json.gz')
    evidence = read_json(root/'connectors/evidence.json.gz')
    result = route_motor_buildings(context, manifest, selection, graph, evidence, root=root, policy=frozen['policy'])
    write_json(root/'building_results/routing.json.gz', result)
    return {'status': result['status'], 'summary': result['summary'], 'diagnostics': result['diagnostics']}


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('stage', choices=['prepare', 'route'])
    p.add_argument('--root', required=True, type=Path)
    a = p.parse_args()
    result = prepare(a.root) if a.stage == 'prepare' else route(a.root)
    print(json.dumps(result, sort_keys=True, allow_nan=False))


### location_intelligent_li3.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_li3.py
"""LI-3 explicit acquire/build/replay commands. Existing LI-1/LI-2 are reused."""
import argparse
import csv
from collections import Counter
from copy import deepcopy
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

from location_intelligent_li3_sources import read_json, write_json, canonical_bytes, build_queries, acquire_overpass, utc
from location_intelligent_validator import canonical_hash

REPO = Path(__file__).resolve().parents[1]
CORE = REPO/'notebooks/location_intelligent_li3_core.py'


def run_core(root, stage):
    env = {**os.environ, 'PROJ_NETWORK': 'OFF', 'PYTHONDONTWRITEBYTECODE': '1'}
    cmd = ['/usr/bin/python3', '-B', str(CORE), stage, '--root', str(root)]
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode:
        raise RuntimeError('Core stage failed: '+stage+' '+result.stderr[-1800:]+result.stdout[-1800:])
    return json.loads(result.stdout)


def freeze(root):
    from pyproj import Transformer
    from location_intelligent_config import get_config
    if root.exists():
        raise ValueError('Choose a new pilot directory')
    p = get_config(); li = p['li3']
    forward = Transformer.from_crs(4326, li['metric_crs'], always_xy=True)
    inverse = Transformer.from_crs(li['metric_crs'], 4326, always_xy=True)
    x, y = forward.transform(li['longitude'], li['latitude']); radius = li['network_bbox_radius_m']
    corners = [inverse.transform(a, b) for a in (x-radius, x+radius) for b in (y-radius, y+radius)]
    cache = REPO/li['building_cache']
    checksums = {str(f.relative_to(REPO)): hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(cache.iterdir()) if f.is_file()}
    frozen = {'schema_version': 'li3_pilot_config_v1', 'algorithm_version': 'LI3_V1',
              'analysis_run_id': root.name, 'frozen_at': utc(), 'policy': p, 'target_xy': [x,y],
              'extraction_bbox_wgs84': [min(c[0] for c in corners), min(c[1] for c in corners), max(c[0] for c in corners), max(c[1] for c in corners)],
              'reference_metric_bbox': [x-radius,y-radius,x+radius,y+radius], 'building_cache_files': checksums,
              'target_snap_decision': 'OWNER_APPROVED_NEAREST_VALID_ROAD', 'source_scope': 'N1 technical pilot only; not TGDI',
              'core_python': '/usr/bin/python3', 'diagnostic_gap_m': 10.0}
    frozen['config_hash'] = canonical_hash(frozen)
    write_json(root/'config/pilot.json', frozen, exclusive=True)
    return frozen


def acquire(root):
    f = read_json(root/'config/pilot.json'); p = f['policy']; li = p['li3']
    for name, digest in f['building_cache_files'].items():
        source = REPO/name
        if hashlib.sha256(source.read_bytes()).hexdigest() != digest:
            raise ValueError('Building cache changed: '+name)
        destination = root/'raw/building_cache'/source.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copyfile(source, destination)
    queries = build_queries(f['extraction_bbox_wgs84'], p)
    write_json(root/'config/queries.json', queries)
    for name in ('roads', 'features_pinned', 'features'):
        path = root/'raw'/f'osm_{name}.json.gz'
        if path.exists():
            continue
        query = queries['roads' if name == 'roads' else 'features']
        if name != 'roads':
            roads = read_json(root/'raw/osm_roads.json.gz')
            if roads['status'] != 'SUCCEEDED':
                return {'roads': roads['status'], 'features': 'NOT_ATTEMPTED_ROAD_SOURCE_FAILED'}
            asof = roads['payload']['osm3s'].get('timestamp_osm_base')
            if name == 'features' and (root/'raw/osm_features_pinned.json.gz').exists() and read_json(root/'raw/osm_features_pinned.json.gz')['status'] == 'SUCCEEDED':
                continue
            if name == 'features_pinned' and asof:
                query = query.replace(';', f'[date:"{asof}"];', 1)
        result = acquire_overpass(query, li['overpass_endpoints'], li['request_timeout_s'], li['max_response_bytes'])
        result['actual_query'] = query
        if name == 'features_pinned':
            result['requested_snapshot_as_of'] = asof
        write_json(path, result, exclusive=True)
    selected, warnings = select_feature_source(root)
    return {'roads': read_json(root/'raw/osm_roads.json.gz')['status'],
            'features': read_json(selected)['status'] if selected.exists() else 'SOURCE_MISSING', 'warnings': warnings}


def select_feature_source(root):
    """Prefer a successful exact-as-of extract, preserve failed attempts."""
    pinned = root/'raw/osm_features_pinned.json.gz'
    if pinned.exists() and read_json(pinned)['status'] == 'SUCCEEDED':
        return pinned, []
    tiles = sorted((root/'raw').glob('osm_features_tile_*.json.gz'))
    if len(tiles) == 4 and all(read_json(p)['status'] == 'SUCCEEDED' for p in tiles):
        from location_intelligent_li3_graph import merge_elements
        values = [read_json(p) for p in tiles]
        elements = {}
        for v in values:
            elements = merge_elements(list(elements.values()), v['payload']['elements'])
        combined = {'status': 'SUCCEEDED', 'payload': {'elements': sorted(elements.values(), key=lambda e: (e['type'], e['id'])),
                    'osm3s': values[0]['payload']['osm3s'], 'generator': values[0]['payload'].get('generator')},
                    'requested_snapshot_as_of': values[0].get('requested_snapshot_as_of'), 'tile_sources': [str(p.relative_to(root)) for p in tiles]}
        target = root/'raw/osm_features_combined.json.gz'
        write_json(target, combined)
        return target, []
    fallback = root/'raw/osm_features.json.gz'
    warnings = []
    if fallback.exists() and read_json(fallback)['status'] == 'SUCCEEDED':
        road_source = read_json(root/'raw/osm_roads.json.gz') if (root/'raw/osm_roads.json.gz').exists() else {}
        road_asof = (road_source.get('payload') or {}).get('osm3s', {}).get('timestamp_osm_base')
        feature = read_json(fallback)
        feature_asof = feature.get('requested_snapshot_as_of', feature['payload']['osm3s'].get('timestamp_osm_base'))
        if feature_asof != road_asof:
            warnings.append({'code': 'NON_ATOMIC_SOURCE_TIMESTAMPS', 'road_as_of': road_asof, 'feature_as_of': feature_asof,
                             'effect': 'Shared source elements must match exactly; no temporal completeness claim'})
    return fallback, warnings


def map_decisions(selection, routing, connector_package):
    rows = []
    rejected = connector_package.get('rejected_candidates', [])
    by_id = {r['building_id']: r for r in routing.get('routes', [])}
    connectors = connector_package.get('connectors', [])
    own_connectors, own_rejections = {}, {}
    for c in connectors:
        own_connectors.setdefault(c.get('building_id'), []).append(c)
    for r in rejected:
        own_rejections.setdefault(r.get('building_id'), []).append(r)
    for b in selection['selected']:
        key = b['building_id']; route = by_id.get(key)
        own = own_connectors.get(key, [])
        own_rejects = own_rejections.get(key, [])
        if routing.get('status') in {'FAILED', 'BLOCKED'}:
            state = 'ACCESS_UNVERIFIED' if routing['status'] == 'BLOCKED' else 'PROCESSING_ERROR'
        elif route and route['route_found'] is True:
            state = 'ACCEPTED' if route['scenario_accessible'] else 'OVER_TIME_LIMIT'
        elif not own:
            uncertain = any('UNVERIFIED' in r['reason'] for r in own_rejects)
            state = 'ACCESS_UNVERIFIED' if uncertain else 'NO_VALID_CONNECTOR'
        elif route and route['route_found'] is None:
            state = 'ACCESS_UNVERIFIED'
        else:
            state = 'NO_ROUTE'
        rows.append({'building_id': key, 'source_row_index': b['source_row_index'], 'selection_status': 'SELECTED',
                     'decision_state': state, 'reason': route.get('reason') if route else 'ROUTING_UNAVAILABLE',
                     'routing_ref': key, 'capacity_ref': key, 'connector_count': len(own), 'rejected_connector_candidates': len(own_rejects),
                     'selected_candidate': route.get('selected_candidate') if route else None,
                     'verification_required': True, 'claim_type': 'CONFIGURED_SCENARIO_NOT_PHYSICAL_VERIFICATION'})
    for rejection in selection['reject_ledger']:
        rows.append({'building_id': rejection.get('building_id'), 'source_row_index': rejection['source_row_index'],
                     'selection_status': rejection['disposition'], 'decision_state': 'INVALID_GEOMETRY' if rejection.get('stage') == 'GEOMETRY' else 'EXCLUDED_BY_SELECTION',
                     'reason': rejection['code'], 'capacity_ref': None, 'routing_ref': None})
        if rejection['disposition'] == 'REJECTED' and rejection.get('stage') != 'GEOMETRY':
            rows[-1]['decision_state'] = 'PROCESSING_ERROR'
    return sorted(rows, key=lambda r: r['source_row_index'])


def build(root):
    from location_intelligent_li3_graph import build_graph, graph_diagnostics
    from location_intelligent_li3_connectors import generate_connectors
    f = read_json(root/'config/pilot.json')
    expected = f.pop('config_hash')
    if canonical_hash(f) != expected:
        raise ValueError('Frozen configuration changed')
    f['config_hash'] = expected
    prep = run_core(root, 'prepare')
    roads = read_json(root/'raw/osm_roads.json.gz') if (root/'raw/osm_roads.json.gz').exists() else {'status': 'SOURCE_MISSING'}
    feature_path, warnings = select_feature_source(root)
    features = read_json(feature_path) if feature_path.exists() else {'status': 'SOURCE_MISSING'}
    resolved = {'road_source': 'raw/osm_roads.json.gz', 'feature_source': str(feature_path.relative_to(root)), 'warnings': warnings}
    write_json(root/'config/resolved_sources.json', resolved)
    base = build_graph(roads, features, f)
    write_json(root/'diagnostics/source_graph.json', base['diagnostics'])
    if base['graph'] is None:
        selection = read_json(root/'building_results/selection.json')
        report = {'status': 'BLOCKED', 'reason': base['status'], 'accepted': None,
                  'rows': [{'building_id': b['building_id'], 'decision_state': base['status']} for b in selection['selected']]}
        write_json(root/'building_results/decisions.json.gz', report)
        write_json(root/'report.json', report)
        return report
    write_json(root/'graph/base_graph.json.gz', base['graph'])
    write_json(root/'graph/base_edge_sources.json.gz', base['edge_sources'])
    metrics = read_json(root/'normalized/metric_buildings.json')
    package = generate_connectors(base, metrics, features, f)
    for edge in package['graph']['edges']:
        edge['access_allowed'] = package['edge_rules'][edge['edge_id']]['allowed']
    selection = read_json(root/'building_results/selection.json')
    context = read_json(root/'normalized/context.json')
    evidence = {'schema_version': 'li_motor_evidence_v1', 'graph_hash': canonical_hash(package['graph']),
                'selection_hash': canonical_hash(selection), 'context_hash': context['context_hash'], 'policy_hash': canonical_hash(f['policy']),
                'edge_rules': package['edge_rules'], 'blocked_turns': package['blocked_turns'],
                'restrictions_complete': True, 'connectors': package['connectors']}
    write_json(root/'graph/graph.json.gz', package['graph'])
    write_json(root/'graph/edge_sources.json.gz', package['edge_sources'])
    write_json(root/'connectors/evidence.json.gz', evidence)
    write_json(root/'connectors/connectors.json.gz', package['connectors'])
    write_json(root/'connectors/rejected.json.gz', package['rejected_candidates'])
    write_json(root/'diagnostics/connectors.json', package['diagnostics'])
    gd = graph_diagnostics(package['graph'], package['edge_rules'], f['diagnostic_gap_m'])
    write_json(root/'diagnostics/graph.json', gd)
    outcome = run_core(root, 'route')
    return finalize(root, package, prep, warnings)


def finalize(root, package=None, prep=None, warnings=None):
    f = read_json(root/'config/pilot.json')
    selection = read_json(root/'building_results/selection.json')
    if package is None:
        package = {'connectors': read_json(root/'connectors/connectors.json.gz'),
                   'rejected_candidates': read_json(root/'connectors/rejected.json.gz'),
                   'diagnostics': read_json(root/'diagnostics/connectors.json')}
    if prep is None:
        prep = {'capacity': read_json(root/'building_results/capacity.json')['summary_by_unit']}
    if warnings is None:
        warnings = read_json(root/'config/resolved_sources.json')['warnings']
    gd = read_json(root/'diagnostics/graph.json')
    routing = read_json(root/'building_results/routing.json.gz')
    rows = map_decisions(selection, routing, package)
    write_json(root/'building_results/decisions.json.gz', rows)
    by_building = {b['building_id']: b for b in selection['selected']}
    capacity_rows = {b['building_id']: b for b in read_json(root/'building_results/capacity.json')['rows']}
    with (root/'building_results/decisions.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['source_row_index','building_id','selection_status','decision_state','reason',
            'class','subclass','area_m2','capacity_reference_rounded','capacity_unit','total_time_min','threshold_time_min',
            'building_connector_id','target_connector_id','verification_required'])
        writer.writeheader()
        for row in rows:
            b = by_building.get(row['building_id'], {}); cap = capacity_rows.get(row['building_id'], {}); candidate = row.get('selected_candidate') or {}
            writer.writerow({**{k:row.get(k) for k in ['source_row_index','building_id','selection_status','decision_state','reason','verification_required']},
                'class': b.get('class'), 'subclass': b.get('subclass'), 'area_m2': b.get('area_m2'),
                'capacity_reference_rounded': cap.get('capacity_reference_rounded'), 'capacity_unit': cap.get('unit'),
                **{k:candidate.get(k) for k in ['total_time_min','threshold_time_min','building_connector_id','target_connector_id']}})
    selected_rows = [r for r in rows if r['selection_status'] == 'SELECTED']
    summary = dict(sorted(Counter(r['decision_state'] for r in selected_rows).items()))
    capacity_by_decision = {}
    for row in selected_rows:
        cap = capacity_rows[row['building_id']]
        group = capacity_by_decision.setdefault(row['decision_state'], {}).setdefault(cap['unit'], {'buildings':0,'known_subtotal':0,'unknown_buildings':0})
        group['buildings'] += 1
        if cap['capacity_reference_rounded'] is None:
            group['unknown_buildings'] += 1
        else:
            group['known_subtotal'] += cap['capacity_reference_rounded']
    report = {'status': 'PARTIAL' if warnings or routing['status'] == 'PARTIAL' else routing['status'],
              'analysis_run_id': f['analysis_run_id'], 'source_records': selection['summary']['source_records'],
              'selected_buildings': len(selected_rows), 'all_source_rows_accounted': len(rows),
              'decision_counts_selected': summary, 'decision_denominator': len(selected_rows),
              'graph': gd, 'connectors': len(package['connectors']), 'connector_diagnostics': package['diagnostics'],
              'routing_status': routing['status'], 'routing_diagnostics': routing['diagnostics'],
              'source_warnings': warnings, 'capacity_by_unit': prep['capacity'],
              'capacity_by_decision_state_and_unit': capacity_by_decision,
              'limitations': ['Reference7kmh, not observed ETA', 'Owner entrance/alley/gate assumptions, not physical/legal verification',
                              'No mixed-unit capacity/population claim', 'Bounded OSM graph, unknown permissions retained',
                              'Conditional and complex restrictions conservatively quarantined, not fully interpreted']}
    if len(rows) != selection['summary']['source_records']:
        raise ValueError('Building decisions do not reconcile to source records')
    write_json(root/'report.json', report)
    text = '# LI-3 N1 pilot — actual result\n\n' + 'Status: **'+report['status']+'**.\n\n'
    text += f"{report['source_records']} source records; {report['selected_buildings']} selected; {report['all_source_rows_accounted']} rows accounted.\n\n"
    text += '| Selected-building decision | Count |\n|---|---:|\n' + ''.join(f'| {k} | {v} |\n' for k,v in summary.items())
    text += '\nDenominator: all selected LI-1 buildings. Source exclusions have separate rows.\n\n'
    text += '\n'.join('- '+v for v in report['limitations'])+'\n\n'
    text += 'Full source/query/provenance and uncertainties: report.json, config/, raw/, diagnostics/.\n'
    (root/'report.md').write_text(text)
    return report


def decide(root):
    run_core(root, 'route')
    return finalize(root)


def make_manifest(root, filename='manifest.json'):
    stage = lambda p: p.split('/')[0]
    files = []
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p in {root/filename, (root/filename).with_suffix('.tmp')}:
            continue
        if not p.resolve().is_relative_to(root.resolve()) or p.is_symlink():
            raise ValueError('Artifact outside pilot root')
        name = str(p.relative_to(root)); data = p.read_bytes()
        if name.startswith('normalized/'):
            deps = ['config/pilot.json'] + [str(x.relative_to(root)) for x in sorted((root/'raw/building_cache').glob('*')) if x.is_file()]
        elif name.startswith('graph/'):
            deps = ['config/pilot.json', 'normalized/metric_buildings.json', 'raw/osm_roads.json.gz', read_json(root/'config/resolved_sources.json')['feature_source']]
        elif name.startswith('connectors/'):
            deps = ['graph/graph.json.gz', 'normalized/metric_buildings.json', 'config/pilot.json']
        elif name.startswith('building_results/'):
            deps = ['normalized/buildings.geojson', 'config/pilot.json']
            if 'routing' in name or 'decisions' in name:
                deps += ['graph/graph.json.gz', 'connectors/evidence.json.gz']
        else:
            deps = []
        deps = [d for d in deps if (root/d).is_file()]
        if name == 'normalized/metric_buildings.json':
            deps.append('building_results/selection.json')
        if name == 'building_results/selection.json':
            deps += ['normalized/context.json', 'normalized/input_manifest.json']
        if name == 'building_results/capacity.json':
            deps.append('building_results/selection.json')
        if name == 'building_results/routing.json.gz':
            deps.append('building_results/selection.json')
        if name in {'building_results/decisions.json.gz', 'building_results/decisions.csv'} and (root/'building_results/routing.json.gz').exists():
            deps += ['building_results/routing.json.gz', 'building_results/capacity.json', 'connectors/rejected.json.gz']
        if name in {'report.json', 'report.md'}:
            deps += ['building_results/decisions.json.gz']
        files.append({'path': name, 'sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data), 'role': stage(name), 'dependencies': sorted(set(deps))})
    manifest = {'schema_version': 'li3_manifest_v1', 'analysis_run_id': read_json(root/'config/pilot.json')['analysis_run_id'],
                'status': read_json(root/'report.json')['status'], 'files': files,
                'notes': ['Manifest excludes its own bytes', 'Checksums do not prove data accuracy/completeness']}
    available = {r['path'] for r in files}
    if any(d not in available for row in files for d in row['dependencies']):
        raise ValueError('Manifest dependency missing')
    path = root/filename
    temporary = path.with_suffix('.tmp')
    write_json(temporary, manifest)
    os.replace(temporary, path)
    return manifest


def verify_manifest(root):
    manifest = read_json(root/'manifest.json')
    paths = [r['path'] for r in manifest['files']]
    if len(set(paths)) != len(paths):
        raise ValueError('Duplicate manifest path')
    actual = {str(p.relative_to(root)) for p in root.rglob('*') if p.is_file() and p != root/'manifest.json'}
    if set(paths) != actual:
        raise ValueError('Manifest missing files or contains unlisted artifacts')
    dependencies = {r['path']: r.get('dependencies', []) for r in manifest['files']}
    visited, visiting = set(), set()
    def visit(name):
        if name in visiting:
            raise ValueError('Manifest dependency cycle')
        if name in visited:
            return
        if name not in dependencies:
            raise ValueError('Manifest dependency missing')
        visiting.add(name)
        for dependency in dependencies[name]:
            visit(dependency)
        visiting.remove(name); visited.add(name)
    for name in paths:
        visit(name)
    for row in manifest['files']:
        p = root/row['path']
        if not p.resolve().is_relative_to(root.resolve()) or not p.is_file() or p.is_symlink():
            raise ValueError('Missing/unsafe artifact '+row['path'])
        data = p.read_bytes()
        if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
            raise ValueError('Artifact checksum mismatch '+row['path'])
    return manifest


def replay(root, output):
    manifest = verify_manifest(root)
    if output.exists():
        raise ValueError('Replay output must be new')
    versions_path = root/'config/source_software.json'
    if versions_path.exists():
        import importlib.metadata
        import platform
        current = {'python': platform.python_version(), 'packages': {name: importlib.metadata.version(name) for name in read_json(versions_path)['packages']}}
        if current != read_json(versions_path):
            raise ValueError('Replay software differs from frozen source environment')
    core_versions = root/'config/core_software.json'
    if core_versions.exists():
        cmd = ['/usr/bin/python3', '-c', 'import platform,json,pyproj;from osgeo import gdal;print(json.dumps({"python":platform.python_version(),"gdal":gdal.VersionInfo("RELEASE_NAME"),"pyproj":pyproj.__version__}))']
        current = json.loads(subprocess.check_output(cmd))
        if current != read_json(core_versions):
            raise ValueError('Replay core software differs from frozen environment')
    for row in manifest['files']:
        if row['path'].startswith('engine_source/'):
            current = REPO/'notebooks'/Path(row['path']).name
            if not current.is_file() or hashlib.sha256(current.read_bytes()).hexdigest() != row['sha256']:
                raise ValueError('Recorded algorithm differs: '+str(current))
    output.mkdir(parents=True)
    shutil.copytree(root/'config', output/'config')
    shutil.copytree(root/'raw', output/'raw')
    # Block accidental source requests during replay, including future regressions.
    import requests
    import urllib.request
    from unittest.mock import patch
    with patch.object(requests.Session, 'request', side_effect=RuntimeError('NETWORK_FORBIDDEN_DURING_REPLAY')), patch.object(urllib.request, 'urlopen', side_effect=RuntimeError('NETWORK_FORBIDDEN_DURING_REPLAY')):
        build(output)
    compared, differences = [], []
    for row in manifest['files']:
        name = row['path']
        if name.startswith(('normalized/', 'graph/', 'connectors/', 'building_results/', 'diagnostics/')) or name in {'report.json', 'report.md'}:
            p = output/name
            matched = p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == row['sha256']
            compared.append(name)
            if not matched:
                differences.append(name)
    result = {'status': 'PASS' if not differences else 'FAIL', 'compared': len(compared), 'differences': differences,
              'network_forbidden': True, 'source_manifest_sha256': hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest()}
    write_json(output/'replay_verification.json', result)
    return result


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=['freeze', 'acquire', 'build', 'decide', 'manifest', 'replay', 'verify'])
    parser.add_argument('--root', required=True, type=Path)
    parser.add_argument('--output', type=Path)
    args = parser.parse_args()
    if args.command == 'replay':
        if args.output is None:
            parser.error('--output required for replay')
        result = replay(args.root, args.output)
    else:
        result = {'freeze': freeze, 'acquire': acquire, 'build': build, 'decide': decide, 'manifest': make_manifest, 'verify': verify_manifest}[args.command](args.root)
    print(json.dumps(result, sort_keys=True, allow_nan=False))


### location_intelligent_map_pilot.py


In [ ]:
%%writefile /kaggle/working/area_coverage_cpu_source/notebooks/location_intelligent_map_pilot.py
"""Readable CPU notebook support: new-target sources, BQ layers and interactive map.

This is a scoped notebook prototype, not the approved implementation of the
Area modularization audit. Existing selection/routing algorithms stay unchanged.
"""
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import subprocess
import sys
import re

from location_intelligent_li3_sources import read_json, write_json, canonical_bytes


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')


def query_bbox(latitude, longitude, radius_m):
    """Conservative bounding prefilter only; final predicates determine inclusion."""
    from pyproj import Geod
    geod = Geod(ellps='WGS84')
    positions = [geod.fwd(longitude, latitude, azimuth, radius_m)[:2] for azimuth in range(0, 360, 5)]
    return [min(p[0] for p in positions), min(p[1] for p in positions),
            max(p[0] for p in positions), max(p[1] for p in positions)]


def acquire_buildings(config, output, *, extension_directory=None, partition_index=None):
    """Official pinned STAC items + bbox predicate + explicit nontruncated rows."""
    import requests
    import duckdb
    output = Path(output)
    if output.exists():
        raise ValueError('Raw building output immutable; choose a new source revision')
    release = config['overture_release']
    prefix = f'https://overturemaps-us-west-2.s3.us-west-2.amazonaws.com/release/{release}/theme=buildings/type=building/'
    url = f'https://stac.overturemaps.org/{release}/buildings/building/collection.json'
    bbox = query_bbox(config['latitude'], config['longitude'], config['building_radius_m'] + config['building_query_padding_m'])
    def metadata(url):
        response = requests.get(url, timeout=40)
        response.raise_for_status()
        if len(response.content) > 4_000_000:
            raise ValueError('STAC metadata exceeds bound')
        return response.json(), hashlib.sha256(response.content).hexdigest()
    collection, catalog_hash = metadata(url)
    links = sorted({link['href'] for link in collection['links'] if link['rel'] == 'item'})
    if not links or len(links) > 2000:
        raise ValueError('Unexpected STAC item inventory')
    def fetch_item(link):
        item, checksum = metadata(link)
        return {'url': link, 'bbox': item['bbox'], 'asset': item['assets']['aws']['href'], 'sha256': checksum}
    # Read every advertised item bbox; never infer completeness from link order.
    if partition_index is not None:
        if sha(partition_index) != config['partition_index_sha256']:
            raise ValueError('Partition inventory checksum differs')
        inventory=read_json(partition_index)
        if inventory['release'] != release or inventory['errors'] or inventory['items_checked'] != len(inventory['items']):
            raise ValueError('Unverified partition inventory')
        if catalog_hash != config['partition_catalog_sha256'] or {r['url'] for r in inventory['items']} != set(links):
            raise ValueError('Pinned partition inventory/catalog differs')
        items=inventory['items']
    else:
        with ThreadPoolExecutor(max_workers=4) as pool:
            items = list(pool.map(fetch_item, links))
    matching = [item for item in items if item['bbox'][0] <= bbox[2] and item['bbox'][2] >= bbox[0]
                and item['bbox'][1] <= bbox[3] and item['bbox'][3] >= bbox[1]]
    if not matching or len(matching) > 8 or any(not item['asset'].startswith(prefix) for item in matching):
        raise ValueError('Unsupported/incomplete partition selection')
    # Recheck the actual matching item against live release metadata as well.
    if partition_index is not None:
        for item in matching:
            current=fetch_item(item['url'])
            if current['asset'] != item['asset'] or current['bbox'] != item['bbox']:
                raise ValueError('Matching partition changed from verified inventory')
    assets = sorted({item['asset'] for item in matching})
    connection = duckdb.connect()
    try:
        if extension_directory:
            connection.execute('SET extension_directory=?', [str(extension_directory)])
        try:
            connection.execute('LOAD httpfs')
        except duckdb.Error:
            connection.execute('INSTALL httpfs')
            connection.execute('LOAD httpfs')
        for statement in ["SET enable_geoparquet_conversion=false", "SET http_timeout=45", "SET http_retries=1", "SET threads=2", "SET memory_limit='1GB'"]:
            connection.execute(statement)
        schema = connection.execute('DESCRIBE SELECT * FROM read_parquet(?)', [assets]).fetchall()
        fields = {row[0]: row[1] for row in schema}
        expected = {'id': 'VARCHAR', 'class': 'VARCHAR', 'subtype': 'VARCHAR', 'geometry': 'BLOB',
                    'height': 'DOUBLE', 'num_floors': 'INTEGER', 'version': 'INTEGER'}
        if any(fields.get(k) != v for k, v in expected.items()) or not {'sources', 'bbox'} <= set(fields):
            raise ValueError('Overture consumed-column schema changed')
        query = ('SELECT id,class,subtype,height,num_floors,version,sources,bbox,hex(geometry) AS geometry_wkb_hex '
                 'FROM read_parquet(?) WHERE bbox.xmin<=? AND bbox.xmax>=? AND bbox.ymin<=? AND bbox.ymax>=? '
                 'ORDER BY id LIMIT ?')
        cursor = connection.execute(query, [assets, bbox[2], bbox[0], bbox[3], bbox[1], config['max_building_rows'] + 1])
        names = [item[0] for item in cursor.description]
        rows = [dict(zip(names, row)) for row in cursor.fetchall()]
        if len(rows) > config['max_building_rows']:
            raise ValueError('Building row cap reached; do not publish a truncated coverage result')
    finally:
        connection.close()
    value = {'release': release, 'schema_version': config['overture_schema'], 'collection_url': url,
        'collection_sha256': catalog_hash, 'bbox': bbox, 'items': matching, 'items_checked': len(items),
        'partition_inventory': items, 'physical_schema': schema, 'sql': query,
        'row_cap': config['max_building_rows'], 'truncated': False, 'rows': rows,
        'acquired_at': now(), 'bounded_query_complete': True,
        'coverage_meaning': 'All intersecting advertised STAC items queried; not proof of physical building completeness'}
    write_json(output, value, exclusive=True)
    return value


def normalize_cache(raw_path, cache, config):
    """Runs under GDAL worker: same original WKB normalization and selection input."""
    from location_intelligent_overture_probe import normalize
    from location_intelligent_validator import canonical_hash, context_hash
    cache = Path(cache);cache.mkdir(parents=True, exist_ok=False)
    raw = read_json(raw_path)
    if raw['truncated'] or not raw['bounded_query_complete']:
        raise ValueError('Incomplete building input')
    features = normalize(raw)
    write_json(cache/'raw_probe.json.gz', raw)
    write_json(cache/'buildings.geojson', features)
    stamp = raw['acquired_at']
    run_id = 'area-800m-' + hashlib.sha256(canonical_bytes(config)).hexdigest()[:12]
    context = {'schema_version': 'location_intelligent_v1', 'analysis_run_id': run_id,
        'parent_analysis_run_id': None, 'created_at': stamp,
        'target': {'store_code': None, 'store_name': None, 'latitude': config['latitude'], 'longitude': config['longitude'],
                   'crs': 'EPSG:4326', 'unknown_reasons': {'store_code': 'User supplied coordinates only', 'store_name': 'User supplied coordinates only'}},
        'config': {'version': 'area-map-800m-v1', 'area_coverage': {'metric_crs': config['metric_crs'],
            'parameters': {'building_radius_m': {'value': config['building_radius_m'], 'unit': 'm'},
                'min_area_m2': {'value': config['min_area_m2'], 'unit': 'm2'},
                'max_area_m2': {'value': config['max_area_m2'], 'unit': 'm2'}}, 'unresolved_decisions': []}}}
    context['context_hash'] = context_hash(context)
    source = {'source_ref_id': 'overture-buildings-new-target', 'inventory_id': 'SRC04', 'kind': 'LOCAL_FILE',
        'provider': 'OVERTURE_MAPS', 'locator': 'buildings.geojson', 'acquisition_mode': 'LIVE_ACQUIRED',
        'availability': 'AVAILABLE', 'source_version': config['overture_release'], 'source_as_of': None,
        'acquired_at': stamp, 'schema_version': 'li_building_snapshot_v1',
        'fingerprint': {'algorithm': 'sha256', 'value': sha(cache/'buildings.geojson')},
        'byte_count': (cache/'buildings.geojson').stat().st_size, 'row_count': len(features['features']),
        'coverage': {'scope': f'Complete bounded STAC query bbox={raw["bbox"]}; not physical completeness', 'completeness': 'COMPLETE'},
        'crs': 'EPSG:4326', 'units': {'coordinates': 'degree'}, 'freshness': 'UNKNOWN',
        'provenance': {'origin': 'LIVE_ACQUIRED', 'description': 'Actual pinned Overture WKB; raw SHA256='+sha(raw_path), 'dependencies': []},
        'unknown_reasons': {'source_as_of': 'Provider release is known; individual observation dates vary'}}
    manifest = {'schema_version': 'location_intelligent_v1', 'analysis_run_id': run_id, 'context_hash': context['context_hash'],
        'sources': [source], 'artifacts': [], 'modules': [{'module_run_id': 'area-selection-new-target', 'module': 'area_coverage',
        'analysis_run_id': run_id, 'context_hash': context['context_hash'], 'config_hash': canonical_hash(context['config']),
        'status': 'NOT_RUN', 'required_inputs': [source['source_ref_id']], 'artifact_refs': [],
        'engine': None, 'started_at': None, 'finished_at': None, 'error': None}]}
    write_json(cache/'context.json', context)
    write_json(cache/'manifest.json', manifest)
    write_json(cache/'report.json', {'bounded_query_complete': True, 'source_records': len(features['features']),
        'raw_sha256': sha(raw_path), 'schema_validation_scope': 'Consumed columns/types and canonical geometry; not full official provider schema'})


def prepare_pilot(config, root, *, core_python='/usr/bin/python3', extension_directory=None, partition_index=None):
    """Acquire a NEW target; never relabel or edit the old N1 snapshot."""
    from pyproj import Transformer
    from location_intelligent_config import get_config
    from location_intelligent_validator import canonical_hash
    from location_intelligent_li3_sources import build_queries, acquire_overpass
    root = Path(root)
    if root.exists():
        raise ValueError('Pilot directory exists; use frozen mode or a new source revision')
    root.mkdir(parents=True)
    write_json(root/'requested_config.json', config)
    raw = acquire_buildings(config, root/'overture_raw.json.gz', extension_directory=extension_directory, partition_index=partition_index)
    cache = root/'raw/building_cache'
    completed = subprocess.run([str(core_python), '-B', str(Path(__file__).resolve()), 'normalize',
        str(root/'overture_raw.json.gz'), str(cache), str(root/'requested_config.json')], capture_output=True, text=True)
    if completed.returncode:
        raise RuntimeError('GDAL normalization failed: '+completed.stderr[-1400:])
    policy = get_config()
    policy['li3'].update(latitude=config['latitude'], longitude=config['longitude'], metric_crs=config['metric_crs'],
        building_cache=str(cache), network_bbox_radius_m=config['network_radius_m'], pilot_location_source='USER_COORDINATE_20261024')
    forward = Transformer.from_crs(4326, config['metric_crs'], always_xy=True)
    inverse = Transformer.from_crs(config['metric_crs'], 4326, always_xy=True)
    x,y = forward.transform(config['longitude'],config['latitude']); radius=config['network_radius_m']
    corners=[inverse.transform(a,b) for a in (x-radius,x+radius) for b in (y-radius,y+radius)]
    frozen={'schema_version':'li3_pilot_config_v1','algorithm_version':'LI3_V1','analysis_run_id':root.name,
        'frozen_at':raw['acquired_at'],'policy':policy,'target_xy':[x,y],
        'extraction_bbox_wgs84':[min(p[0] for p in corners),min(p[1] for p in corners),max(p[0] for p in corners),max(p[1] for p in corners)],
        'reference_metric_bbox':[x-radius,y-radius,x+radius,y+radius],
        'building_cache_files':{'raw/building_cache/'+p.name:sha(p) for p in sorted(cache.iterdir()) if p.is_file()},
        'target_snap_decision':'OWNER_APPROVED_NEAREST_VALID_ROAD','source_scope':'User target; building800m / POI2000m',
        'core_python':str(core_python),'diagnostic_gap_m':10.0}
    frozen['config_hash']=canonical_hash(frozen);write_json(root/'config/pilot.json',frozen)
    queries=build_queries(frozen['extraction_bbox_wgs84'],policy);write_json(root/'config/queries.json',queries)
    roads=acquire_overpass(queries['roads'],policy['li3']['overpass_endpoints'],policy['li3']['request_timeout_s'],policy['li3']['max_response_bytes'])
    roads['actual_query']=queries['roads'];write_json(root/'raw/osm_roads.json.gz',roads)
    if roads['status']!='SUCCEEDED':raise RuntimeError('Road acquisition failed; inspect attempts, not a valid empty graph')
    stamp=roads['payload']['osm3s'].get('timestamp_osm_base')
    feature_query=queries['features'].replace(';',f'[date:"{stamp}"];',1) if stamp else queries['features']
    features=acquire_overpass(feature_query,policy['li3']['overpass_endpoints'],policy['li3']['request_timeout_s'],policy['li3']['max_response_bytes'])
    features['actual_query']=feature_query;write_json(root/'raw/osm_features_pinned.json.gz',features)
    if features['status']!='SUCCEEDED':
        features=acquire_overpass(queries['features'],policy['li3']['overpass_endpoints'],policy['li3']['request_timeout_s'],policy['li3']['max_response_bytes'])
        features['actual_query']=queries['features'];write_json(root/'raw/osm_features.json.gz',features)
    if features['status']!='SUCCEEDED':raise RuntimeError('Barrier/restriction acquisition failed; cannot assume clear access')
    return frozen


def bq_credentials(secret_name='GCP_SERVICE_ACCOUNT_JSON'):
    """No secret is displayed, written to output, or embedded in the notebook."""
    import google.auth
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])[0]
    # Kaggle caller explicitly grants access to this named secret.
    from google.oauth2 import service_account
    info=json.loads(UserSecretsClient().get_secret(secret_name))
    return service_account.Credentials.from_service_account_info(info,scopes=['https://www.googleapis.com/auth/cloud-platform'])


def resolve_bq_fields(table, kind, overrides=None):
    """Schema-driven binding; aliases are proposals and every selection is recorded."""
    schema={f.name.upper():f for f in table.schema}
    def pick(aliases, required=False):
        found=next((schema[n.upper()] for n in aliases if n.upper() in schema),None)
        if required and found is None:raise ValueError('Required BigQuery coordinate columns not found')
        return None if found is None else {'name':found.name,'type':found.field_type}
    cfg=overrides or {}
    if any(not isinstance(v,str) or not v.strip() for v in cfg.values()):raise ValueError('Nonempty field names required in overrides')
    if set(cfg)-{'latitude','longitude','point','id','name'}:raise ValueError('Unknown BigQuery schema override')
    if bool(cfg.get('latitude'))!=bool(cfg.get('longitude')):raise ValueError('Both numeric coordinate overrides required')
    lat=pick([cfg['latitude']] if cfg.get('latitude') else ['LAT','LATITUDE'],required=bool(cfg.get('latitude')))
    lon=pick([cfg['longitude']] if cfg.get('longitude') else ['LONG','LON','LONGITUDE'],required=bool(cfg.get('longitude')))
    point=pick([cfg['point']] if cfg.get('point') else ['KOORDINAT','GEOGRAPHY','GEOMETRY'])
    if cfg.get('point'):
        point=pick([cfg['point']],required=True);lat=lon=None
    elif not cfg.get('latitude') and kind=='indomaret' and point is not None:
        # Existing store reference uses KOORDINAT; explicit numeric overrides win.
        lat=lon=None
    if not (lat and lon) and (not point or point['type'] not in {'STRING','GEOGRAPHY'}):
        raise ValueError('Unsupported coordinate schema; set BQ_SCHEMA_OVERRIDES explicitly')
    return {'latitude':lat,'longitude':lon,'point':point,
        'id':pick([cfg['id']] if cfg.get('id') else ['KDTK','STORE_ID','KODE_TOKO','ID'],required=bool(cfg.get('id'))),
        'name':pick([cfg['name']] if cfg.get('name') else ['NAMA_TOKO','NAMA','NAME','COMPETITOR','BRAND'],required=bool(cfg.get('name'))),
        'table_kind':kind}


def build_bq_query(table_id, fields):
    if not re.fullmatch(r'[A-Za-z0-9_-]+\.[A-Za-z0-9_]+\.[A-Za-z0-9_]+',table_id):raise ValueError('Invalid table identifier')
    def column(field):
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',field['name']):raise ValueError('Unsupported field identifier')
        return '`'+field['name']+'`'
    sid='CAST('+column(fields['id'])+' AS STRING)' if fields['id'] else 'CAST(NULL AS STRING)'
    name='CAST('+column(fields['name'])+' AS STRING)' if fields['name'] else 'CAST(NULL AS STRING)'
    if fields['latitude'] and fields['longitude']:
        coordinates='SAFE_CAST('+column(fields['latitude'])+' AS FLOAT64) AS latitude,SAFE_CAST('+column(fields['longitude'])+' AS FLOAT64) AS longitude'
        cte=f'raw AS (SELECT {sid} AS store_id,{name} AS store_name,{coordinates} FROM `{table_id}`)'
    else:
        p=column(fields['point'])
        point=p if fields['point']['type']=='GEOGRAPHY' else f'SAFE.ST_GEOGFROMTEXT({p})'
        cte=f'points AS (SELECT {sid} AS store_id,{name} AS store_name,{point} AS p FROM `{table_id}`), raw AS (SELECT store_id,store_name,SAFE.ST_Y(p) AS latitude,SAFE.ST_X(p) AS longitude FROM points)'
    return (f'WITH {cte}, quality AS (SELECT COUNT(*) AS _source_rows,COUNTIF(latitude IS NULL OR longitude IS NULL OR NOT(latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180)) AS _invalid_coordinates FROM raw), '
      'valid AS (SELECT *,ST_DISTANCE(CASE WHEN latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180 THEN ST_GEOGPOINT(longitude,latitude) ELSE NULL END,ST_GEOGPOINT(@longitude,@latitude),TRUE) AS distance_m FROM raw '
      'WHERE latitude BETWEEN @south AND @north AND longitude BETWEEN @west AND @east AND latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180), '
      'matched AS (SELECT DISTINCT store_id,store_name,latitude,longitude,distance_m FROM valid WHERE distance_m<=@radius_m) '
      'SELECT matched.*,quality._source_rows,quality._invalid_coordinates FROM quality LEFT JOIN matched ON TRUE ORDER BY distance_m,store_id,store_name,latitude,longitude')


def query_bq_layer(config, table_id, kind, *, credentials=None, overrides=None):
    """Metadata → dry run → bounded read; errors are unavailable, never zero stores."""
    from google.cloud import bigquery
    client=bigquery.Client(project=config['bq_billing_project'],credentials=credentials)
    table=client.get_table(table_id,timeout=30)
    fields=resolve_bq_fields(table,kind,overrides)
    print('BQ VERIFIED SCHEMA BINDING',table_id,fields)
    sql=build_bq_query(table_id,fields)
    west,south,east,north=query_bbox(config['latitude'],config['longitude'],config['poi_radius_m']+100.0)
    params=[bigquery.ScalarQueryParameter(k,'FLOAT64',v) for k,v in {
        'latitude':config['latitude'],'longitude':config['longitude'],'radius_m':config['poi_radius_m'],
        'west':west,'south':south,'east':east,'north':north}.items()]
    location=table.location
    dry=client.query(sql,location=location,job_config=bigquery.QueryJobConfig(dry_run=True,use_query_cache=False,query_parameters=params))
    estimate=dry.total_bytes_processed
    print('BQ DRY RUN BYTES',table_id,estimate,'budget',config['bq_maximum_bytes_billed'])
    if estimate is None or estimate>config['bq_maximum_bytes_billed']:
        raise ValueError('BigQuery dry-run byte budget exceeded/unknown; adjust budget explicitly')
    job=client.query(sql,location=location,job_config=bigquery.QueryJobConfig(query_parameters=params,
        maximum_bytes_billed=config['bq_maximum_bytes_billed'],use_query_cache=True))
    try:returned=[dict(r.items()) for r in job.result(timeout=120)]
    except BaseException:
        job.cancel();raise
    if not returned:raise ValueError('BigQuery quality row missing')
    quality={'total_source_rows':returned[0]['_source_rows'],'invalid_coordinates_global':returned[0]['_invalid_coordinates'],
             'scope':'Global parse quality; invalid coordinates cannot be assigned to a radius'}
    rows=[]
    for record in returned:
        row={k:v for k,v in record.items() if not k.startswith('_')}
        if row['latitude'] is None and row['longitude'] is None and row['distance_m'] is None:continue
        rows.append(row)
    for row in rows:
        if not (math.isfinite(row['distance_m']) and 0<=row['distance_m']<=config['poi_radius_m']):raise ValueError('BigQuery radius binding differs')
    return {'status':'SUCCEEDED','table':table_id,'kind':kind,'rows':rows,'count_source_rows_distinct':len(rows),
        'fields':fields,'coordinate_quality':quality,'quality':'PARTIAL_INVALID_COORDINATES' if quality['invalid_coordinates_global'] else 'COMPLETE_QUERY_RESULT','table_schema':[{'name':f.name,'type':f.field_type,'mode':f.mode} for f in table.schema],
        'table_modified':table.modified.isoformat() if table.modified else None,'location':location,
        'radius_m':config['poi_radius_m'],'distance_method':'BIGQUERY_ST_DISTANCE_WGS84_SPHEROID_TRUE',
        'sql':sql,'job_id':job.job_id,'dry_run_bytes':estimate,'billed_bytes':job.total_bytes_billed,'acquired_at':now()}


def poi_locations(layer):
    """Collapse repeated history at the same exact ID/location; retain conflicts."""
    groups={};by_id={}
    for row in layer.get('rows',[]):
        key=(row['store_id'],row['latitude'],row['longitude'])
        g=groups.setdefault(key,{'store_id':row['store_id'],'latitude':row['latitude'],'longitude':row['longitude'],
            'distance_m':row['distance_m'],'names':set(),'source_rows':0})
        if row.get('store_name'):g['names'].add(row['store_name'])
        g['source_rows']+=1
        if row['store_id'] is not None:by_id.setdefault(row['store_id'],set()).add((row['latitude'],row['longitude']))
    result=[]
    for key,g in groups.items():
        g['names']=sorted(g['names']);g['identity_location_conflict']=g['store_id'] is not None and len(by_id[g['store_id']])>1
        result.append(g)
    return sorted(result,key=lambda g:(g['distance_m'],str(g['store_id']),g['latitude'],g['longitude']))


def interactive_map(config, pilot, layers, output):
    """Display-only Folium layers; no routing, capacity or source fetch here."""
    import folium
    from folium.plugins import MarkerCluster, Fullscreen, MeasureControl
    from html import escape
    from shapely.geometry import LineString,mapping,Point
    from shapely.ops import transform
    from pyproj import Transformer
    pilot=Path(pilot)
    decisions=read_json(pilot/'building_results/decisions.json.gz')
    selection=read_json(pilot/'building_results/selection.json')
    buildings=read_json(pilot/'normalized/buildings.geojson')
    graph=read_json(pilot/'graph/graph.json.gz')
    report=read_json(pilot/'report.json')
    frozen=read_json(pilot/'config/pilot.json')['policy']['li3']
    context=read_json(pilot/'normalized/context.json')
    if frozen['latitude']!=config['latitude'] or frozen['longitude']!=config['longitude'] or frozen['metric_crs']!=config['metric_crs']:
        raise ValueError('Map target/CRS differs from actual Area result')
    if context['config']['area_coverage']['parameters']['building_radius_m']['value']!=config['building_radius_m']:
        raise ValueError('Map radius differs from actual selection')
    if len(decisions)!=report['source_records'] or len(selection['selected'])!=report['selected_buildings']:
        raise ValueError('Map ledger/source denominator differs')
    by_id={r['building_id']:r for r in decisions if r.get('building_id')}
    m=folium.Map(location=[config['latitude'],config['longitude']],zoom_start=15,tiles='OpenStreetMap',prefer_canvas=True)
    folium.Marker([config['latitude'],config['longitude']],tooltip='Target pengguna',popup='Koordinat pengguna; bukan ID toko hasil tebakan',icon=folium.Icon(color='red',icon='flag')).add_to(m)
    folium.Circle([config['latitude'],config['longitude']],radius=config['building_radius_m'],color='#2563eb',fill=False,tooltip=f'Seleksi bangunan {config["building_radius_m"]:g} m').add_to(m)
    folium.Circle([config['latitude'],config['longitude']],radius=config['poi_radius_m'],color='#7c3aed',fill=False,tooltip=f'Radius toko/kompetitor {config["poi_radius_m"]:g} m').add_to(m)
    selected_ids={b['building_id'] for b in selection['selected']}
    colors={'ACCEPTED':'#16a34a','ACCESS_UNVERIFIED':'#f59e0b','NO_VALID_CONNECTOR':'#64748b','NO_ROUTE':'#dc2626','OVER_TIME_LIMIT':'#a855f7','EXCLUDED_BY_SELECTION':'#cbd5e1'}
    selected_group=folium.FeatureGroup(name=f'Bangunan terpilih {config["building_radius_m"]:g} m — status skenario',show=True)
    excluded_group=folium.FeatureGroup(name='Footprint sumber di luar seleksi / excluded',show=False)
    selected_features=[];excluded_features=[]
    for feature in buildings['features']:
        bid=feature['properties']['building_id'];row=by_id.get(bid,{})
        state=row.get('decision_state','UNKNOWN')
        view={'type':'Feature','geometry':feature['geometry'],'properties':{'building_id':bid,'state':state,
            'capacity_reference':row.get('capacity_reference_rounded'),'capacity_unit':row.get('capacity_unit'),
            'claim':'Skenario; bukan akses fisik/legal terverifikasi'}}
        (selected_features if bid in selected_ids else excluded_features).append(view)
    for group,features in [(selected_group,selected_features),(excluded_group,excluded_features)]:
        if features:
            folium.GeoJson({'type':'FeatureCollection','features':features},
                style_function=lambda f:{'fillColor':colors.get(f['properties']['state'],'#ef4444'),'color':colors.get(f['properties']['state'],'#ef4444'),'weight':.6,'fillOpacity':.45},
                tooltip=folium.GeoJsonTooltip(fields=['building_id','state','capacity_reference','capacity_unit','claim'])).add_to(group)
    selected_group.add_to(m);excluded_group.add_to(m)
    inverse=Transformer.from_crs(config['metric_crs'],4326,always_xy=True)
    forward=Transformer.from_crs(4326,config['metric_crs'],always_xy=True)
    xy=forward.transform(config['longitude'],config['latitude']);display_disk=Point(*xy).buffer(config['poi_radius_m'],quad_segs=64)
    roads=[];seen=set()
    for edge in graph['edges']:
        geom=edge.get('geometry')
        if not geom:continue
        line=LineString(geom['coordinates']);key=line.wkb
        if key in seen:continue
        seen.add(key);clipped=line.intersection(display_disk)
        if clipped.is_empty or clipped.geom_type not in {'LineString','MultiLineString'}:continue
        roads.append({'type':'Feature','geometry':mapping(transform(inverse.transform,clipped)),
            'properties':{'edge_id':edge['edge_id'],'highway':edge.get('highway'),'claim':'OSM graph; displayed geometry clipped only'}})
    folium.GeoJson({'type':'FeatureCollection','features':roads},name=f'Network OSM — tampilan {config["poi_radius_m"]:g} m',
        style_function=lambda f:{'color':'#334155','weight':1.3,'opacity':.55},tooltip=folium.GeoJsonTooltip(fields=['highway','edge_id'])).add_to(m)
    statuses=[]
    for kind,title,color in [('indomaret','Indomaret','blue'),('competitor','Kompetitor','orange')]:
        layer=layers[kind]
        if layer['status']=='SUCCEEDED':
            request=layer.get('request_binding',{})
            if request.get('latitude')!=config['latitude'] or request.get('longitude')!=config['longitude'] or request.get('radius_m')!=config['poi_radius_m']:
                raise ValueError('POI snapshot belongs to a different target/radius')
            if any(not math.isfinite(r['distance_m']) or not 0<=r['distance_m']<=config['poi_radius_m'] for r in layer['rows']):
                raise ValueError('Out-of-radius POI cannot enter map')
        locations=poi_locations(layer) if layer['status']=='SUCCEEDED' else None
        group=MarkerCluster(name=title+f' ≤{config["poi_radius_m"]:g}m'+(' — BELUM TERSEDIA' if locations is None else ''),show=True)
        for row in locations or []:
            popup='<br>'.join([escape(title),escape(str(row['store_id'])),escape(' / '.join(row['names'])),
                f"Jarak {row['distance_m']:.1f} m",'KONFLIK ID/lokasi' if row['identity_location_conflict'] else '',escape(layer['table'])])
            folium.Marker([row['latitude'],row['longitude']],tooltip=escape(title+': '+(' / '.join(row['names']) or str(row['store_id']))),
                popup=folium.Popup(popup,max_width=400),icon=folium.Icon(color=color,icon='shopping-cart')).add_to(group)
        group.add_to(m)
        statuses.append(f'{title}: '+('BELUM TERSEDIA — '+escape(layer.get('error_type','UNKNOWN')) if locations is None else str(len(locations))+' lokasi marker'+(' — ada koordinat sumber invalid' if layer.get('coordinate_quality',{}).get('invalid_coordinates_global') else '')))
    legend=("<div style='position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:12px;border:1px solid #777;max-width:380px;font-size:12px'>"
        f'<b>AREA COVERAGE — SKENARIO EKSPERIMEN</b><br>Bangunan {config["building_radius_m"]:g} m; toko/kompetitor {config["poi_radius_m"]:g} m.<br>'
        +'<br>'.join(statuses)+f"<br>Selected: {len(selection['selected'])}; accepted skenario: {report['decision_counts_selected'].get('ACCEPTED',0)}"
        '<br>Hijau=accepted skenario; jingga=unverified; abu=no connector; ungu=over-time; merah=no route.'
        '<br>7km/jam; round(menit)≤10; kapasitas bukan populasi.<br>OSM/Overture attribution tetap berlaku.</div>')
    m.get_root().html.add_child(folium.Element(legend))
    folium.LayerControl(collapsed=False).add_to(m);Fullscreen().add_to(m);MeasureControl().add_to(m)
    west,south,east,north=query_bbox(config['latitude'],config['longitude'],config['poi_radius_m']+100)
    m.fit_bounds([[south,west],[north,east]])
    m.save(str(output))
    return m


def load_bq_layers(config, cache, *, secret_name='GCP_SERVICE_ACCOUNT_JSON', refresh=False, overrides=None):
    """A successful cached result can reopen without credentials or a new query."""
    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)
    definitions=config.get('bq_tables',{'indomaret':'locdeproject.DWH_NS_LOC.AREA_COVERAGE','competitor':'locdeproject.DWH_NS_LOC.COMPETITOR'})
    result={};credentials=None;credential_error=None
    for kind,table in definitions.items():
        path=cache/(kind+'.json')
        binding={'latitude':config['latitude'],'longitude':config['longitude'],'radius_m':config['poi_radius_m'],
            'table':table,'overrides':(overrides or {}).get(kind,{})}
        fingerprint=hashlib.sha256(canonical_bytes(binding)).hexdigest()
        if path.exists() and not refresh:
            layer=read_json(path)
            if layer.get('request_fingerprint')!=fingerprint or layer.get('status')!='SUCCEEDED':
                raise ValueError('BigQuery cache differs; use a new explicit source revision')
            result[kind]=layer
            continue
        if path.exists() and refresh:
            raise ValueError('REFRESH requires a new cache directory before querying')
        try:
            if credentials is None:
                if credential_error:raise RuntimeError(credential_error)
                try:credentials=bq_credentials(secret_name)
                except Exception as error:
                    credential_error=type(error).__name__;raise
            layer=query_bq_layer(config,table,kind,credentials=credentials,overrides=(overrides or {}).get(kind))
            layer['request_fingerprint']=fingerprint
            layer['request_binding']=binding
            # Cache only actual complete query results. Failure is never a valid empty cache.
            if path.exists():raise ValueError('REFRESH requires a new cache directory; finalized query result is immutable')
            write_json(path,layer,exclusive=True)
            result[kind]=layer
        except Exception as error:
            result[kind]={'status':'BLOCKED' if credentials is None else 'FAILED','kind':kind,'table':table,
                'rows':None,'count_source_rows_distinct':None,'error_type':credential_error or type(error).__name__,
                'radius_m':config['poi_radius_m'],'request_fingerprint':fingerprint,
                'next_action':'Enable Kaggle Secret and verify BigQuery read/job permissions; rerun BQ/map cells'}
    return result


if __name__=='__main__':
    if len(sys.argv)==5 and sys.argv[1]=='normalize':normalize_cache(sys.argv[2],sys.argv[3],read_json(sys.argv[4]))
    else:raise SystemExit('This helper is executed by the visible notebook cells')


In [ ]:
sys.path.insert(0,str(SOURCE_ROOT/'notebooks'))
os.environ['PYTHONPATH']=str(SOURCE_ROOT/'notebooks')
from location_intelligent_li3_sources import read_json,write_json
from location_intelligent_map_pilot import sha,prepare_pilot,load_bq_layers,interactive_map
from location_intelligent_li3 import build,make_manifest,verify_manifest
from IPython.display import display,FileLink,HTML
import importlib,platform,importlib.metadata
for name in ['location_intelligent_li3','location_intelligent_motor_routing','location_intelligent_map_pilot']:
 module=importlib.import_module(name)
 assert Path(module.__file__).resolve().is_relative_to(SOURCE_ROOT)
 print('ACTIVE SOURCE:',module.__file__)
CODE_SHA=hashlib.sha256(json.dumps({p.name:sha(p) for p in sorted((SOURCE_ROOT/'notebooks').glob('*.py'))},sort_keys=True).encode()).hexdigest()
key=hashlib.sha256(json.dumps({'config':CONFIG,'code':CODE_SHA,'revision':SOURCE_REVISION},sort_keys=True).encode()).hexdigest()[:16]
OUTPUT=Path('/kaggle/working/results/area_coverage_cpu')/key
OUTPUT.mkdir(parents=True,exist_ok=True)
PILOT=OUTPUT/'pilot';print('OUTPUT:',OUTPUT)


## 4. Snapshot actual dan pilihan acquisition
Data sumber dipilih melalui inventory ber-checksum, bukan nama folder asumsi.
Sumber N1 lama tidak direlabel. Snapshot default ini memang berasal dari target/radius baru.
Perhitungan di bawah tetap menjalankan source cell pada CPU; tidak sekadar membuka HTML hasil lama.


In [ ]:
EXPECTED_MANIFEST='58b7ce21d87c48481ab560ffd81bc4bdd10b6bf0848c5a9577cbbebb45537f9a'
SOURCE_PACKET=None
for p in Path('/kaggle/input').rglob('manifest.json'):
 if p.is_file() and sha(p)==EXPECTED_MANIFEST:
  SOURCE_PACKET=p.parent;break
if SOURCE_MODE=='FROZEN':
 if SOURCE_PACKET is None:
  # Only data may be unpacked here; runtime helper imports remain under SOURCE_ROOT.
  archives=[p for p in Path('/kaggle/input').rglob('area_source_snapshot.snapshot') if sha(p)=='a2393d48d0ebcffae510c2c43c526bc502dc9d53a4275c6332fafdc08bdcab19']
  assert len(archives)==1,'Attach chrisbiran/area-coverage-800m-source-snapshot; no implicit live fallback'
  import zipfile
  unpack=OUTPUT/'input_snapshot';unpack.mkdir(exist_ok=True)
  with zipfile.ZipFile(archives[0]) as z:
   for item in z.infolist():
    assert (unpack/item.filename).resolve().is_relative_to(unpack.resolve())
    assert ((item.external_attr>>16)&0o170000)!=0o120000
   z.extractall(unpack)
  found=[p for p in unpack.rglob('manifest.json') if sha(p)==EXPECTED_MANIFEST]
  assert len(found)==1;SOURCE_PACKET=found[0].parent
 verify_manifest(SOURCE_PACKET)
 requested=read_json(SOURCE_PACKET/'requested_config.json')
 for field in ['latitude','longitude','building_radius_m','min_area_m2','max_area_m2','metric_crs','network_radius_m','overture_release','overture_schema']:
  assert requested[field]==CONFIG[field],f'Frozen source differs: {field}; REFRESH/new revision required'
 print('VERIFIED FROZEN SOURCE:',SOURCE_PACKET)
elif SOURCE_MODE!='REFRESH':
 raise ValueError('Explicit SOURCE_MODE FROZEN or REFRESH required')



## 5. Full CPU calculation — seluruh source rows
Semua bangunan sumber diproses; tidak sampling footprint. Rule selection/routing/capacity asli dipertahankan. Runtime aktual direkam. Source OSM yang non-atomic/unknown tetap PARTIAL, tidak diubah menjadi izin.


In [ ]:
if (PILOT/'manifest.json').exists():
 verify_manifest(PILOT)
 print('REOPEN existing calculation — no new routing')
else:
 if SOURCE_MODE=='FROZEN':
  PILOT.mkdir(parents=True,exist_ok=False)
  shutil.copytree(SOURCE_PACKET/'raw',PILOT/'raw')
  shutil.copytree(SOURCE_PACKET/'config',PILOT/'config')
  shutil.copyfile(SOURCE_PACKET/'requested_config.json',PILOT/'requested_config.json')
 else:
  index_hits=list(Path('/kaggle/input').rglob('overture_partition_index.data'))
  assert len(index_hits)==1,'Pinned partition inventory required for explicit refresh'
  CONFIG['partition_index_sha256']=sha(index_hits[0]);CONFIG['partition_catalog_sha256']='42e6cc7b99176797e08ecebc85bc21dc7428008e56b47151463d3f10edfa5986'
  prepare_pilot(CONFIG,PILOT,core_python=CORE_PYTHON,partition_index=index_hits[0])
 packages=['duckdb','geopandas','networkx','numpy','osmnx','pandas','pyproj','requests','shapely']
 actual_source={'python':platform.python_version(),'packages':{p:importlib.metadata.version(p) for p in packages}}
 write_json(PILOT/'config/source_software.json',actual_source)
 command='import platform,json,pyproj;from osgeo import gdal;print(json.dumps({"python":platform.python_version(),"gdal":gdal.VersionInfo("RELEASE_NAME"),"pyproj":pyproj.__version__}))'
 actual_core=json.loads(subprocess.check_output([CORE_PYTHON,'-c',command]))
 write_json(PILOT/'config/core_software.json',actual_core)
 print('ACTUAL CPU CORE:',actual_core)
 report=build(PILOT)
 make_manifest(PILOT)
 print('CPU calculation complete:',report['source_records'],'source rows')
report=read_json(PILOT/'report.json')
selection=read_json(PILOT/'building_results/selection.json')
assert report['all_source_rows_accounted']==report['source_records']
assert sum(report['decision_counts_selected'].values())==report['selected_buildings']
import pandas as pd
print('ANALYSIS STATUS:',report['status'])
display(pd.DataFrame([selection['summary']]))
display(pd.DataFrame([{'decision':k,'buildings':v} for k,v in report['decision_counts_selected'].items()]))
print('CAPACITY — reference split units, not population:',report['capacity_by_unit'])
print('SOURCE WARNINGS:',report['source_warnings'])



## 6. Akses BigQuery dan radius2000m
1. Buat service account pembaca di project `locdeproject`.
2. Grant BigQuery Job User pada billing project dan Data Viewer pada kedua tabel.
3. Buat JSON key dan simpan sebagai Kaggle Secret `GCP_SERVICE_ACCOUNT_JSON`; aktifkan akses notebook.
4. Jangan masukkan key ke source/outputs/GitHub/chat.

Metadata schema dibaca lebih dahulu. Nama kolom belum diklaim terverifikasi sebelum metadata berhasil.
Query memakai parameter, jarak WGS84 spheroid≤2000m, dry run dan byte budget256MiB per query.
Data historis yang sama ID/lokasinya digabung untuk marker; konflik satu ID pada lokasi berbeda tetap terlihat.
Kegagalan auth/query ditampilkan sebagai BLOCKED/FAILED dan jumlah null, bukan toko0.


In [ ]:
BQ_CACHE=OUTPUT/'bq_snapshot'
if BQ_FROZEN_CACHE_INPUT is not None and not BQ_CACHE.exists():
 shutil.copytree(Path(BQ_FROZEN_CACHE_INPUT),BQ_CACHE)
if BQ_MODE=='FROZEN':
 assert BQ_CACHE.exists() and all((BQ_CACHE/(k+'.json')).exists() for k in ['indomaret','competitor']),'Provide a successful BQ snapshot from the same request; missing is not zero'
elif BQ_MODE!='ACQUIRE_ONCE':
 raise ValueError('BQ_MODE must be FROZEN or ACQUIRE_ONCE')
layers=load_bq_layers(CONFIG,BQ_CACHE,secret_name=BQ_SECRET_NAME,overrides=BQ_SCHEMA_OVERRIDES)
write_json(OUTPUT/'bq_readiness.json',layers)
from location_intelligent_map_pilot import poi_locations
rows=[]
for kind,layer in layers.items():
 locations=poi_locations(layer) if layer['status']=='SUCCEEDED' else None
 rows.append({'layer':kind,'table':layer['table'],'status':layer['status'],'marker_locations':None if locations is None else len(locations),'error_type':layer.get('error_type')})
 if layer['status']=='SUCCEEDED':
  print('VERIFIED SCHEMA BINDING',kind,layer['fields']);display(pd.DataFrame(locations))
display(pd.DataFrame(rows))
view_identity={'pilot_manifest_sha256':sha(PILOT/'manifest.json'),'code_sha256':CODE_SHA,
 'poi_snapshots':{k:hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':')).encode()).hexdigest() for k,v in layers.items()}}
VIEW_VERSION=hashlib.sha256(json.dumps(view_identity,sort_keys=True,separators=(',',':')).encode()).hexdigest()
VIEW=OUTPUT/'views'/VIEW_VERSION
VIEW.mkdir(parents=True,exist_ok=True)
REOPEN_VIEW=(VIEW/'view_manifest.json').exists()
if REOPEN_VIEW:
 for row in read_json(VIEW/'view_manifest.json')['files']:
  assert sha(VIEW/row['path'])==row['sha256'],'Saved view changed'
print('VIEW VERSION:',VIEW_VERSION)



## 7. Peta interaktif di output Kaggle
Layer: target, lingkaran800m/2000m, footprint dengan keputusan skenario, network OSM, marker Indomaret dan kompetitor.
Layer Control, zoom/pan, tooltip/popup, marker cluster, fullscreen dan alat ukur tersedia.
Marker hanya berasal dari hasil sumber aktual. Jika BQ belum tersedia, banner menyatakan keterbatasannya.
Basemap OSM adalah konteks tampilan online; angka berasal dari snapshot, bukan perubahan tile basemap.


In [ ]:
if not REOPEN_VIEW:
 m=interactive_map(CONFIG,PILOT,layers,VIEW/'area_coverage_map.html')
 display(m)
else:
 from html import escape
 saved=(VIEW/'area_coverage_map.html').read_text()
 display(HTML('<iframe srcdoc="'+escape(saved,quote=True)+'" style="width:100%;height:800px;border:0"></iframe>'))
display(FileLink(str(VIEW/'area_coverage_map.html')))



## 8. Export dan kesiapan hasil
Peta HTML/CSV/GeoJSON serta source manifest disimpan. Tidak memasukkan query gagal sebagai data kosong. Hasil lama immutable; perubahan sumber/policy/runtime membuat revision baru. Untuk reopen di session berikutnya, lampirkan saved output/source snapshot; koordinat saja bukan identitas data.


In [ ]:
from location_intelligent_map_pilot import poi_locations
public_metadata=[]
for kind,layer in layers.items():
 if layer['status']=='SUCCEEDED' and not REOPEN_VIEW:
  frame=pd.DataFrame(poi_locations(layer));frame.to_csv(VIEW/(kind+'_within_2km.csv'),index=False)
  features=[{'type':'Feature','geometry':{'type':'Point','coordinates':[r['longitude'],r['latitude']]},'properties':{k:v for k,v in r.items() if k not in ['latitude','longitude']}} for r in poi_locations(layer)]
  write_json(VIEW/(kind+'_within_2km.geojson'),{'type':'FeatureCollection','features':features})
 public_metadata.append({'layer':kind,'status':layer['status'],'table':layer['table'],'count':None if layer['status']!='SUCCEEDED' else len(poi_locations(layer))})
summary={'result_version':VIEW_VERSION,'status':'MAP_READY' if all(x['status']=='SUCCEEDED' for x in layers.values()) else 'PARTIAL_BQ_BLOCKED_OR_FAILED',
 'target':{'latitude':CONFIG['latitude'],'longitude':CONFIG['longitude']},'building_radius_m':CONFIG['building_radius_m'],'poi_radius_m':CONFIG['poi_radius_m'],
 'analysis_status':report['status'],'source_records':report['source_records'],'selected_buildings':report['selected_buildings'],
 'decisions':report['decision_counts_selected'],'poi_layers':public_metadata,'source_manifest_sha256':sha(PILOT/'manifest.json'),
 'code_sha256':CODE_SHA,'gpu_used':False,'execution':'FROZEN_SOURCE_CPU_CALCULATION_OR_VERIFIED_REOPEN',
 'claim':'EXPERIMENTAL_SCENARIO_NOT_PHYSICAL_ACCESS_OR_POPULATION_VALIDATION'}
if not REOPEN_VIEW:
 write_json(VIEW/'run_summary.json',summary)
 write_json(VIEW/'view_manifest.json',{'schema_version':'area_cpu_map_outputs_v1','result_version':VIEW_VERSION,'files':[{'path':str(p.relative_to(VIEW)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(VIEW.rglob('*')) if p.is_file() and p.name!='view_manifest.json']})
write_json(OUTPUT/'latest_view.json',{'result_version':VIEW_VERSION,'path':str(VIEW.relative_to(OUTPUT))})
print(json.dumps(summary,indent=2));print('IMMUTABLE VIEW DIRECTORY:',VIEW)

